In [1]:
# ==========================================================================================
# STEP 1 — MOUNT GOOGLE DRIVE AND LOCATE PROJECT
# Inspection only — NO model/API calls, NO experiment execution
# ==========================================================================================

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

print("\n" + "=" * 100)
print("STEP 1 — LOCATING DETERMINISTIC INTENT TRANSLATION PROJECT")
print("=" * 100)

SEARCH_ROOT = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

print(f"\nSearch root: {SEARCH_ROOT}")
print(f"Exists     : {SEARCH_ROOT.exists()}")

if not SEARCH_ROOT.exists():
    raise FileNotFoundError(
        f"STOP: Expected Google Drive directory not found:\n{SEARCH_ROOT}"
    )

print("\nTop-level contents:")
print("-" * 100)

for p in sorted(SEARCH_ROOT.iterdir()):
    kind = "DIR " if p.is_dir() else "FILE"
    print(f"{kind:4}  {p.name}")

print("\n" + "=" * 100)
print("STEP 1 COMPLETE — INSPECTION ONLY")
print("=" * 100)
print("No files created.")
print("No files modified.")
print("No model/API calls made.")
print("No experiments executed.")

Mounted at /content/drive

STEP 1 — LOCATING DETERMINISTIC INTENT TRANSLATION PROJECT

Search root: /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation
Exists     : True

Top-level contents:
----------------------------------------------------------------------------------------------------
DIR   experiment-1-v2-together_FINAL
DIR   journal_experiments_persistent

STEP 1 COMPLETE — INSPECTION ONLY
No files created.
No files modified.
No model/API calls made.
No experiments executed.


In [2]:
# ==========================================================================================
# STEP 2 — VERIFY RUNNER AND AVAILABLE E3-v3 DATASETS
# Inspection only — NO model/API calls, NO experiment execution
# ==========================================================================================

from pathlib import Path
import hashlib

MASTER = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/"
    "experiment-1-v2-together_FINAL"
)

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

CANDIDATE = MASTER / (
    "experiment-2b/heldout/candidates/"
    "e3v3_heldout_candidates_120_v2.csv"
)

FROZEN = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

OFFICIAL_RUN = MASTER / (
    "experiment-2b/heldout/results/"
    "e3v3_heldout_benchmark_run_v1.csv"
)

def sha256(path):
    if not path.exists():
        return "NOT FOUND"

    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

print("=" * 110)
print("STEP 2 — VERIFY E3-v3 RUNNER AND DATASETS")
print("=" * 110)

items = [
    ("MASTER", MASTER),
    ("RUNNER", RUNNER),
    ("CANDIDATE DATASET", CANDIDATE),
    ("FROZEN BENCHMARK", FROZEN),
    ("HISTORICAL OFFICIAL RUN", OFFICIAL_RUN),
]

for name, path in items:
    print(f"\n{name}")
    print("-" * 110)
    print("Path   :", path)
    print("Exists :", path.exists())

    if path.is_file():
        print("Size   :", path.stat().st_size, "bytes")
        print("SHA256 :", sha256(path))

print("\n" + "=" * 110)
print("KNOWN FROZEN REFERENCES FROM PRIOR EXPERIMENT")
print("=" * 110)

print("""
Expected canonical E3-v3 runner SHA256:
40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257

Expected frozen 120-case benchmark SHA256:
84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de
""")

print("=" * 110)
print("STEP 2 COMPLETE — INSPECTION ONLY")
print("=" * 110)
print("No files created.")
print("No files modified.")
print("No model/API calls made.")
print("No experiments executed.")

STEP 2 — VERIFY E3-v3 RUNNER AND DATASETS

MASTER
--------------------------------------------------------------------------------------------------------------
Path   : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL
Exists : True

RUNNER
--------------------------------------------------------------------------------------------------------------
Path   : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL/experiment-2b/implementation/e3_v3/run_experiment2b_e3v3_r7_dev_v1.py
Exists : True
Size   : 26903 bytes
SHA256 : 40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257

CANDIDATE DATASET
--------------------------------------------------------------------------------------------------------------
Path   : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL/experiment-2b/heldout/candidates/e3v3_heldout_candidates_120_v2.csv
Ex

In [3]:
# ==========================================================================================
# STEP 3 — CREATE ISOLATED CONFIRMATORY RERUN WORKSPACE
# NO model/API calls, NO benchmark execution
# ==========================================================================================

from pathlib import Path
import json
from datetime import datetime, timezone

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RERUN = BASE / "rerun-missing-artifacts"

DIRS = [
    RERUN,
    RERUN / "preregistration",
    RERUN / "results",
    RERUN / "results" / "smoke",
    RERUN / "results" / "raw",
    RERUN / "results" / "metrics",
    RERUN / "results" / "comparison",
    RERUN / "logs",
    RERUN / "manifests",
]

print("=" * 110)
print("STEP 3 — CREATE CONFIRMATORY RERUN WORKSPACE")
print("=" * 110)

print("\nProtected master:")
print(MASTER)

print("\nRerun workspace:")
print(RERUN)

if not MASTER.exists():
    raise FileNotFoundError("STOP: Protected master is missing.")

# Safety: rerun directory MUST NOT be inside protected master.
try:
    RERUN.relative_to(MASTER)
    raise RuntimeError(
        "STOP: rerun-missing-artifacts unexpectedly resolves inside protected master."
    )
except ValueError:
    pass

for d in DIRS:
    d.mkdir(parents=True, exist_ok=True)

print("\nDirectory structure created:")
print("-" * 110)

for d in DIRS:
    print(d.relative_to(BASE))

# ------------------------------------------------------------------------------------------
# Write preregistration BEFORE any smoke/model execution
# ------------------------------------------------------------------------------------------

PREREG = {
    "experiment": "confirmatory_rerun_missing_experiment_A_raw_artifacts",
    "date": "2026-10-03",

    "purpose": (
        "Confirmatory recovery rerun of Experiment A arms whose ephemeral raw "
        "artifacts were lost after a Google Colab runtime restart."
    ),

    "status": "PRE_EXECUTION",

    "important_boundary": (
        "These reruns do not replace, modify, reconstruct, or overwrite the "
        "previously frozen Experiment A results. Today's outputs are stored "
        "separately and will be compared retrospectively with the frozen "
        "recorded metrics."
    ),

    "runner": {
        "path": (
            "experiment-2b/implementation/e3_v3/"
            "run_experiment2b_e3v3_r7_dev_v1.py"
        ),
        "sha256": (
            "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
        ),
    },

    "dataset": {
        "path": (
            "experiment-2b/reproducibility/"
            "e3v3_heldout_benchmark_v1.0/"
            "e3v3_heldout_benchmark_120_v1.csv"
        ),
        "sha256": (
            "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de"
        ),
        "cases": 120,
        "valid_cases": 58,
        "invalid_cases": 62,
    },

    "historical_official_run": {
        "sha256": (
            "12c3a82bd8cde2d07de352155e9ee14913c821fb745d401c7ffe2f207e850e5e"
        )
    },

    "arms": {
        "A2": {
            "model": "meta-llama/Llama-3.3-70B-Instruct-Turbo",
            "recorded_accuracy_pct": 83.33,
            "recorded_VAR_pct": 82.76,
            "recorded_IRR_pct": 83.87,
            "recorded_containment_violations": 0,
        },
        "A7": {
            "model": "arize-ai/qwen-2-1.5b-instruct",
            "recorded_accuracy_pct": 60.00,
            "recorded_VAR_pct": 18.97,
            "recorded_IRR_pct": 98.39,
            "recorded_containment_violations": 0,
        },
        "A8": {
            "model": "Prism-ML/Ternary-Bonsai-27B",
            "recorded_accuracy_pct": 75.83,
            "recorded_VAR_pct": 70.69,
            "recorded_IRR_pct": 80.65,
            "recorded_containment_violations": 0,
        },
        "A9": {
            "model": "deepseek-ai/DeepSeek-V4.1-Flash",
            "recorded_accuracy_pct": 84.17,
            "recorded_VAR_pct": 87.93,
            "recorded_IRR_pct": 80.65,
            "recorded_containment_violations": 0,
        },
    },

    "execution_plan": [
        "Verify environment and API credential availability.",
        "Run 5-case smoke test independently for A2, A7, A8, and A9.",
        "Inspect every smoke output before full execution.",
        "Execute full 120-case rerun only for arms whose endpoint is operational.",
        "Check full outputs for exceptions/API failures and completeness.",
        "Compute today's metrics from today's outputs.",
        "Compare today's results with previously frozen recorded metrics.",
    ],

    "comparison_rule": (
        "Today's rerun is confirmatory only. Agreement or disagreement will be "
        "reported as observed. Previously frozen values will not be changed."
    ),

    "created_utc": datetime.now(timezone.utc).isoformat(),
}

PREREG_FILE = (
    RERUN
    / "preregistration"
    / "experiment_A_missing_artifacts_confirmatory_rerun_prereg_v1.json"
)

with open(PREREG_FILE, "w", encoding="utf-8") as f:
    json.dump(PREREG, f, indent=2)

print("\nPreregistration created:")
print(PREREG_FILE)

print("\n" + "=" * 110)
print("STEP 3 COMPLETE")
print("=" * 110)

print("Created isolated rerun workspace.")
print("Created pre-execution preregistration.")
print("Protected master was NOT modified.")
print("No model/API calls made.")
print("No benchmark executed.")

STEP 3 — CREATE CONFIRMATORY RERUN WORKSPACE

Protected master:
/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL

Rerun workspace:
/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts

Directory structure created:
--------------------------------------------------------------------------------------------------------------
rerun-missing-artifacts
rerun-missing-artifacts/preregistration
rerun-missing-artifacts/results
rerun-missing-artifacts/results/smoke
rerun-missing-artifacts/results/raw
rerun-missing-artifacts/results/metrics
rerun-missing-artifacts/results/comparison
rerun-missing-artifacts/logs
rerun-missing-artifacts/manifests

Preregistration created:
/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/preregistration/experiment_A_missing_artifacts_confirmatory_rerun_prereg_v1.json

STEP 3 COMPLETE
Created isolated rerun workspace.
Created pre-execu

In [4]:
# ==========================================================================================
# STEP 4 — CONFIGURE AND VERIFY EXECUTION ENVIRONMENT
# NO model/API calls, NO benchmark execution
# ==========================================================================================

import os
import sys
import getpass
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

RERUN = BASE / "rerun-missing-artifacts"

print("=" * 110)
print("STEP 4 — CONFIGURE EXECUTION ENVIRONMENT")
print("=" * 110)

# ------------------------------------------------------------------------------------------
# 1. Verify Python
# ------------------------------------------------------------------------------------------

print("\nPython")
print("-" * 110)
print("Executable :", sys.executable)
print("Version    :", sys.version.split()[0])

# ------------------------------------------------------------------------------------------
# 2. Set repository root on PYTHONPATH
# ------------------------------------------------------------------------------------------

os.environ["PYTHONPATH"] = str(MASTER)

if str(MASTER) not in sys.path:
    sys.path.insert(0, str(MASTER))

print("\nRepository environment")
print("-" * 110)
print("PYTHONPATH :", os.environ["PYTHONPATH"])
print("MASTER     :", MASTER)
print("RUNNER     :", RUNNER)
print("DATASET    :", DATASET)

# ------------------------------------------------------------------------------------------
# 3. Verify critical files
# ------------------------------------------------------------------------------------------

assert MASTER.exists(), "STOP: Master repository missing."
assert RUNNER.exists(), "STOP: Canonical runner missing."
assert DATASET.exists(), "STOP: Frozen benchmark missing."
assert RERUN.exists(), "STOP: rerun-missing-artifacts workspace missing."

print("\nCritical paths:")
print("MASTER  : OK")
print("RUNNER  : OK")
print("DATASET : OK")
print("RERUN   : OK")

# ------------------------------------------------------------------------------------------
# 4. Check required imports
# ------------------------------------------------------------------------------------------

print("\nImport checks")
print("-" * 110)

modules = [
    "pandas",
    "numpy",
    "requests",
]

for module in modules:
    try:
        __import__(module)
        print(f"{module:20} : OK")
    except Exception as e:
        print(f"{module:20} : FAILED -> {type(e).__name__}: {e}")

# Check project src package separately
try:
    import src
    print(f"{'src':20} : OK")
except Exception as e:
    print(f"{'src':20} : FAILED -> {type(e).__name__}: {e}")

# ------------------------------------------------------------------------------------------
# 5. Securely request Together API key
# ------------------------------------------------------------------------------------------

print("\nTogether API credential")
print("-" * 110)

existing = os.environ.get("TOGETHER_API_KEY")

if existing:
    print("TOGETHER_API_KEY already exists in environment.")
else:
    key = getpass.getpass(
        "Enter TOGETHER_API_KEY (input will be hidden): "
    ).strip()

    if not key:
        raise RuntimeError("STOP: No Together API key was provided.")

    os.environ["TOGETHER_API_KEY"] = key

# Never print the key itself.
api_key = os.environ.get("TOGETHER_API_KEY", "")

print("TOGETHER_API_KEY present :", bool(api_key))
print("Key length               :", len(api_key))
print("Key value                : [HIDDEN]")

if len(api_key) < 10:
    raise RuntimeError(
        "STOP: API key exists but appears unexpectedly short."
    )

print("\n" + "=" * 110)
print("STEP 4 COMPLETE")
print("=" * 110)
print("Environment configured.")
print("API credential loaded into runtime memory.")
print("API key was NOT written to rerun-missing-artifacts.")
print("No model/API calls made.")
print("No benchmark cases executed.")

STEP 4 — CONFIGURE EXECUTION ENVIRONMENT

Python
--------------------------------------------------------------------------------------------------------------
Executable : /usr/bin/python3
Version    : 3.13.15

Repository environment
--------------------------------------------------------------------------------------------------------------
PYTHONPATH : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL
MASTER     : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL
RUNNER     : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL/experiment-2b/implementation/e3_v3/run_experiment2b_e3v3_r7_dev_v1.py
DATASET    : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL/experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3_heldout_benchmark_120_v1.csv

Critical paths:
MASTER  : OK
RUNNER  : 

In [5]:
# ==========================================================================================
# STEP 5 — A2 SMOKE TEST
# 5 cases only
# Model: meta-llama/Llama-3.3-70B-Instruct-Turbo
# ==========================================================================================

import os
import sys
import subprocess
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

OUT = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "smoke"
    / "_smoke_A2.csv"
)

MODEL = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

print("=" * 110)
print("STEP 5 — A2 SMOKE TEST")
print("=" * 110)

print("\nModel   :", MODEL)
print("Dataset :", DATASET)
print("Output  :", OUT)
print("Limit   : 5")

# Do not overwrite an earlier smoke result accidentally.
if OUT.exists():
    raise FileExistsError(
        f"STOP: Smoke output already exists:\n{OUT}\n"
        "Do not overwrite existing evidence."
    )

cmd = [
    sys.executable,
    str(RUNNER),
    "--dataset", str(DATASET),
    "--out", str(OUT),
    "--model", MODEL,
    "--limit", "5",
]

env = os.environ.copy()
env["PYTHONPATH"] = str(MASTER)

print("\nStarting A2 smoke test...")
print("-" * 110)

result = subprocess.run(
    cmd,
    cwd=str(MASTER),
    env=env,
    text=True,
    capture_output=True,
)

print("\nRETURN CODE:", result.returncode)

print("\nSTDOUT")
print("-" * 110)
print(result.stdout if result.stdout else "[no stdout]")

print("\nSTDERR")
print("-" * 110)
print(result.stderr if result.stderr else "[no stderr]")

print("\nOUTPUT FILE")
print("-" * 110)
print("Exists :", OUT.exists())

if OUT.exists():
    print("Size   :", OUT.stat().st_size, "bytes")

print("\n" + "=" * 110)

if result.returncode == 0 and OUT.exists():
    print("STEP 5 EXECUTION COMPLETED")
    print("Do NOT run A7/A8/A9 yet.")
    print("Next step will inspect the A2 smoke artifact.")
else:
    print("STEP 5 DID NOT COMPLETE SUCCESSFULLY")
    print("STOP HERE — do not run another model.")

print("=" * 110)

STEP 5 — A2 SMOKE TEST

Model   : meta-llama/Llama-3.3-70B-Instruct-Turbo
Dataset : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL/experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3_heldout_benchmark_120_v1.csv
Output  : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/smoke/_smoke_A2.csv
Limit   : 5

Starting A2 smoke test...
--------------------------------------------------------------------------------------------------------------

RETURN CODE: 1

STDOUT
--------------------------------------------------------------------------------------------------------------
[no stdout]

STDERR
--------------------------------------------------------------------------------------------------------------
Traceback (most recent call last):
  File "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL/experiment-2b/implementation/e3_v3

In [6]:
# ==========================================================================================
# STEP 5A — CHECK REQUIRED TOGETHER PACKAGE VERSION
# Inspection only — NO model/API calls
# ==========================================================================================

from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

print("=" * 110)
print("STEP 5A — CHECK TOGETHER PACKAGE REQUIREMENT")
print("=" * 110)

candidate_files = [
    MASTER / "requirements.txt",
    MASTER / "requirements-dev.txt",
    MASTER / "pyproject.toml",
    MASTER / "setup.py",
    MASTER / "environment.yml",
]

found_requirement_files = []

for path in candidate_files:
    if path.exists():
        found_requirement_files.append(path)

        print(f"\nFILE: {path}")
        print("-" * 110)

        text = path.read_text(
            encoding="utf-8",
            errors="replace"
        )

        matches = [
            line.strip()
            for line in text.splitlines()
            if "together" in line.lower()
        ]

        if matches:
            for line in matches:
                print(line)
        else:
            print("[No Together dependency found in this file]")

print("\n" + "=" * 110)

if not found_requirement_files:
    print("No dependency specification files found.")
else:
    print(f"Dependency files inspected: {len(found_requirement_files)}")

print("=" * 110)
print("STEP 5A COMPLETE")
print("=" * 110)
print("No packages installed.")
print("No files created or modified.")
print("No model/API calls made.")
print("A2 smoke test was NOT rerun.")

STEP 5A — CHECK TOGETHER PACKAGE REQUIREMENT

FILE: /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL/requirements.txt
--------------------------------------------------------------------------------------------------------------
together>=1.5

Dependency files inspected: 1
STEP 5A COMPLETE
No packages installed.
No files created or modified.
No model/API calls made.
A2 smoke test was NOT rerun.


In [7]:
# ==========================================================================================
# STEP 5B — INSTALL REQUIRED TOGETHER SDK
# Environment setup only — NO model/API calls
# ==========================================================================================

import sys
import subprocess

print("=" * 110)
print("STEP 5B — INSTALL TOGETHER SDK")
print("=" * 110)

print("\nInstalling repository requirement: together>=1.5")
print("-" * 110)

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "together>=1.5"],
    text=True,
    capture_output=True
)

print("\nRETURN CODE:", result.returncode)

print("\nPIP OUTPUT")
print("-" * 110)
print(result.stdout[-5000:] if result.stdout else "[no stdout]")

if result.stderr:
    print("\nPIP STDERR")
    print("-" * 110)
    print(result.stderr[-3000:])

if result.returncode != 0:
    print("\nSTOP: Together SDK installation failed.")
else:
    print("\n" + "=" * 110)
    print("VERIFY INSTALLATION")
    print("=" * 110)

    try:
        import together

        print("Import together : OK")

        version = getattr(together, "__version__", "version attribute unavailable")
        print("Version         :", version)

        from together import Together
        print("from together import Together : OK")

    except Exception as e:
        print("IMPORT VERIFICATION FAILED")
        print(type(e).__name__, ":", e)

print("\n" + "=" * 110)
print("STEP 5B COMPLETE")
print("=" * 110)
print("No project files modified.")
print("No rerun evidence modified.")
print("No model/API calls made.")
print("No benchmark cases executed.")

STEP 5B — INSTALL TOGETHER SDK

Installing repository requirement: together>=1.5
--------------------------------------------------------------------------------------------------------------

RETURN CODE: 0

PIP OUTPUT
--------------------------------------------------------------------------------------------------------------
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 890.5/890.5 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.3/279.3 kB 19.0 MB/s eta 0:00:00
  Attempting uninstall: rich
    Found existing installation: rich 13.9.4
    Uninstalling rich-13.9.4:
      Successfully uninstalled rich-13.9.4


PIP STDERR
--------------------------------------------------------------------------------------------------------------
ERROR: pip's dependency resolver does not currently take into account all 

In [8]:
# ==========================================================================================
# STEP 5C — A2 SMOKE TEST RETRY AFTER TOGETHER SDK INSTALLATION
# 5 cases only
# ==========================================================================================

import os
import sys
import subprocess
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

OUT = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "smoke"
    / "_smoke_A2.csv"
)

MODEL = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

print("=" * 110)
print("STEP 5C — A2 SMOKE TEST RETRY")
print("=" * 110)

print("\nModel   :", MODEL)
print("Dataset :", DATASET)
print("Output  :", OUT)
print("Limit   : 5")

# Safety check
if OUT.exists():
    raise FileExistsError(
        f"STOP: Output already exists:\n{OUT}\n"
        "Do not overwrite existing evidence."
    )

if not os.environ.get("TOGETHER_API_KEY"):
    raise RuntimeError(
        "STOP: TOGETHER_API_KEY is no longer present in the runtime."
    )

cmd = [
    sys.executable,
    str(RUNNER),
    "--dataset", str(DATASET),
    "--out", str(OUT),
    "--model", MODEL,
    "--limit", "5",
]

env = os.environ.copy()
env["PYTHONPATH"] = str(MASTER)

print("\nStarting A2 5-case smoke test...")
print("-" * 110)

result = subprocess.run(
    cmd,
    cwd=str(MASTER),
    env=env,
    text=True,
    capture_output=True,
)

print("\nRETURN CODE:", result.returncode)

print("\nSTDOUT")
print("-" * 110)
print(result.stdout if result.stdout else "[no stdout]")

print("\nSTDERR")
print("-" * 110)
print(result.stderr if result.stderr else "[no stderr]")

print("\nOUTPUT FILE")
print("-" * 110)
print("Exists :", OUT.exists())

if OUT.exists():
    print("Size   :", OUT.stat().st_size, "bytes")

print("\n" + "=" * 110)

if result.returncode == 0 and OUT.exists():
    print("STEP 5C EXECUTION COMPLETED")
    print("A2 produced a smoke-test artifact.")
    print("DO NOT run the full A2 benchmark yet.")
    print("DO NOT run A7/A8/A9 yet.")
else:
    print("STEP 5C DID NOT COMPLETE SUCCESSFULLY")
    print("STOP HERE.")

print("=" * 110)

STEP 5C — A2 SMOKE TEST RETRY

Model   : meta-llama/Llama-3.3-70B-Instruct-Turbo
Dataset : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL/experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3_heldout_benchmark_120_v1.csv
Output  : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/smoke/_smoke_A2.csv
Limit   : 5

Starting A2 5-case smoke test...
--------------------------------------------------------------------------------------------------------------

RETURN CODE: 0

STDOUT
--------------------------------------------------------------------------------------------------------------
[1/5] HO001
[2/5] HO002
[3/5] HO003
[4/5] HO004
[5/5] HO005

Rows written: 5
Output: /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/smoke/_smoke_A2.csv


STDERR
-------------------------------------------------------------------------------

In [9]:
# ==========================================================================================
# STEP 6 — VALIDATE A2 SMOKE ARTIFACT
# Inspection only — NO model/API calls
# ==========================================================================================

from pathlib import Path
import pandas as pd
import hashlib

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

FILE = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "smoke"
    / "_smoke_A2.csv"
)

print("=" * 110)
print("STEP 6 — VALIDATE A2 SMOKE ARTIFACT")
print("=" * 110)

if not FILE.exists():
    raise FileNotFoundError(f"STOP: Smoke artifact not found:\n{FILE}")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

print("\nFILE")
print("-" * 110)
print("Path   :", FILE)
print("Size   :", FILE.stat().st_size, "bytes")
print("SHA256 :", sha256(FILE))

df = pd.read_csv(FILE)

print("\nBASIC STRUCTURE")
print("-" * 110)
print("Rows    :", len(df))
print("Columns :", len(df.columns))

print("\nCOLUMN NAMES")
print("-" * 110)
for i, col in enumerate(df.columns, 1):
    print(f"{i:02d}. {col}")

# ------------------------------------------------------------------------------------------
# Case IDs
# ------------------------------------------------------------------------------------------

possible_id_cols = [
    c for c in df.columns
    if c.lower() in {
        "case_id", "id", "test_id", "intent_id",
        "sample_id", "benchmark_id"
    }
]

print("\nCASE IDENTIFIERS")
print("-" * 110)

if possible_id_cols:
    id_col = possible_id_cols[0]
    print("Detected ID column :", id_col)
    print("Values             :", df[id_col].tolist())
else:
    print("No standard case-ID column automatically detected.")

# ------------------------------------------------------------------------------------------
# Error / exception inspection
# ------------------------------------------------------------------------------------------

error_cols = [
    c for c in df.columns
    if any(
        token in c.lower()
        for token in [
            "error",
            "exception",
            "failure",
            "api_error",
            "llm_error"
        ]
    )
]

print("\nERROR / EXCEPTION CHECK")
print("-" * 110)

total_populated_errors = 0

if not error_cols:
    print("No error/exception-named columns detected.")
else:
    print("Detected columns:", error_cols)

    for col in error_cols:
        values = df[col].fillna("").astype(str).str.strip()

        # Treat common empty-like values as empty
        populated = ~values.str.lower().isin(
            ["", "nan", "none", "null", "false"]
        )

        count = int(populated.sum())
        total_populated_errors += count

        print(f"{col:40} populated rows: {count}/{len(df)}")

        if count:
            cols_to_show = []
            if possible_id_cols:
                cols_to_show.append(possible_id_cols[0])
            cols_to_show.append(col)

            print(
                df.loc[populated, cols_to_show]
                .to_string(index=False)
            )

# ------------------------------------------------------------------------------------------
# Important execution/result fields
# ------------------------------------------------------------------------------------------

interesting_tokens = [
    "llm",
    "parse",
    "accept",
    "reject",
    "valid",
    "stage",
    "descriptor",
    "exception",
    "error",
]

interesting_cols = [
    c for c in df.columns
    if any(token in c.lower() for token in interesting_tokens)
]

print("\nIMPORTANT EXECUTION FIELDS")
print("-" * 110)

if interesting_cols:
    print(df[interesting_cols].to_string(index=False))
else:
    print("No matching execution fields detected.")

# ------------------------------------------------------------------------------------------
# Missing values
# ------------------------------------------------------------------------------------------

print("\nMISSING VALUES")
print("-" * 110)

missing = df.isna().sum()
missing = missing[missing > 0]

if len(missing) == 0:
    print("No missing values.")
else:
    print(missing.to_string())

# ------------------------------------------------------------------------------------------
# Final smoke assessment
# ------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("A2 SMOKE ASSESSMENT")
print("=" * 110)

row_ok = len(df) == 5
error_ok = total_populated_errors == 0

print("Expected rows              :", 5)
print("Actual rows                :", len(df))
print("Row-count check            :", "PASS" if row_ok else "FAIL")
print("Populated error indicators :", total_populated_errors)
print("Error check                :", "PASS" if error_ok else "REVIEW")

if row_ok and error_ok:
    print("\nPRELIMINARY RESULT: A2 smoke artifact appears structurally clean.")
    print("Do NOT start the 120-case run yet.")
else:
    print("\nPRELIMINARY RESULT: A2 requires investigation.")
    print("STOP — do not run another model.")

print("\nSTEP 6 COMPLETE")
print("No model/API calls made.")
print("No benchmark cases executed.")
print("=" * 110)

STEP 6 — VALIDATE A2 SMOKE ARTIFACT

FILE
--------------------------------------------------------------------------------------------------------------
Path   : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/smoke/_smoke_A2.csv
Size   : 19419 bytes
SHA256 : 4638509336a0167bec15d5b4fc1b951b6c1e0e86c63e309d9c1d76c3a39707fc

BASIC STRUCTURE
--------------------------------------------------------------------------------------------------------------
Rows    : 5
Columns : 41

COLUMN NAMES
--------------------------------------------------------------------------------------------------------------
01. id
02. intent
03. family_id
04. repair_id
05. subcategory
06. difficulty
07. linguistic_form
08. constraint_type
09. control_type
10. notes
11. expected_valid
12. expected_service
13. pipeline
14. accepted
15. decision_correct
16. llm_invoked
17. parse_ok
18. llm_latency_ms
19. end_to_end_latency_ms
20. finish_reason
21. rejection_stage
2

In [10]:
# ==========================================================================================
# STEP 7 — A7 SMOKE TEST
# 5 cases only
# ==========================================================================================

import os
import sys
import subprocess
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

OUT = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "smoke"
    / "_smoke_A7.csv"
)

MODEL = "arize-ai/qwen-2-1.5b-instruct"

print("=" * 110)
print("STEP 7 — A7 SMOKE TEST")
print("=" * 110)

print("\nModel   :", MODEL)
print("Dataset :", DATASET)
print("Output  :", OUT)
print("Limit   : 5")

if OUT.exists():
    raise FileExistsError(
        f"STOP: Output already exists:\n{OUT}\n"
        "Do not overwrite existing evidence."
    )

if not os.environ.get("TOGETHER_API_KEY"):
    raise RuntimeError("STOP: TOGETHER_API_KEY is missing.")

cmd = [
    sys.executable,
    str(RUNNER),
    "--dataset", str(DATASET),
    "--out", str(OUT),
    "--model", MODEL,
    "--limit", "5",
]

env = os.environ.copy()
env["PYTHONPATH"] = str(MASTER)

print("\nStarting A7 5-case smoke test...")
print("-" * 110)

result = subprocess.run(
    cmd,
    cwd=str(MASTER),
    env=env,
    text=True,
    capture_output=True,
)

print("\nRETURN CODE:", result.returncode)

print("\nSTDOUT")
print("-" * 110)
print(result.stdout if result.stdout else "[no stdout]")

print("\nSTDERR")
print("-" * 110)
print(result.stderr if result.stderr else "[no stderr]")

print("\nOUTPUT FILE")
print("-" * 110)
print("Exists :", OUT.exists())

if OUT.exists():
    print("Size   :", OUT.stat().st_size, "bytes")

print("\n" + "=" * 110)

if result.returncode == 0 and OUT.exists():
    print("STEP 7 EXECUTION COMPLETED")
    print("A7 produced a smoke-test artifact.")
    print("Do NOT run the full benchmark yet.")
else:
    print("STEP 7 DID NOT COMPLETE SUCCESSFULLY")
    print("STOP HERE.")

print("=" * 110)

STEP 7 — A7 SMOKE TEST

Model   : arize-ai/qwen-2-1.5b-instruct
Dataset : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL/experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3_heldout_benchmark_120_v1.csv
Output  : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/smoke/_smoke_A7.csv
Limit   : 5

Starting A7 5-case smoke test...
--------------------------------------------------------------------------------------------------------------

RETURN CODE: 0

STDOUT
--------------------------------------------------------------------------------------------------------------
[1/5] HO001
[2/5] HO002
[3/5] HO003
[4/5] HO004
[5/5] HO005

Rows written: 5
Output: /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/smoke/_smoke_A7.csv


STDERR
------------------------------------------------------------------------------------------------

In [11]:
# ==========================================================================================
# STEP 8 — VALIDATE A7 SMOKE ARTIFACT
# Inspection only — NO model/API calls
# ==========================================================================================

from pathlib import Path
import pandas as pd
import hashlib

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

FILE = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "smoke"
    / "_smoke_A7.csv"
)

print("=" * 110)
print("STEP 8 — VALIDATE A7 SMOKE ARTIFACT")
print("=" * 110)

if not FILE.exists():
    raise FileNotFoundError(f"STOP: Smoke artifact not found:\n{FILE}")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

df = pd.read_csv(FILE)

print("\nFILE")
print("-" * 110)
print("Path   :", FILE)
print("Size   :", FILE.stat().st_size, "bytes")
print("SHA256 :", sha256(FILE))

print("\nSTRUCTURE")
print("-" * 110)
print("Rows    :", len(df))
print("Columns :", len(df.columns))
print("IDs     :", df["id"].tolist() if "id" in df.columns else "ID column missing")

# ------------------------------------------------------------------------------------------
# Actual execution exceptions
# IMPORTANT: error_codes_json is NOT counted as an API/runtime exception.
# ------------------------------------------------------------------------------------------

print("\nACTUAL EXCEPTION CHECK")
print("-" * 110)

if "exception" not in df.columns:
    print("WARNING: exception column not found.")
    exception_count = None
else:
    exceptions = df["exception"].fillna("").astype(str).str.strip()
    populated = ~exceptions.str.lower().isin(
        ["", "nan", "none", "null", "false"]
    )

    exception_count = int(populated.sum())

    print(f"Populated exception rows : {exception_count}/{len(df)}")

    if populated.any():
        print("\nException details:")
        print(
            df.loc[populated, ["id", "exception"]]
            .to_string(index=False)
        )

# ------------------------------------------------------------------------------------------
# LLM invocation and parsing
# ------------------------------------------------------------------------------------------

print("\nLLM EXECUTION CHECK")
print("-" * 110)

if "llm_invoked" in df.columns:
    invoked = df["llm_invoked"].fillna(False).astype(bool)
    print("LLM invoked :", int(invoked.sum()), "/", len(df))
else:
    invoked = None
    print("llm_invoked column missing")

if "parse_ok" in df.columns and invoked is not None:
    invoked_rows = df.loc[invoked]

    parse_success = (
        invoked_rows["parse_ok"]
        .fillna(False)
        .astype(bool)
        .sum()
    )

    print(
        "Parse OK on invoked rows :",
        int(parse_success),
        "/",
        len(invoked_rows)
    )

    if int(parse_success) != len(invoked_rows):
        print("\nRows with LLM invocation but unsuccessful parse:")

        bad = invoked_rows[
            ~invoked_rows["parse_ok"].fillna(False).astype(bool)
        ]

        cols = [
            c for c in
            ["id", "finish_reason", "rejection_stage",
             "error_codes_json", "exception"]
            if c in bad.columns
        ]

        print(bad[cols].to_string(index=False))

# ------------------------------------------------------------------------------------------
# Compact per-case view
# ------------------------------------------------------------------------------------------

print("\nPER-CASE EXECUTION SUMMARY")
print("-" * 110)

cols = [
    c for c in [
        "id",
        "expected_valid",
        "accepted",
        "decision_correct",
        "llm_invoked",
        "parse_ok",
        "finish_reason",
        "rejection_stage",
        "error_codes_json",
        "exception",
    ]
    if c in df.columns
]

print(df[cols].to_string(index=False))

# ------------------------------------------------------------------------------------------
# Assessment
# ------------------------------------------------------------------------------------------

row_ok = len(df) == 5

if invoked is not None and "parse_ok" in df.columns:
    parse_ok = (
        df.loc[invoked, "parse_ok"]
        .fillna(False)
        .astype(bool)
        .all()
    )
else:
    parse_ok = False

exception_ok = (
    exception_count == 0
    if exception_count is not None
    else False
)

print("\n" + "=" * 110)
print("A7 SMOKE ASSESSMENT")
print("=" * 110)

print("5 expected rows       :", "PASS" if row_ok else "FAIL")
print("No runtime exceptions :", "PASS" if exception_ok else "FAIL")
print("Invoked rows parse OK :", "PASS" if parse_ok else "FAIL")

if row_ok and exception_ok and parse_ok:
    print("\nA7 SMOKE STATUS: HEALTHY")
else:
    print("\nA7 SMOKE STATUS: REQUIRES INVESTIGATION")
    print("STOP — do not proceed to A8.")

print("\nNOTE:")
print("error_codes_json represents deterministic pipeline validation/rejection")
print("and is NOT treated as an API/runtime exception.")

print("\nSTEP 8 COMPLETE")
print("No model/API calls made.")
print("No benchmark cases executed.")
print("=" * 110)

STEP 8 — VALIDATE A7 SMOKE ARTIFACT

FILE
--------------------------------------------------------------------------------------------------------------
Path   : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/smoke/_smoke_A7.csv
Size   : 17206 bytes
SHA256 : 24e2b275fe7f3db135a4d3d41a60cf09a063d2f74c491d381a0ae845b4e45ff6

STRUCTURE
--------------------------------------------------------------------------------------------------------------
Rows    : 5
Columns : 41
IDs     : ['HO001', 'HO002', 'HO003', 'HO004', 'HO005']

ACTUAL EXCEPTION CHECK
--------------------------------------------------------------------------------------------------------------
Populated exception rows : 0/5

LLM EXECUTION CHECK
--------------------------------------------------------------------------------------------------------------
LLM invoked : 4 / 5
Parse OK on invoked rows : 4 / 4

PER-CASE EXECUTION SUMMARY
----------------------------------------

/tmp/ipykernel_1648/3305988281.py:97: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
/tmp/ipykernel_1648/3305988281.py:159: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


In [12]:
# ==========================================================================================
# STEP 9 — A8 SMOKE TEST
# 5 cases only
# Model: Prism-ML/Ternary-Bonsai-27B
# ==========================================================================================

import os
import sys
import subprocess
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

OUT = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "smoke"
    / "_smoke_A8.csv"
)

MODEL = "Prism-ML/Ternary-Bonsai-27B"

print("=" * 110)
print("STEP 9 — A8 SMOKE TEST")
print("=" * 110)

print("\nModel   :", MODEL)
print("Dataset :", DATASET)
print("Output  :", OUT)
print("Limit   : 5")

if OUT.exists():
    raise FileExistsError(
        f"STOP: Output already exists:\n{OUT}\n"
        "Do not overwrite existing evidence."
    )

if not os.environ.get("TOGETHER_API_KEY"):
    raise RuntimeError("STOP: TOGETHER_API_KEY is missing.")

cmd = [
    sys.executable,
    str(RUNNER),
    "--dataset", str(DATASET),
    "--out", str(OUT),
    "--model", MODEL,
    "--limit", "5",
]

env = os.environ.copy()
env["PYTHONPATH"] = str(MASTER)

print("\nStarting A8 5-case smoke test...")
print("-" * 110)

result = subprocess.run(
    cmd,
    cwd=str(MASTER),
    env=env,
    text=True,
    capture_output=True,
)

print("\nRETURN CODE:", result.returncode)

print("\nSTDOUT")
print("-" * 110)
print(result.stdout if result.stdout else "[no stdout]")

print("\nSTDERR")
print("-" * 110)
print(result.stderr if result.stderr else "[no stderr]")

print("\nOUTPUT FILE")
print("-" * 110)
print("Exists :", OUT.exists())

if OUT.exists():
    print("Size   :", OUT.stat().st_size, "bytes")

print("\n" + "=" * 110)

if result.returncode == 0 and OUT.exists():
    print("STEP 9 EXECUTION COMPLETED")
    print("A8 produced a smoke-test artifact.")
    print("Do NOT run the full benchmark yet.")
else:
    print("STEP 9 DID NOT COMPLETE SUCCESSFULLY")
    print("STOP HERE — do not proceed to A9.")

print("=" * 110)

STEP 9 — A8 SMOKE TEST

Model   : Prism-ML/Ternary-Bonsai-27B
Dataset : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL/experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3_heldout_benchmark_120_v1.csv
Output  : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/smoke/_smoke_A8.csv
Limit   : 5

Starting A8 5-case smoke test...
--------------------------------------------------------------------------------------------------------------

RETURN CODE: 0

STDOUT
--------------------------------------------------------------------------------------------------------------
[1/5] HO001
[2/5] HO002
[3/5] HO003
[4/5] HO004
[5/5] HO005

Rows written: 5
Output: /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/smoke/_smoke_A8.csv


STDERR
--------------------------------------------------------------------------------------------------

In [13]:
# ==========================================================================================
# STEP 10 — VALIDATE A8 SMOKE ARTIFACT
# Inspection only — NO model/API calls
# ==========================================================================================

from pathlib import Path
import pandas as pd
import hashlib

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

FILE = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "smoke"
    / "_smoke_A8.csv"
)

print("=" * 110)
print("STEP 10 — VALIDATE A8 SMOKE ARTIFACT")
print("=" * 110)

if not FILE.exists():
    raise FileNotFoundError(f"STOP: Smoke artifact not found:\n{FILE}")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

df = pd.read_csv(FILE)

print("\nFILE")
print("-" * 110)
print("Path   :", FILE)
print("Size   :", FILE.stat().st_size, "bytes")
print("SHA256 :", sha256(FILE))

print("\nSTRUCTURE")
print("-" * 110)
print("Rows    :", len(df))
print("Columns :", len(df.columns))
print("IDs     :", df["id"].tolist() if "id" in df.columns else "ID column missing")

# ------------------------------------------------------------------------------------------
# Actual runtime/API exceptions
# error_codes_json is deliberately NOT treated as an execution exception.
# ------------------------------------------------------------------------------------------

print("\nACTUAL EXCEPTION CHECK")
print("-" * 110)

if "exception" not in df.columns:
    print("WARNING: exception column not found.")
    exception_count = None
else:
    exceptions = df["exception"].fillna("").astype(str).str.strip()

    populated = ~exceptions.str.lower().isin(
        ["", "nan", "none", "null", "false"]
    )

    exception_count = int(populated.sum())

    print(f"Populated exception rows : {exception_count}/{len(df)}")

    if populated.any():
        print("\nException details:")
        print(
            df.loc[populated, ["id", "exception"]]
            .to_string(index=False)
        )

# ------------------------------------------------------------------------------------------
# LLM invocation / parse status
# ------------------------------------------------------------------------------------------

print("\nLLM EXECUTION CHECK")
print("-" * 110)

if "llm_invoked" in df.columns:
    invoked = df["llm_invoked"].eq(True)
    print("LLM invoked             :", int(invoked.sum()), "/", len(df))
else:
    invoked = None
    print("llm_invoked column missing")

if "parse_ok" in df.columns and invoked is not None:

    invoked_rows = df.loc[invoked]

    parse_success = invoked_rows["parse_ok"].eq(True).sum()

    print(
        "Parse OK on invoked rows:",
        int(parse_success),
        "/",
        len(invoked_rows)
    )

    if int(parse_success) != len(invoked_rows):

        print("\nRows with invocation but unsuccessful parse:")

        bad = invoked_rows[
            ~invoked_rows["parse_ok"].eq(True)
        ]

        cols = [
            c for c in [
                "id",
                "finish_reason",
                "rejection_stage",
                "error_codes_json",
                "exception"
            ]
            if c in bad.columns
        ]

        print(bad[cols].to_string(index=False))

# ------------------------------------------------------------------------------------------
# Compact case view
# ------------------------------------------------------------------------------------------

print("\nPER-CASE EXECUTION SUMMARY")
print("-" * 110)

cols = [
    c for c in [
        "id",
        "expected_valid",
        "accepted",
        "decision_correct",
        "llm_invoked",
        "parse_ok",
        "finish_reason",
        "rejection_stage",
        "error_codes_json",
        "exception",
    ]
    if c in df.columns
]

print(df[cols].to_string(index=False))

# ------------------------------------------------------------------------------------------
# Assessment
# ------------------------------------------------------------------------------------------

row_ok = len(df) == 5

if invoked is not None and "parse_ok" in df.columns:
    parse_ok_check = df.loc[invoked, "parse_ok"].eq(True).all()
else:
    parse_ok_check = False

exception_ok = (
    exception_count == 0
    if exception_count is not None
    else False
)

print("\n" + "=" * 110)
print("A8 SMOKE ASSESSMENT")
print("=" * 110)

print("5 expected rows       :", "PASS" if row_ok else "FAIL")
print("No runtime exceptions :", "PASS" if exception_ok else "FAIL")
print("Invoked rows parse OK :", "PASS" if parse_ok_check else "FAIL")

if row_ok and exception_ok and parse_ok_check:
    print("\nA8 SMOKE STATUS: HEALTHY")
else:
    print("\nA8 SMOKE STATUS: REQUIRES INVESTIGATION")
    print("STOP — do not proceed to A9.")

print("\nNOTE:")
print("error_codes_json contains deterministic pipeline validation/rejection information.")
print("It is not classified as an API/runtime exception.")

print("\nSTEP 10 COMPLETE")
print("No model/API calls made.")
print("No benchmark cases executed.")
print("=" * 110)

STEP 10 — VALIDATE A8 SMOKE ARTIFACT

FILE
--------------------------------------------------------------------------------------------------------------
Path   : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/smoke/_smoke_A8.csv
Size   : 19503 bytes
SHA256 : bc840b52adcd3e4960cd2d32ec9477816f4171427d043f17e6fc7ba328e2cb9a

STRUCTURE
--------------------------------------------------------------------------------------------------------------
Rows    : 5
Columns : 41
IDs     : ['HO001', 'HO002', 'HO003', 'HO004', 'HO005']

ACTUAL EXCEPTION CHECK
--------------------------------------------------------------------------------------------------------------
Populated exception rows : 0/5

LLM EXECUTION CHECK
--------------------------------------------------------------------------------------------------------------
LLM invoked             : 4 / 5
Parse OK on invoked rows: 4 / 4

PER-CASE EXECUTION SUMMARY
----------------------------

In [14]:
# ==========================================================================================
# STEP 11 — A9 SMOKE TEST
# 5 cases only
# Model: deepseek-ai/DeepSeek-V4.1-Flash
# ==========================================================================================

import os
import sys
import subprocess
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

OUT = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "smoke"
    / "_smoke_A9.csv"
)

MODEL = "deepseek-ai/DeepSeek-V4.1-Flash"

print("=" * 110)
print("STEP 11 — A9 SMOKE TEST")
print("=" * 110)

print("\nModel   :", MODEL)
print("Dataset :", DATASET)
print("Output  :", OUT)
print("Limit   : 5")

if OUT.exists():
    raise FileExistsError(
        f"STOP: Output already exists:\n{OUT}\n"
        "Do not overwrite existing evidence."
    )

if not os.environ.get("TOGETHER_API_KEY"):
    raise RuntimeError("STOP: TOGETHER_API_KEY is missing.")

cmd = [
    sys.executable,
    str(RUNNER),
    "--dataset", str(DATASET),
    "--out", str(OUT),
    "--model", MODEL,
    "--limit", "5",
]

env = os.environ.copy()
env["PYTHONPATH"] = str(MASTER)

print("\nStarting A9 5-case smoke test...")
print("-" * 110)

result = subprocess.run(
    cmd,
    cwd=str(MASTER),
    env=env,
    text=True,
    capture_output=True,
)

print("\nRETURN CODE:", result.returncode)

print("\nSTDOUT")
print("-" * 110)
print(result.stdout if result.stdout else "[no stdout]")

print("\nSTDERR")
print("-" * 110)
print(result.stderr if result.stderr else "[no stderr]")

print("\nOUTPUT FILE")
print("-" * 110)
print("Exists :", OUT.exists())

if OUT.exists():
    print("Size   :", OUT.stat().st_size, "bytes")

print("\n" + "=" * 110)

if result.returncode == 0 and OUT.exists():
    print("STEP 11 EXECUTION COMPLETED")
    print("A9 produced a smoke-test artifact.")
    print("Do NOT run the full benchmark yet.")
else:
    print("STEP 11 DID NOT COMPLETE SUCCESSFULLY")
    print("STOP HERE.")

print("=" * 110)

STEP 11 — A9 SMOKE TEST

Model   : deepseek-ai/DeepSeek-V4.1-Flash
Dataset : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL/experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3_heldout_benchmark_120_v1.csv
Output  : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/smoke/_smoke_A9.csv
Limit   : 5

Starting A9 5-case smoke test...
--------------------------------------------------------------------------------------------------------------

RETURN CODE: 0

STDOUT
--------------------------------------------------------------------------------------------------------------
[1/5] HO001
[2/5] HO002
[3/5] HO003
[4/5] HO004
[5/5] HO005

Rows written: 5
Output: /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/smoke/_smoke_A9.csv


STDERR
---------------------------------------------------------------------------------------------

In [15]:
# ==========================================================================================
# STEP 12 — VALIDATE A9 SMOKE ARTIFACT
# Inspection only — NO model/API calls
# ==========================================================================================

from pathlib import Path
import pandas as pd
import hashlib

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

FILE = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "smoke"
    / "_smoke_A9.csv"
)

print("=" * 110)
print("STEP 12 — VALIDATE A9 SMOKE ARTIFACT")
print("=" * 110)

if not FILE.exists():
    raise FileNotFoundError(f"STOP: Smoke artifact not found:\n{FILE}")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

df = pd.read_csv(FILE)

print("\nFILE")
print("-" * 110)
print("Path   :", FILE)
print("Size   :", FILE.stat().st_size, "bytes")
print("SHA256 :", sha256(FILE))

print("\nSTRUCTURE")
print("-" * 110)
print("Rows    :", len(df))
print("Columns :", len(df.columns))
print("IDs     :", df["id"].tolist() if "id" in df.columns else "ID column missing")

# ------------------------------------------------------------------------------------------
# Runtime/API exceptions
# ------------------------------------------------------------------------------------------

print("\nACTUAL EXCEPTION CHECK")
print("-" * 110)

if "exception" not in df.columns:
    print("WARNING: exception column not found.")
    exception_count = None
else:
    exceptions = df["exception"].fillna("").astype(str).str.strip()

    populated = ~exceptions.str.lower().isin(
        ["", "nan", "none", "null", "false"]
    )

    exception_count = int(populated.sum())

    print(f"Populated exception rows : {exception_count}/{len(df)}")

    if populated.any():
        print("\nException details:")
        print(
            df.loc[populated, ["id", "exception"]]
            .to_string(index=False)
        )

# ------------------------------------------------------------------------------------------
# LLM invocation / parsing
# ------------------------------------------------------------------------------------------

print("\nLLM EXECUTION CHECK")
print("-" * 110)

if "llm_invoked" in df.columns:
    invoked = df["llm_invoked"].eq(True)
    print("LLM invoked             :", int(invoked.sum()), "/", len(df))
else:
    invoked = None
    print("llm_invoked column missing")

if "parse_ok" in df.columns and invoked is not None:
    invoked_rows = df.loc[invoked]

    parse_success = invoked_rows["parse_ok"].eq(True).sum()

    print(
        "Parse OK on invoked rows:",
        int(parse_success),
        "/",
        len(invoked_rows)
    )

    if int(parse_success) != len(invoked_rows):
        print("\nRows with invocation but unsuccessful parse:")

        bad = invoked_rows[
            ~invoked_rows["parse_ok"].eq(True)
        ]

        cols = [
            c for c in [
                "id",
                "finish_reason",
                "rejection_stage",
                "error_codes_json",
                "exception"
            ]
            if c in bad.columns
        ]

        print(bad[cols].to_string(index=False))

# ------------------------------------------------------------------------------------------
# Compact per-case summary
# ------------------------------------------------------------------------------------------

print("\nPER-CASE EXECUTION SUMMARY")
print("-" * 110)

cols = [
    c for c in [
        "id",
        "expected_valid",
        "accepted",
        "decision_correct",
        "llm_invoked",
        "parse_ok",
        "finish_reason",
        "rejection_stage",
        "error_codes_json",
        "exception",
    ]
    if c in df.columns
]

print(df[cols].to_string(index=False))

# ------------------------------------------------------------------------------------------
# Assessment
# ------------------------------------------------------------------------------------------

row_ok = len(df) == 5

if invoked is not None and "parse_ok" in df.columns:
    parse_ok_check = df.loc[invoked, "parse_ok"].eq(True).all()
else:
    parse_ok_check = False

exception_ok = (
    exception_count == 0
    if exception_count is not None
    else False
)

print("\n" + "=" * 110)
print("A9 SMOKE ASSESSMENT")
print("=" * 110)

print("5 expected rows       :", "PASS" if row_ok else "FAIL")
print("No runtime exceptions :", "PASS" if exception_ok else "FAIL")
print("Invoked rows parse OK :", "PASS" if parse_ok_check else "FAIL")

if row_ok and exception_ok and parse_ok_check:
    print("\nA9 SMOKE STATUS: HEALTHY")
    print("\nALL FOUR TARGET ARMS HAVE NOW PASSED SMOKE TESTING.")
else:
    print("\nA9 SMOKE STATUS: REQUIRES INVESTIGATION")
    print("STOP — do not begin full reruns.")

print("\nNOTE:")
print("error_codes_json is deterministic pipeline validation/rejection information,")
print("not an API/runtime exception.")

print("\nSTEP 12 COMPLETE")
print("No model/API calls made.")
print("No benchmark cases executed.")
print("=" * 110)

STEP 12 — VALIDATE A9 SMOKE ARTIFACT

FILE
--------------------------------------------------------------------------------------------------------------
Path   : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/smoke/_smoke_A9.csv
Size   : 18662 bytes
SHA256 : a16aa70c44f039c82b63d18ccd4036d2e3213a95febb4770280c1f4dccbb1f6d

STRUCTURE
--------------------------------------------------------------------------------------------------------------
Rows    : 5
Columns : 41
IDs     : ['HO001', 'HO002', 'HO003', 'HO004', 'HO005']

ACTUAL EXCEPTION CHECK
--------------------------------------------------------------------------------------------------------------
Populated exception rows : 0/5

LLM EXECUTION CHECK
--------------------------------------------------------------------------------------------------------------
LLM invoked             : 4 / 5
Parse OK on invoked rows: 4 / 4

PER-CASE EXECUTION SUMMARY
----------------------------

In [16]:
# ==========================================================================================
# STEP 13 — FULL CONFIRMATORY RERUN: A2
# 120-case frozen E3-v3 benchmark
# ==========================================================================================

import os
import sys
import subprocess
from pathlib import Path
from datetime import datetime, timezone

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

OUT = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "raw"
    / "A2_llama-3.3-70b_rerun.csv"
)

LOG = (
    BASE
    / "rerun-missing-artifacts"
    / "logs"
    / "A2_llama-3.3-70b_rerun.log"
)

MODEL = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

print("=" * 110)
print("STEP 13 — FULL A2 CONFIRMATORY RERUN")
print("=" * 110)

print("\nArm     : A2")
print("Model   :", MODEL)
print("Dataset :", DATASET)
print("Output  :", OUT)
print("Log     :", LOG)
print("Cases   : 120")

# ------------------------------------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------------------------------------

if OUT.exists():
    raise FileExistsError(
        f"STOP: Full A2 rerun output already exists:\n{OUT}\n"
        "Existing evidence will NOT be overwritten."
    )

if LOG.exists():
    raise FileExistsError(
        f"STOP: A2 rerun log already exists:\n{LOG}\n"
        "Existing evidence will NOT be overwritten."
    )

if not os.environ.get("TOGETHER_API_KEY"):
    raise RuntimeError("STOP: TOGETHER_API_KEY is missing.")

if not RUNNER.exists():
    raise FileNotFoundError("STOP: Canonical runner missing.")

if not DATASET.exists():
    raise FileNotFoundError("STOP: Frozen benchmark missing.")

# ------------------------------------------------------------------------------------------
# Command
# ------------------------------------------------------------------------------------------

cmd = [
    sys.executable,
    str(RUNNER),
    "--dataset", str(DATASET),
    "--out", str(OUT),
    "--model", MODEL,
]

env = os.environ.copy()
env["PYTHONPATH"] = str(MASTER)

started = datetime.now(timezone.utc)

print("\nStarted UTC:", started.isoformat())
print("\nRunning A2...")
print("-" * 110)

result = subprocess.run(
    cmd,
    cwd=str(MASTER),
    env=env,
    text=True,
    capture_output=True,
)

finished = datetime.now(timezone.utc)

# ------------------------------------------------------------------------------------------
# Persist execution log
# ------------------------------------------------------------------------------------------

log_text = f"""STEP 13 — FULL A2 CONFIRMATORY RERUN

Arm: A2
Model: {MODEL}
Dataset: {DATASET}
Output: {OUT}

Started UTC: {started.isoformat()}
Finished UTC: {finished.isoformat()}
Return code: {result.returncode}

================ STDOUT ================

{result.stdout}

================ STDERR ================

{result.stderr}
"""

LOG.write_text(log_text, encoding="utf-8")

# ------------------------------------------------------------------------------------------
# Display result
# ------------------------------------------------------------------------------------------

print("\nRETURN CODE:", result.returncode)

print("\nSTDOUT")
print("-" * 110)
print(result.stdout if result.stdout else "[no stdout]")

print("\nSTDERR")
print("-" * 110)
print(result.stderr if result.stderr else "[no stderr]")

print("\nOUTPUT")
print("-" * 110)
print("Exists :", OUT.exists())

if OUT.exists():
    print("Size   :", OUT.stat().st_size, "bytes")

print("\nLOG")
print("-" * 110)
print("Exists :", LOG.exists())
print("Path   :", LOG)

print("\nFinished UTC:", finished.isoformat())

print("\n" + "=" * 110)

if result.returncode == 0 and OUT.exists():
    print("STEP 13 EXECUTION COMPLETED")
    print("A2 full rerun artifact was created.")
    print()
    print("IMPORTANT: A2 is NOT yet accepted as a valid recovery run.")
    print("We must inspect all 120 rows for exceptions and completeness.")
    print("DO NOT run A7 yet.")
else:
    print("STEP 13 DID NOT COMPLETE SUCCESSFULLY")
    print("STOP HERE.")
    print("DO NOT run A7.")

print("=" * 110)

STEP 13 — FULL A2 CONFIRMATORY RERUN

Arm     : A2
Model   : meta-llama/Llama-3.3-70B-Instruct-Turbo
Dataset : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL/experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3_heldout_benchmark_120_v1.csv
Output  : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/raw/A2_llama-3.3-70b_rerun.csv
Log     : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/logs/A2_llama-3.3-70b_rerun.log
Cases   : 120

Started UTC: 2026-10-03T08:37:45.874791+00:00

Running A2...
--------------------------------------------------------------------------------------------------------------

RETURN CODE: 0

STDOUT
--------------------------------------------------------------------------------------------------------------
[1/120] HO001
[2/120] HO002
[3/120] HO003
[4/120] HO004
[5/120] HO005
[6/120] HO006
[7/120] HO007


In [17]:
# ==========================================================================================
# STEP 14 — VALIDATE FULL A2 RERUN ARTIFACT
# Inspection only — NO model/API calls
# ==========================================================================================

from pathlib import Path
import pandas as pd
import hashlib

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

FILE = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "raw"
    / "A2_llama-3.3-70b_rerun.csv"
)

print("=" * 110)
print("STEP 14 — VALIDATE FULL A2 RERUN ARTIFACT")
print("=" * 110)

if not FILE.exists():
    raise FileNotFoundError(f"STOP: A2 rerun artifact not found:\n{FILE}")

if not DATASET.exists():
    raise FileNotFoundError(f"STOP: Frozen benchmark not found:\n{DATASET}")


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


df = pd.read_csv(FILE)
benchmark = pd.read_csv(DATASET)

# ==========================================================================================
# 1. FILE INTEGRITY
# ==========================================================================================

print("\nFILE")
print("-" * 110)
print("Path   :", FILE)
print("Size   :", FILE.stat().st_size, "bytes")
print("SHA256 :", sha256(FILE))

print("\nFROZEN DATASET")
print("-" * 110)
print("Rows   :", len(benchmark))
print("SHA256 :", sha256(DATASET))

# ==========================================================================================
# 2. STRUCTURE
# ==========================================================================================

print("\nSTRUCTURE")
print("-" * 110)
print("Result rows    :", len(df))
print("Result columns :", len(df.columns))

row_ok = len(df) == 120

print("Expected rows  : 120")
print("Row count      :", "PASS" if row_ok else "FAIL")

# ==========================================================================================
# 3. ID INTEGRITY
# ==========================================================================================

print("\nID INTEGRITY")
print("-" * 110)

id_checks_possible = "id" in df.columns and "id" in benchmark.columns

if id_checks_possible:

    result_ids = df["id"].astype(str)
    benchmark_ids = benchmark["id"].astype(str)

    duplicate_mask = result_ids.duplicated(keep=False)
    duplicate_count = int(duplicate_mask.sum())

    result_set = set(result_ids)
    benchmark_set = set(benchmark_ids)

    missing_ids = sorted(benchmark_set - result_set)
    unexpected_ids = sorted(result_set - benchmark_set)

    order_match = result_ids.tolist() == benchmark_ids.tolist()

    print("Unique result IDs :", result_ids.nunique(), "/", len(df))
    print("Duplicate rows    :", duplicate_count)
    print("Missing IDs       :", len(missing_ids))
    print("Unexpected IDs    :", len(unexpected_ids))
    print("Dataset order     :", "MATCH" if order_match else "DIFFERENT")

    if duplicate_count:
        print("\nDuplicate IDs:")
        print(df.loc[duplicate_mask, ["id"]].to_string(index=False))

    if missing_ids:
        print("\nMissing IDs:")
        print(missing_ids)

    if unexpected_ids:
        print("\nUnexpected IDs:")
        print(unexpected_ids)

    id_ok = (
        duplicate_count == 0
        and len(missing_ids) == 0
        and len(unexpected_ids) == 0
        and len(df) == len(benchmark)
    )

else:
    print("FAIL: id column missing from result or benchmark.")
    id_ok = False
    order_match = False

# ==========================================================================================
# 4. ACTUAL RUNTIME/API EXCEPTIONS
# ==========================================================================================

print("\nACTUAL EXCEPTION CHECK")
print("-" * 110)

if "exception" not in df.columns:

    print("FAIL: exception column not found.")
    exception_count = None
    exception_ok = False

else:

    exceptions = df["exception"].fillna("").astype(str).str.strip()

    populated = ~exceptions.str.lower().isin(
        ["", "nan", "none", "null", "false"]
    )

    exception_count = int(populated.sum())
    exception_ok = exception_count == 0

    print(
        "Populated exception rows :",
        f"{exception_count}/{len(df)}"
    )

    if populated.any():

        print("\nEXCEPTION DETAILS")
        print("-" * 110)

        cols = [
            c for c in [
                "id",
                "llm_invoked",
                "parse_ok",
                "finish_reason",
                "rejection_stage",
                "exception"
            ]
            if c in df.columns
        ]

        print(df.loc[populated, cols].to_string(index=False))

# ==========================================================================================
# 5. LLM INVOCATION / PARSE CHECK
# ==========================================================================================

print("\nLLM EXECUTION CHECK")
print("-" * 110)

if "llm_invoked" in df.columns:

    invoked = df["llm_invoked"].eq(True)

    invoked_count = int(invoked.sum())
    pre_llm_count = len(df) - invoked_count

    print("LLM invoked      :", invoked_count, "/", len(df))
    print("Pre-LLM no-call  :", pre_llm_count, "/", len(df))

else:

    invoked = None
    invoked_count = None
    print("FAIL: llm_invoked column missing.")

if invoked is not None and "parse_ok" in df.columns:

    invoked_rows = df.loc[invoked]

    parse_success_mask = invoked_rows["parse_ok"].eq(True)

    parse_success = int(parse_success_mask.sum())
    parse_failure = len(invoked_rows) - parse_success

    print("Parse OK         :", parse_success, "/", len(invoked_rows))
    print("Parse failures   :", parse_failure)

    if parse_failure:

        print("\nINVOKED ROWS WITH UNSUCCESSFUL PARSE")
        print("-" * 110)

        bad = invoked_rows.loc[~parse_success_mask]

        cols = [
            c for c in [
                "id",
                "expected_valid",
                "accepted",
                "finish_reason",
                "rejection_stage",
                "error_codes_json",
                "exception"
            ]
            if c in bad.columns
        ]

        print(bad[cols].to_string(index=False))

    parse_ok_check = parse_failure == 0

else:

    print("FAIL: parse_ok column missing or invocation status unavailable.")
    parse_success = None
    parse_failure = None
    parse_ok_check = False

# ==========================================================================================
# 6. BASIC DECISION FIELD SANITY
# ==========================================================================================

print("\nDECISION FIELD SANITY")
print("-" * 110)

required_decision_columns = [
    "expected_valid",
    "accepted",
    "decision_correct"
]

missing_decision_columns = [
    c for c in required_decision_columns
    if c not in df.columns
]

if missing_decision_columns:

    print("Missing columns:", missing_decision_columns)
    decision_fields_ok = False

else:

    print("expected_valid populated :", int(df["expected_valid"].notna().sum()), "/", len(df))
    print("accepted populated       :", int(df["accepted"].notna().sum()), "/", len(df))
    print("decision_correct populated:", int(df["decision_correct"].notna().sum()), "/", len(df))

    decision_fields_ok = all(
        df[c].notna().all()
        for c in required_decision_columns
    )

    print(
        "Decision fields complete :",
        "PASS" if decision_fields_ok else "FAIL"
    )

# ==========================================================================================
# 7. FINAL TECHNICAL ASSESSMENT
# ==========================================================================================

print("\n" + "=" * 110)
print("A2 FULL-RUN TECHNICAL ASSESSMENT")
print("=" * 110)

print("120 expected rows       :", "PASS" if row_ok else "FAIL")
print("Benchmark ID coverage   :", "PASS" if id_ok else "FAIL")
print("Benchmark order         :", "PASS" if order_match else "FAIL")
print("No runtime exceptions   :", "PASS" if exception_ok else "FAIL")
print("Invoked rows parse OK   :", "PASS" if parse_ok_check else "FAIL")
print("Decision fields complete:", "PASS" if decision_fields_ok else "FAIL")

technical_valid = all([
    row_ok,
    id_ok,
    order_match,
    exception_ok,
    parse_ok_check,
    decision_fields_ok,
])

if technical_valid:

    print("\nA2 FULL RERUN STATUS: TECHNICALLY VALID")
    print("The raw A2 recovery artifact is complete and suitable for metric computation.")
    print()
    print("Do NOT run A7 yet.")
    print("Next: compute A2 rerun metrics and compare them with the frozen A2 aggregate.")

else:

    print("\nA2 FULL RERUN STATUS: REQUIRES INVESTIGATION")
    print("STOP — do not compute confirmatory metrics and do not run A7.")

print("\nIMPORTANT:")
print("This validation does NOT replace or modify yesterday's frozen A2 result.")
print("Today's file remains separate confirmatory/recovery evidence.")

print("\nSTEP 14 COMPLETE")
print("No model/API calls made.")
print("=" * 110)

STEP 14 — VALIDATE FULL A2 RERUN ARTIFACT

FILE
--------------------------------------------------------------------------------------------------------------
Path   : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/raw/A2_llama-3.3-70b_rerun.csv
Size   : 375671 bytes
SHA256 : 5856caf2437d528c510f5432436b0f9ce101d4360258d14a902598ec75dd6e28

FROZEN DATASET
--------------------------------------------------------------------------------------------------------------
Rows   : 120
SHA256 : 84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de

STRUCTURE
--------------------------------------------------------------------------------------------------------------
Result rows    : 120
Result columns : 41
Expected rows  : 120
Row count      : PASS

ID INTEGRITY
--------------------------------------------------------------------------------------------------------------
Unique result IDs : 120 / 120
Duplicate rows    : 0
Missing

In [18]:
# ==========================================================================================
# STEP 15 — COMPUTE A2 RERUN METRICS AND COMPARE WITH FROZEN A2
# Analysis only — NO model/API calls
# ==========================================================================================

from pathlib import Path
import pandas as pd
import json
import hashlib
from datetime import datetime, timezone

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

RAW = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "raw"
    / "A2_llama-3.3-70b_rerun.csv"
)

METRICS_OUT = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "metrics"
    / "A2_llama-3.3-70b_rerun_metrics_v1.json"
)

print("=" * 110)
print("STEP 15 — A2 RERUN METRICS + FROZEN COMPARISON")
print("=" * 110)

if not RAW.exists():
    raise FileNotFoundError(f"STOP: A2 raw artifact missing:\n{RAW}")

if METRICS_OUT.exists():
    raise FileExistsError(
        f"STOP: Metrics artifact already exists:\n{METRICS_OUT}\n"
        "Existing evidence will NOT be overwritten."
    )


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


df = pd.read_csv(RAW)

# ==========================================================================================
# 1. CONFUSION MATRIX
#
# Positive class = INVALID / REJECT
#
# TP = expected invalid and rejected
# FP = expected valid and rejected
# FN = expected invalid and accepted
# TN = expected valid and accepted
# ==========================================================================================

expected_valid = df["expected_valid"].eq(True)
accepted = df["accepted"].eq(True)

expected_invalid = ~expected_valid
rejected = ~accepted

TP = int((expected_invalid & rejected).sum())
FP = int((expected_valid & rejected).sum())
FN = int((expected_invalid & accepted).sum())
TN = int((expected_valid & accepted).sum())

N = len(df)

expected_valid_n = int(expected_valid.sum())
expected_invalid_n = int(expected_invalid.sum())

# ==========================================================================================
# 2. PRIMARY METRICS
# ==========================================================================================

accuracy = (TP + TN) / N

# Valid Acceptance Rate:
# valid cases correctly accepted / all expected-valid cases
VAR = TN / expected_valid_n

# Invalid Rejection Rate:
# invalid cases correctly rejected / all expected-invalid cases
IRR = TP / expected_invalid_n

precision = TP / (TP + FP) if (TP + FP) else float("nan")
recall = TP / (TP + FN) if (TP + FN) else float("nan")

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall)
    else float("nan")
)

decision_correct_count = int(df["decision_correct"].eq(True).sum())

# ==========================================================================================
# 3. EXECUTION COUNTS
# ==========================================================================================

llm_invoked = int(df["llm_invoked"].eq(True).sum())
pre_llm_no_call = N - llm_invoked

parse_ok = int(
    df.loc[df["llm_invoked"].eq(True), "parse_ok"]
    .eq(True)
    .sum()
)

accepted_count = int(accepted.sum())
rejected_count = int(rejected.sum())

# ==========================================================================================
# 4. CONTAINMENT VIOLATION CHECK
#
# For this experiment, use the runner's decision/descriptor evidence rather than inventing
# a new containment definition.
#
# A containment violation is counted conservatively here as an invalid case that was
# accepted AND produced a non-empty descriptor.
#
# This is only an artifact-level check for this rerun.
# ==========================================================================================

def has_nonempty_descriptor(value):
    if pd.isna(value):
        return False

    s = str(value).strip()

    if s.lower() in ["", "nan", "none", "null", "{}", "[]"]:
        return False

    return True


if "descriptor_json" in df.columns:

    descriptor_present = df["descriptor_json"].apply(
        has_nonempty_descriptor
    )

    containment_mask = (
        expected_invalid
        & accepted
        & descriptor_present
    )

    containment_violations = int(containment_mask.sum())

else:

    descriptor_present = pd.Series(False, index=df.index)
    containment_mask = pd.Series(False, index=df.index)
    containment_violations = None

# ==========================================================================================
# 5. FROZEN A2 TARGET
# ==========================================================================================

FROZEN = {
    "accuracy_pct": 83.33,
    "var_pct": 82.76,
    "irr_pct": 83.87,
    "containment_violations": 0,
}

today = {
    "accuracy_pct": accuracy * 100,
    "var_pct": VAR * 100,
    "irr_pct": IRR * 100,
}

delta = {
    "accuracy_pp": today["accuracy_pct"] - FROZEN["accuracy_pct"],
    "var_pp": today["var_pct"] - FROZEN["var_pct"],
    "irr_pp": today["irr_pct"] - FROZEN["irr_pct"],
}

# ==========================================================================================
# 6. DISPLAY RESULTS
# ==========================================================================================

print("\nCONFUSION MATRIX")
print("-" * 110)

print("Positive class = INVALID / REJECT")
print()
print("TP :", TP)
print("FP :", FP)
print("FN :", FN)
print("TN :", TN)
print()
print("Total :", TP + FP + FN + TN)

print("\nDATASET COUNTS")
print("-" * 110)

print("Expected valid   :", expected_valid_n)
print("Expected invalid :", expected_invalid_n)
print("Accepted         :", accepted_count)
print("Rejected         :", rejected_count)

print("\nTODAY'S A2 METRICS")
print("-" * 110)

print(f"Accuracy : {accuracy * 100:.2f}%")
print(f"VAR      : {VAR * 100:.2f}%")
print(f"IRR      : {IRR * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall   : {recall * 100:.2f}%")
print(f"F1       : {f1 * 100:.2f}%")

print("\nDECISION-CORRECT CROSS-CHECK")
print("-" * 110)

print("Confusion-matrix correct :", TP + TN)
print("decision_correct == True :", decision_correct_count)

decision_crosscheck_ok = (
    (TP + TN) == decision_correct_count
)

print(
    "Cross-check              :",
    "PASS" if decision_crosscheck_ok else "FAIL"
)

print("\nEXECUTION COUNTS")
print("-" * 110)

print("LLM invoked     :", llm_invoked)
print("Pre-LLM no-call :", pre_llm_no_call)
print("Parse OK        :", parse_ok, "/", llm_invoked)

print("\nCONTAINMENT CHECK")
print("-" * 110)

print(
    "Observed containment violations :",
    containment_violations
)

if containment_violations:
    print("\nPotential containment-violation rows:")
    cols = [
        c for c in [
            "id",
            "expected_valid",
            "accepted",
            "decision_correct",
            "descriptor_json"
        ]
        if c in df.columns
    ]
    print(df.loc[containment_mask, cols].to_string(index=False))

print("\nFROZEN A2 vs TODAY'S CONFIRMATORY RERUN")
print("-" * 110)

print(
    f"{'Metric':<22}"
    f"{'Frozen A2':>15}"
    f"{'Today':>15}"
    f"{'Delta':>15}"
)

print("-" * 67)

print(
    f"{'Accuracy':<22}"
    f"{FROZEN['accuracy_pct']:>14.2f}%"
    f"{today['accuracy_pct']:>14.2f}%"
    f"{delta['accuracy_pp']:>+14.2f} pp"
)

print(
    f"{'VAR':<22}"
    f"{FROZEN['var_pct']:>14.2f}%"
    f"{today['var_pct']:>14.2f}%"
    f"{delta['var_pp']:>+14.2f} pp"
)

print(
    f"{'IRR':<22}"
    f"{FROZEN['irr_pct']:>14.2f}%"
    f"{today['irr_pct']:>14.2f}%"
    f"{delta['irr_pp']:>+14.2f} pp"
)

print(
    f"{'Containment violations':<22}"
    f"{FROZEN['containment_violations']:>15}"
    f"{str(containment_violations):>15}"
)

# ==========================================================================================
# 7. SAVE METRICS ARTIFACT
# ==========================================================================================

record = {
    "experiment": "Experiment A confirmatory missing-artifact recovery rerun",
    "arm": "A2",
    "model": "meta-llama/Llama-3.3-70B-Instruct-Turbo",

    "evidence_status": (
        "confirmatory recovery evidence only; "
        "does not replace frozen Experiment A result"
    ),

    "raw_artifact": str(RAW),
    "raw_artifact_sha256": sha256(RAW),

    "n": N,

    "expected_valid": expected_valid_n,
    "expected_invalid": expected_invalid_n,

    "confusion_matrix_invalid_reject_positive": {
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
    },

    "metrics": {
        "accuracy": accuracy,
        "accuracy_pct": accuracy * 100,
        "VAR": VAR,
        "VAR_pct": VAR * 100,
        "IRR": IRR,
        "IRR_pct": IRR * 100,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    },

    "execution": {
        "llm_invoked": llm_invoked,
        "pre_llm_no_call": pre_llm_no_call,
        "parse_ok": parse_ok,
        "runtime_exceptions": 0,
    },

    "containment": {
        "observed_violations": containment_violations,
        "artifact_level_check": (
            "expected-invalid AND accepted AND "
            "non-empty descriptor_json"
        ),
    },

    "frozen_A2_reference": FROZEN,

    "delta_percentage_points": delta,

    "decision_correct_crosscheck": {
        "confusion_matrix_correct": TP + TN,
        "decision_correct_true": decision_correct_count,
        "passed": decision_crosscheck_ok,
    },

    "generated_utc": datetime.now(timezone.utc).isoformat(),
}

METRICS_OUT.write_text(
    json.dumps(record, indent=2),
    encoding="utf-8"
)

print("\nMETRICS ARTIFACT")
print("-" * 110)

print("Path   :", METRICS_OUT)
print("SHA256 :", sha256(METRICS_OUT))

print("\n" + "=" * 110)
print("STEP 15 COMPLETE")
print("=" * 110)

print("No model/API calls made.")
print("Frozen Experiment A evidence was NOT modified.")
print("Do NOT run A7 yet.")
print("=" * 110)

STEP 15 — A2 RERUN METRICS + FROZEN COMPARISON

CONFUSION MATRIX
--------------------------------------------------------------------------------------------------------------
Positive class = INVALID / REJECT

TP : 51
FP : 10
FN : 11
TN : 48

Total : 120

DATASET COUNTS
--------------------------------------------------------------------------------------------------------------
Expected valid   : 58
Expected invalid : 62
Accepted         : 59
Rejected         : 61

TODAY'S A2 METRICS
--------------------------------------------------------------------------------------------------------------
Accuracy : 82.50%
VAR      : 82.76%
IRR      : 82.26%
Precision: 83.61%
Recall   : 82.26%
F1       : 82.93%

DECISION-CORRECT CROSS-CHECK
--------------------------------------------------------------------------------------------------------------
Confusion-matrix correct : 99
decision_correct == True : 99
Cross-check              : PASS

EXECUTION COUNTS
---------------------------------------

In [19]:
# ==========================================================================================
# STEP 16 — RECOVER CANONICAL EXPERIMENT-A CONTAINMENT DEFINITION
# Inspection only — NO model/API calls — NO file modifications
# ==========================================================================================

from pathlib import Path
import re

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

SEARCH_ROOTS = [
    BASE / "journal_experiments_persistent" / "experiment_A_model_sweep",
    BASE / "experiment-1-v2-together_FINAL",
]

print("=" * 110)
print("STEP 16 — RECOVER CANONICAL EXPERIMENT-A CONTAINMENT DEFINITION")
print("=" * 110)

print("\nIMPORTANT")
print("-" * 110)
print("Step 15 used a NEW proxy definition:")
print("  expected-invalid AND accepted AND non-empty descriptor_json")
print()
print("That proxy is NOT assumed to equal the frozen Experiment-A")
print("containment-violation definition.")
print()
print("We are now locating the original definition/evidence.")
print("No files will be modified.")

# ==========================================================================================
# 1. INSPECT CANONICAL RUNNER
# ==========================================================================================

print("\n" + "=" * 110)
print("1. CANONICAL RUNNER — RELEVANT LINES")
print("=" * 110)

if not RUNNER.exists():
    raise FileNotFoundError(f"STOP: canonical runner missing:\n{RUNNER}")

runner_lines = RUNNER.read_text(
    encoding="utf-8",
    errors="replace"
).splitlines()

patterns = [
    r"contain",
    r"descriptor",
    r"accepted",
    r"decision_correct",
    r"expected_valid",
    r"error_codes",
]

matched_line_numbers = set()

for i, line in enumerate(runner_lines, start=1):
    lower = line.lower()

    if any(re.search(p, lower) for p in patterns):
        matched_line_numbers.add(i)

# Print context around relevant lines.
printed = set()

for line_no in sorted(matched_line_numbers):

    start = max(1, line_no - 2)
    end = min(len(runner_lines), line_no + 2)

    block_key = (start, end)

    if block_key in printed:
        continue

    printed.add(block_key)

    print(f"\n--- runner lines {start}-{end} ---")

    for n in range(start, end + 1):
        print(f"{n:04d}: {runner_lines[n-1]}")

# ==========================================================================================
# 2. SEARCH SURVIVING EXPERIMENT-A TEXT/JSON/CSV/MARKDOWN ARTIFACTS
# ==========================================================================================

print("\n" + "=" * 110)
print("2. SURVIVING EXPERIMENT-A ARTIFACT SEARCH")
print("=" * 110)

search_terms = [
    "containment",
    "containment_violation",
    "containment violations",
    "violation",
]

candidate_extensions = {
    ".json",
    ".md",
    ".txt",
    ".csv",
    ".log",
    ".py",
}

hits = []

for root in SEARCH_ROOTS:

    print("\nSearching:", root)

    if not root.exists():
        print("  [root does not exist]")
        continue

    for path in root.rglob("*"):

        if not path.is_file():
            continue

        if path.suffix.lower() not in candidate_extensions:
            continue

        try:
            text = path.read_text(
                encoding="utf-8",
                errors="replace"
            )
        except Exception:
            continue

        lower = text.lower()

        found_terms = [
            term
            for term in search_terms
            if term.lower() in lower
        ]

        if found_terms:
            hits.append((path, found_terms, text))

print("\nFiles containing containment/violation terminology:", len(hits))

# ==========================================================================================
# 3. PRINT RELEVANT CONTEXT
# ==========================================================================================

print("\n" + "=" * 110)
print("3. MATCHED ARTIFACT CONTEXT")
print("=" * 110)

MAX_FILES = 40
MAX_MATCHES_PER_FILE = 12

for file_index, (path, found_terms, text) in enumerate(
    hits[:MAX_FILES],
    start=1
):

    print("\n" + "-" * 110)
    print(f"[{file_index}] {path}")
    print("Matched terms:", found_terms)
    print("-" * 110)

    lines = text.splitlines()

    match_numbers = []

    for i, line in enumerate(lines, start=1):

        lower = line.lower()

        if any(term.lower() in lower for term in search_terms):
            match_numbers.append(i)

    for line_no in match_numbers[:MAX_MATCHES_PER_FILE]:

        start = max(1, line_no - 3)
        end = min(len(lines), line_no + 3)

        print(f"\nContext lines {start}-{end}")

        for n in range(start, end + 1):
            print(f"{n:04d}: {lines[n-1]}")

# ==========================================================================================
# 4. SUMMARY
# ==========================================================================================

print("\n" + "=" * 110)
print("STEP 16 SUMMARY")
print("=" * 110)

print("Canonical runner inspected :", RUNNER)
print("Matched artifact files     :", len(hits))

if len(hits) == 0:
    print()
    print("No surviving artifact containing containment terminology was found.")
    print("STOP — we must not invent the canonical definition.")
else:
    print()
    print("Review the printed evidence above.")
    print("We will derive the canonical containment check ONLY if the surviving")
    print("artifacts explicitly support it.")

print()
print("NO model/API calls made.")
print("NO files modified.")
print("DO NOT run A7 yet.")
print("=" * 110)

STEP 16 — RECOVER CANONICAL EXPERIMENT-A CONTAINMENT DEFINITION

IMPORTANT
--------------------------------------------------------------------------------------------------------------
Step 15 used a NEW proxy definition:
  expected-invalid AND accepted AND non-empty descriptor_json

That proxy is NOT assumed to equal the frozen Experiment-A
containment-violation definition.

We are now locating the original definition/evidence.
No files will be modified.

1. CANONICAL RUNNER — RELEVANT LINES

--- runner lines 16-20 ---
0016:     5. historical runner remains unchanged.
0017: 
0018: No algorithmic CIR/LLM/validation/descriptor logic is intentionally
0019: changed in this development harness.
0020: """

--- runner lines 57-61 ---
0057: from src.validator import validate_cir
0058: from src.standards.mapper import map_standards
0059: from src.descriptor import compile_descriptor
0060: 
0061: 

--- runner lines 173-177 ---
0173: 
0174: 
0175: def error_codes(obj):
0176:     d = obj_dict(ob

In [20]:
# ==========================================================================================
# STEP 17 — CANONICAL A2 CONTAINMENT RECHECK
#
# Canonical Experiment-A definition:
#   containment violation =
#       accepted == False
#       AND descriptor_json is present
#
# NO model/API calls.
# NO raw files modified.
# STEP 15 artifact is preserved unchanged.
# ==========================================================================================

from pathlib import Path
import pandas as pd
import hashlib
import json
from datetime import datetime, timezone

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

RAW = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "raw"
    / "A2_llama-3.3-70b_rerun.csv"
)

OUT = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "metrics"
    / "A2_canonical_containment_recheck_v1.json"
)

print("=" * 110)
print("STEP 17 — CANONICAL A2 CONTAINMENT RECHECK")
print("=" * 110)

if not RAW.exists():
    raise FileNotFoundError(
        f"STOP: A2 raw artifact missing:\n{RAW}"
    )

if OUT.exists():
    raise FileExistsError(
        f"STOP: Corrected evidence artifact already exists:\n{OUT}\n"
        "Existing evidence will NOT be overwritten."
    )


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def descriptor_present(value):
    """
    True only when descriptor_json contains an actual
    non-empty descriptor representation.
    """

    if pd.isna(value):
        return False

    s = str(value).strip()

    if s.lower() in {
        "",
        "nan",
        "none",
        "null",
        "{}",
        "[]"
    }:
        return False

    return True


df = pd.read_csv(RAW)

# ==========================================================================================
# 1. BASIC COUNTS
# ==========================================================================================

accepted = df["accepted"].eq(True)
rejected = df["accepted"].eq(False)

descriptor_mask = df["descriptor_json"].apply(
    descriptor_present
)

accepted_count = int(accepted.sum())
rejected_count = int(rejected.sum())
descriptor_count = int(descriptor_mask.sum())

# ==========================================================================================
# 2. CANONICAL CONTAINMENT VIOLATIONS
# ==========================================================================================

containment_mask = rejected & descriptor_mask

containment_violations = int(
    containment_mask.sum()
)

# Also useful as an integrity check:
accepted_without_descriptor_mask = (
    accepted & ~descriptor_mask
)

accepted_without_descriptor = int(
    accepted_without_descriptor_mask.sum()
)

# ==========================================================================================
# 3. DISPLAY
# ==========================================================================================

print("\nCANONICAL DEFINITION")
print("-" * 110)

print(
    "Containment violation = "
    "accepted == False AND descriptor_json is present"
)

print("\nARTIFACT COUNTS")
print("-" * 110)

print("Total cases                 :", len(df))
print("Accepted                    :", accepted_count)
print("Rejected                    :", rejected_count)
print("Descriptors present         :", descriptor_count)
print(
    "Accepted without descriptor :",
    accepted_without_descriptor
)

print("\nCANONICAL CONTAINMENT RESULT")
print("-" * 110)

print(
    "Rejected + descriptor present :",
    containment_violations
)

if containment_violations:

    print("\nCONTAINMENT VIOLATION ROWS")
    print("-" * 110)

    cols = [
        c for c in [
            "id",
            "expected_valid",
            "accepted",
            "decision_correct",
            "rejection_stage",
            "error_codes_json",
            "descriptor_json"
        ]
        if c in df.columns
    ]

    print(
        df.loc[
            containment_mask,
            cols
        ].to_string(index=False)
    )

# ==========================================================================================
# 4. RECOMPUTE DECISION METRICS FOR CONSISTENCY
# ==========================================================================================

expected_valid = df["expected_valid"].eq(True)
expected_invalid = ~expected_valid

TP = int((expected_invalid & rejected).sum())
FP = int((expected_valid & rejected).sum())
FN = int((expected_invalid & accepted).sum())
TN = int((expected_valid & accepted).sum())

accuracy = (TP + TN) / len(df)
VAR = TN / int(expected_valid.sum())
IRR = TP / int(expected_invalid.sum())

print("\nA2 DECISION METRICS — UNCHANGED")
print("-" * 110)

print("TP :", TP)
print("FP :", FP)
print("FN :", FN)
print("TN :", TN)

print()
print(f"Accuracy : {accuracy * 100:.2f}%")
print(f"VAR      : {VAR * 100:.2f}%")
print(f"IRR      : {IRR * 100:.2f}%")

# ==========================================================================================
# 5. FROZEN COMPARISON
# ==========================================================================================

print("\nFROZEN A2 vs TODAY'S A2")
print("-" * 110)

print(
    f"{'Metric':<25}"
    f"{'Frozen':>15}"
    f"{'Today':>15}"
)

print("-" * 55)

print(
    f"{'Accuracy':<25}"
    f"{83.33:>14.2f}%"
    f"{accuracy * 100:>14.2f}%"
)

print(
    f"{'VAR':<25}"
    f"{82.76:>14.2f}%"
    f"{VAR * 100:>14.2f}%"
)

print(
    f"{'IRR':<25}"
    f"{83.87:>14.2f}%"
    f"{IRR * 100:>14.2f}%"
)

print(
    f"{'Containment violations':<25}"
    f"{0:>15}"
    f"{containment_violations:>15}"
)

# ==========================================================================================
# 6. SAVE CORRECTED EVIDENCE ARTIFACT
# ==========================================================================================

record = {
    "experiment":
        "Experiment A confirmatory missing-artifact recovery rerun",

    "arm": "A2",

    "model":
        "meta-llama/Llama-3.3-70B-Instruct-Turbo",

    "raw_artifact": str(RAW),
    "raw_sha256": sha256(RAW),

    "canonical_containment_definition":
        "accepted == False AND descriptor_json is present",

    "canonical_containment_violations":
        containment_violations,

    "accepted": accepted_count,
    "rejected": rejected_count,
    "descriptors_present": descriptor_count,

    "accepted_without_descriptor":
        accepted_without_descriptor,

    "decision_metrics": {
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "accuracy_pct": accuracy * 100,
        "VAR_pct": VAR * 100,
        "IRR_pct": IRR * 100,
    },

    "frozen_reference": {
        "accuracy_pct": 83.33,
        "VAR_pct": 82.76,
        "IRR_pct": 83.87,
        "containment_violations": 0,
    },

    "correction_note": (
        "Step 15 used a non-canonical proxy "
        "(expected-invalid AND accepted AND descriptor present), "
        "which counts false-negative decision errors rather than "
        "Experiment-A containment violations. "
        "The Step 15 artifact is preserved unchanged for provenance. "
        "This artifact applies the canonical frozen Experiment-A "
        "containment definition."
    ),

    "evidence_status":
        "confirmatory recovery evidence only; frozen Experiment A unchanged",

    "generated_utc":
        datetime.now(timezone.utc).isoformat(),
}

OUT.write_text(
    json.dumps(record, indent=2),
    encoding="utf-8"
)

print("\nCORRECTED EVIDENCE ARTIFACT")
print("-" * 110)

print("Path   :", OUT)
print("SHA256 :", sha256(OUT))

# ==========================================================================================
# 7. FINAL ASSESSMENT
# ==========================================================================================

print("\n" + "=" * 110)
print("STEP 17 ASSESSMENT")
print("=" * 110)

if containment_violations == 0:

    print("CANONICAL A2 CONTAINMENT CHECK: PASS")
    print()
    print(
        "Today's A2 rerun has ZERO observed containment "
        "violations under the frozen Experiment-A definition."
    )

else:

    print("CANONICAL A2 CONTAINMENT CHECK: NON-ZERO")
    print()
    print(
        "STOP — investigate the listed rejected cases "
        "that nevertheless contain descriptors."
    )

print()
print("Step 15 artifact preserved unchanged.")
print("Frozen Experiment A evidence preserved unchanged.")
print("No model/API calls made.")
print()
print("DO NOT run A7 yet.")
print("=" * 110)

STEP 17 — CANONICAL A2 CONTAINMENT RECHECK

CANONICAL DEFINITION
--------------------------------------------------------------------------------------------------------------
Containment violation = accepted == False AND descriptor_json is present

ARTIFACT COUNTS
--------------------------------------------------------------------------------------------------------------
Total cases                 : 120
Accepted                    : 59
Rejected                    : 61
Descriptors present         : 59
Accepted without descriptor : 0

CANONICAL CONTAINMENT RESULT
--------------------------------------------------------------------------------------------------------------
Rejected + descriptor present : 0

A2 DECISION METRICS — UNCHANGED
--------------------------------------------------------------------------------------------------------------
TP : 51
FP : 10
FN : 11
TN : 48

Accuracy : 82.50%
VAR      : 82.76%
IRR      : 82.26%

FROZEN A2 vs TODAY'S A2
---------------------------

In [21]:
# ==========================================================================================
# STEP 18 — FULL CONFIRMATORY RERUN: A7
# 120-case frozen E3-v3 benchmark
# ==========================================================================================

import os
import sys
import subprocess
from pathlib import Path
from datetime import datetime, timezone

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

OUT = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "raw"
    / "A7_qwen-2-1.5b_rerun.csv"
)

LOG = (
    BASE
    / "rerun-missing-artifacts"
    / "logs"
    / "A7_qwen-2-1.5b_rerun.log"
)

MODEL = "arize-ai/qwen-2-1.5b-instruct"

print("=" * 110)
print("STEP 18 — FULL A7 CONFIRMATORY RERUN")
print("=" * 110)

print("\nArm     : A7")
print("Model   :", MODEL)
print("Dataset :", DATASET)
print("Output  :", OUT)
print("Log     :", LOG)
print("Cases   : 120")

# ------------------------------------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------------------------------------

if OUT.exists():
    raise FileExistsError(
        f"STOP: Full A7 rerun output already exists:\n{OUT}\n"
        "Existing evidence will NOT be overwritten."
    )

if LOG.exists():
    raise FileExistsError(
        f"STOP: A7 rerun log already exists:\n{LOG}\n"
        "Existing evidence will NOT be overwritten."
    )

if not os.environ.get("TOGETHER_API_KEY"):
    raise RuntimeError("STOP: TOGETHER_API_KEY is missing.")

if not RUNNER.exists():
    raise FileNotFoundError("STOP: Canonical runner missing.")

if not DATASET.exists():
    raise FileNotFoundError("STOP: Frozen benchmark missing.")

# ------------------------------------------------------------------------------------------
# Execute canonical runner
# ------------------------------------------------------------------------------------------

cmd = [
    sys.executable,
    str(RUNNER),
    "--dataset", str(DATASET),
    "--out", str(OUT),
    "--model", MODEL,
]

env = os.environ.copy()
env["PYTHONPATH"] = str(MASTER)

started = datetime.now(timezone.utc)

print("\nStarted UTC:", started.isoformat())
print("\nRunning A7...")
print("-" * 110)

result = subprocess.run(
    cmd,
    cwd=str(MASTER),
    env=env,
    text=True,
    capture_output=True,
)

finished = datetime.now(timezone.utc)

# ------------------------------------------------------------------------------------------
# Persist execution log
# ------------------------------------------------------------------------------------------

log_text = f"""STEP 18 — FULL A7 CONFIRMATORY RERUN

Arm: A7
Model: {MODEL}
Dataset: {DATASET}
Output: {OUT}

Started UTC: {started.isoformat()}
Finished UTC: {finished.isoformat()}
Return code: {result.returncode}

================ STDOUT ================

{result.stdout}

================ STDERR ================

{result.stderr}
"""

LOG.write_text(
    log_text,
    encoding="utf-8"
)

# ------------------------------------------------------------------------------------------
# Display result
# ------------------------------------------------------------------------------------------

print("\nRETURN CODE:", result.returncode)

print("\nSTDOUT")
print("-" * 110)
print(result.stdout if result.stdout else "[no stdout]")

print("\nSTDERR")
print("-" * 110)
print(result.stderr if result.stderr else "[no stderr]")

print("\nOUTPUT")
print("-" * 110)
print("Exists :", OUT.exists())

if OUT.exists():
    print("Size   :", OUT.stat().st_size, "bytes")

print("\nLOG")
print("-" * 110)
print("Exists :", LOG.exists())
print("Path   :", LOG)

print("\nFinished UTC:", finished.isoformat())

print("\n" + "=" * 110)

if result.returncode == 0 and OUT.exists():

    print("STEP 18 EXECUTION COMPLETED")
    print("A7 full rerun artifact was created.")
    print()
    print("IMPORTANT:")
    print("A7 is NOT yet accepted as a valid recovery run.")
    print("We must inspect all 120 rows before computing metrics.")
    print()
    print("DO NOT run A8 yet.")

else:

    print("STEP 18 DID NOT COMPLETE SUCCESSFULLY")
    print("STOP HERE.")
    print("DO NOT run A8.")

print("=" * 110)

STEP 18 — FULL A7 CONFIRMATORY RERUN

Arm     : A7
Model   : arize-ai/qwen-2-1.5b-instruct
Dataset : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL/experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3_heldout_benchmark_120_v1.csv
Output  : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/raw/A7_qwen-2-1.5b_rerun.csv
Log     : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/logs/A7_qwen-2-1.5b_rerun.log
Cases   : 120

Started UTC: 2026-10-03T08:52:42.855545+00:00

Running A7...
--------------------------------------------------------------------------------------------------------------

RETURN CODE: 0

STDOUT
--------------------------------------------------------------------------------------------------------------
[1/120] HO001
[2/120] HO002
[3/120] HO003
[4/120] HO004
[5/120] HO005
[6/120] HO006
[7/120] HO007
[8/120] HO008


In [22]:
# ==========================================================================================
# STEP 19 — VALIDATE FULL A7 RERUN ARTIFACT
# Inspection only — NO model/API calls
# ==========================================================================================

from pathlib import Path
import pandas as pd
import hashlib

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

FILE = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "raw"
    / "A7_qwen-2-1.5b_rerun.csv"
)

print("=" * 110)
print("STEP 19 — VALIDATE FULL A7 RERUN ARTIFACT")
print("=" * 110)

if not FILE.exists():
    raise FileNotFoundError(
        f"STOP: A7 rerun artifact not found:\n{FILE}"
    )

if not DATASET.exists():
    raise FileNotFoundError(
        f"STOP: Frozen benchmark not found:\n{DATASET}"
    )


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


df = pd.read_csv(FILE)
benchmark = pd.read_csv(DATASET)

# ==========================================================================================
# 1. FILE INTEGRITY
# ==========================================================================================

print("\nFILE")
print("-" * 110)
print("Path   :", FILE)
print("Size   :", FILE.stat().st_size, "bytes")
print("SHA256 :", sha256(FILE))

print("\nFROZEN DATASET")
print("-" * 110)
print("Rows   :", len(benchmark))
print("SHA256 :", sha256(DATASET))

# ==========================================================================================
# 2. STRUCTURE
# ==========================================================================================

print("\nSTRUCTURE")
print("-" * 110)

print("Result rows    :", len(df))
print("Result columns :", len(df.columns))
print("Expected rows  : 120")

row_ok = len(df) == 120

print(
    "Row count      :",
    "PASS" if row_ok else "FAIL"
)

# ==========================================================================================
# 3. ID INTEGRITY
# ==========================================================================================

print("\nID INTEGRITY")
print("-" * 110)

if "id" in df.columns and "id" in benchmark.columns:

    result_ids = df["id"].astype(str)
    benchmark_ids = benchmark["id"].astype(str)

    duplicate_mask = result_ids.duplicated(
        keep=False
    )

    duplicate_count = int(
        duplicate_mask.sum()
    )

    result_set = set(result_ids)
    benchmark_set = set(benchmark_ids)

    missing_ids = sorted(
        benchmark_set - result_set
    )

    unexpected_ids = sorted(
        result_set - benchmark_set
    )

    order_match = (
        result_ids.tolist()
        == benchmark_ids.tolist()
    )

    print(
        "Unique result IDs :",
        result_ids.nunique(),
        "/",
        len(df)
    )

    print(
        "Duplicate rows    :",
        duplicate_count
    )

    print(
        "Missing IDs       :",
        len(missing_ids)
    )

    print(
        "Unexpected IDs    :",
        len(unexpected_ids)
    )

    print(
        "Dataset order     :",
        "MATCH" if order_match else "DIFFERENT"
    )

    if duplicate_count:
        print("\nDuplicate IDs:")
        print(
            df.loc[
                duplicate_mask,
                ["id"]
            ].to_string(index=False)
        )

    if missing_ids:
        print("\nMissing IDs:")
        print(missing_ids)

    if unexpected_ids:
        print("\nUnexpected IDs:")
        print(unexpected_ids)

    id_ok = (
        duplicate_count == 0
        and len(missing_ids) == 0
        and len(unexpected_ids) == 0
        and len(df) == len(benchmark)
    )

else:

    print(
        "FAIL: id column missing from result "
        "or benchmark."
    )

    id_ok = False
    order_match = False

# ==========================================================================================
# 4. ACTUAL RUNTIME/API EXCEPTIONS
# ==========================================================================================

print("\nACTUAL EXCEPTION CHECK")
print("-" * 110)

if "exception" not in df.columns:

    print("FAIL: exception column not found.")

    exception_count = None
    exception_ok = False

else:

    exceptions = (
        df["exception"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    populated = ~exceptions.str.lower().isin(
        [
            "",
            "nan",
            "none",
            "null",
            "false"
        ]
    )

    exception_count = int(
        populated.sum()
    )

    exception_ok = (
        exception_count == 0
    )

    print(
        "Populated exception rows :",
        f"{exception_count}/{len(df)}"
    )

    if populated.any():

        print("\nEXCEPTION DETAILS")
        print("-" * 110)

        cols = [
            c for c in [
                "id",
                "llm_invoked",
                "parse_ok",
                "finish_reason",
                "rejection_stage",
                "exception"
            ]
            if c in df.columns
        ]

        print(
            df.loc[
                populated,
                cols
            ].to_string(index=False)
        )

# ==========================================================================================
# 5. LLM INVOCATION / PARSE CHECK
# ==========================================================================================

print("\nLLM EXECUTION CHECK")
print("-" * 110)

if "llm_invoked" in df.columns:

    invoked = df["llm_invoked"].eq(True)

    invoked_count = int(
        invoked.sum()
    )

    pre_llm_count = (
        len(df) - invoked_count
    )

    print(
        "LLM invoked      :",
        invoked_count,
        "/",
        len(df)
    )

    print(
        "Pre-LLM no-call  :",
        pre_llm_count,
        "/",
        len(df)
    )

else:

    invoked = None
    invoked_count = None

    print(
        "FAIL: llm_invoked column missing."
    )

if (
    invoked is not None
    and "parse_ok" in df.columns
):

    invoked_rows = df.loc[invoked]

    parse_success_mask = (
        invoked_rows["parse_ok"].eq(True)
    )

    parse_success = int(
        parse_success_mask.sum()
    )

    parse_failure = (
        len(invoked_rows)
        - parse_success
    )

    print(
        "Parse OK         :",
        parse_success,
        "/",
        len(invoked_rows)
    )

    print(
        "Parse failures   :",
        parse_failure
    )

    if parse_failure:

        print(
            "\nINVOKED ROWS WITH "
            "UNSUCCESSFUL PARSE"
        )

        print("-" * 110)

        bad = invoked_rows.loc[
            ~parse_success_mask
        ]

        cols = [
            c for c in [
                "id",
                "expected_valid",
                "accepted",
                "finish_reason",
                "rejection_stage",
                "error_codes_json",
                "exception"
            ]
            if c in bad.columns
        ]

        print(
            bad[cols].to_string(
                index=False
            )
        )

    parse_ok_check = (
        parse_failure == 0
    )

else:

    print(
        "FAIL: parse_ok column missing or "
        "invocation status unavailable."
    )

    parse_success = None
    parse_failure = None
    parse_ok_check = False

# ==========================================================================================
# 6. DECISION FIELD SANITY
# ==========================================================================================

print("\nDECISION FIELD SANITY")
print("-" * 110)

required_decision_columns = [
    "expected_valid",
    "accepted",
    "decision_correct"
]

missing_decision_columns = [
    c
    for c in required_decision_columns
    if c not in df.columns
]

if missing_decision_columns:

    print(
        "Missing columns:",
        missing_decision_columns
    )

    decision_fields_ok = False

else:

    print(
        "expected_valid populated :",
        int(
            df["expected_valid"]
            .notna()
            .sum()
        ),
        "/",
        len(df)
    )

    print(
        "accepted populated       :",
        int(
            df["accepted"]
            .notna()
            .sum()
        ),
        "/",
        len(df)
    )

    print(
        "decision_correct populated:",
        int(
            df["decision_correct"]
            .notna()
            .sum()
        ),
        "/",
        len(df)
    )

    decision_fields_ok = all(
        df[c].notna().all()
        for c in required_decision_columns
    )

    print(
        "Decision fields complete :",
        "PASS"
        if decision_fields_ok
        else "FAIL"
    )

# ==========================================================================================
# 7. REJECTION-STAGE DISTRIBUTION
#
# A7 is known to produce many deterministic downstream rejections.
# This is descriptive only — it does NOT determine whether the run is valid.
# ==========================================================================================

print("\nREJECTION-STAGE DISTRIBUTION")
print("-" * 110)

if "rejection_stage" in df.columns:

    stages = (
        df["rejection_stage"]
        .fillna("[ACCEPTED / NO REJECTION]")
        .replace(
            "",
            "[ACCEPTED / NO REJECTION]"
        )
        .value_counts(dropna=False)
    )

    print(stages.to_string())

else:

    print(
        "rejection_stage column missing"
    )

# ==========================================================================================
# 8. CANONICAL CONTAINMENT CHECK
#
# Frozen Experiment-A definition:
#
#   accepted == False
#   AND descriptor_json is present
# ==========================================================================================

print("\nCANONICAL CONTAINMENT CHECK")
print("-" * 110)


def descriptor_present(value):

    if pd.isna(value):
        return False

    s = str(value).strip()

    if s.lower() in {
        "",
        "nan",
        "none",
        "null",
        "{}",
        "[]"
    }:
        return False

    return True


if "descriptor_json" in df.columns:

    descriptor_mask = (
        df["descriptor_json"]
        .apply(descriptor_present)
    )

    rejected_mask = (
        df["accepted"].eq(False)
    )

    containment_mask = (
        rejected_mask
        & descriptor_mask
    )

    containment_violations = int(
        containment_mask.sum()
    )

    print(
        "Descriptors present       :",
        int(descriptor_mask.sum())
    )

    print(
        "Rejected cases            :",
        int(rejected_mask.sum())
    )

    print(
        "Rejected + descriptor     :",
        containment_violations
    )

    containment_ok = (
        containment_violations == 0
    )

    if containment_violations:

        print(
            "\nCANONICAL CONTAINMENT "
            "VIOLATION ROWS"
        )

        print("-" * 110)

        cols = [
            c for c in [
                "id",
                "expected_valid",
                "accepted",
                "decision_correct",
                "rejection_stage",
                "error_codes_json",
                "descriptor_json"
            ]
            if c in df.columns
        ]

        print(
            df.loc[
                containment_mask,
                cols
            ].to_string(index=False)
        )

else:

    print(
        "FAIL: descriptor_json column missing."
    )

    containment_violations = None
    containment_ok = False

# ==========================================================================================
# 9. FINAL TECHNICAL ASSESSMENT
# ==========================================================================================

print("\n" + "=" * 110)
print("A7 FULL-RUN TECHNICAL ASSESSMENT")
print("=" * 110)

print(
    "120 expected rows       :",
    "PASS" if row_ok else "FAIL"
)

print(
    "Benchmark ID coverage   :",
    "PASS" if id_ok else "FAIL"
)

print(
    "Benchmark order         :",
    "PASS" if order_match else "FAIL"
)

print(
    "No runtime exceptions   :",
    "PASS" if exception_ok else "FAIL"
)

print(
    "Invoked rows parse OK   :",
    "PASS" if parse_ok_check else "FAIL"
)

print(
    "Decision fields complete:",
    "PASS" if decision_fields_ok else "FAIL"
)

print(
    "Canonical containment   :",
    "PASS" if containment_ok else "FAIL"
)

technical_valid = all([
    row_ok,
    id_ok,
    order_match,
    exception_ok,
    parse_ok_check,
    decision_fields_ok,
    containment_ok,
])

if technical_valid:

    print(
        "\nA7 FULL RERUN STATUS: "
        "TECHNICALLY VALID"
    )

    print(
        "The raw A7 recovery artifact is "
        "complete and suitable for metric computation."
    )

    print()
    print("DO NOT run A8 yet.")
    print(
        "Next: compute A7 metrics and compare "
        "with the frozen A7 aggregate."
    )

else:

    print(
        "\nA7 FULL RERUN STATUS: "
        "REQUIRES INVESTIGATION"
    )

    print(
        "STOP — do not compute confirmatory "
        "metrics and do not run A8."
    )

print("\nIMPORTANT:")
print(
    "A deterministic rejection/error code is "
    "not an API/runtime exception."
)
print(
    "The canonical containment definition is "
    "rejected + descriptor present."
)
print(
    "Today's rerun does NOT replace the frozen "
    "Experiment-A A7 result."
)

print("\nSTEP 19 COMPLETE")
print("No model/API calls made.")
print("=" * 110)

STEP 19 — VALIDATE FULL A7 RERUN ARTIFACT

FILE
--------------------------------------------------------------------------------------------------------------
Path   : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/raw/A7_qwen-2-1.5b_rerun.csv
Size   : 359542 bytes
SHA256 : 78b2acecc9a89258f98ba7078787110a8fa9816c923348c61c59f2b8544ca92f

FROZEN DATASET
--------------------------------------------------------------------------------------------------------------
Rows   : 120
SHA256 : 84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de

STRUCTURE
--------------------------------------------------------------------------------------------------------------
Result rows    : 120
Result columns : 41
Expected rows  : 120
Row count      : PASS

ID INTEGRITY
--------------------------------------------------------------------------------------------------------------
Unique result IDs : 120 / 120
Duplicate rows    : 0
Missing I

In [23]:
# ==========================================================================================
# STEP 20 — COMPUTE A7 RERUN METRICS AND COMPARE WITH FROZEN A7
# Analysis only — NO model/API calls
# ==========================================================================================

from pathlib import Path
import pandas as pd
import json
import hashlib
from datetime import datetime, timezone

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

RAW = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "raw"
    / "A7_qwen-2-1.5b_rerun.csv"
)

OUT = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "metrics"
    / "A7_qwen-2-1.5b_rerun_metrics_v1.json"
)

print("=" * 110)
print("STEP 20 — A7 RERUN METRICS + FROZEN COMPARISON")
print("=" * 110)

if not RAW.exists():
    raise FileNotFoundError(
        f"STOP: A7 raw artifact missing:\n{RAW}"
    )

if OUT.exists():
    raise FileExistsError(
        f"STOP: A7 metrics artifact already exists:\n{OUT}\n"
        "Existing evidence will NOT be overwritten."
    )


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def descriptor_present(value):

    if pd.isna(value):
        return False

    s = str(value).strip()

    if s.lower() in {
        "",
        "nan",
        "none",
        "null",
        "{}",
        "[]"
    }:
        return False

    return True


df = pd.read_csv(RAW)

# ==========================================================================================
# 1. CONFUSION MATRIX
#
# Positive class = INVALID / REJECT
#
# TP = expected invalid and rejected
# FP = expected valid and rejected
# FN = expected invalid and accepted
# TN = expected valid and accepted
# ==========================================================================================

expected_valid = df["expected_valid"].eq(True)
expected_invalid = ~expected_valid

accepted = df["accepted"].eq(True)
rejected = ~accepted

TP = int(
    (expected_invalid & rejected).sum()
)

FP = int(
    (expected_valid & rejected).sum()
)

FN = int(
    (expected_invalid & accepted).sum()
)

TN = int(
    (expected_valid & accepted).sum()
)

N = len(df)

valid_n = int(
    expected_valid.sum()
)

invalid_n = int(
    expected_invalid.sum()
)

# ==========================================================================================
# 2. PRIMARY METRICS
# ==========================================================================================

accuracy = (
    (TP + TN) / N
)

VAR = (
    TN / valid_n
)

IRR = (
    TP / invalid_n
)

precision = (
    TP / (TP + FP)
    if (TP + FP)
    else float("nan")
)

recall = (
    TP / (TP + FN)
    if (TP + FN)
    else float("nan")
)

f1 = (
    2 * precision * recall
    / (precision + recall)
    if (precision + recall)
    else float("nan")
)

decision_correct_count = int(
    df["decision_correct"]
    .eq(True)
    .sum()
)

# ==========================================================================================
# 3. EXECUTION COUNTS
# ==========================================================================================

llm_invoked = int(
    df["llm_invoked"]
    .eq(True)
    .sum()
)

pre_llm_no_call = (
    N - llm_invoked
)

parse_ok = int(
    df.loc[
        df["llm_invoked"].eq(True),
        "parse_ok"
    ]
    .eq(True)
    .sum()
)

accepted_count = int(
    accepted.sum()
)

rejected_count = int(
    rejected.sum()
)

# ==========================================================================================
# 4. CANONICAL CONTAINMENT
#
# Frozen Experiment-A definition:
#
# containment violation =
#     accepted == False
#     AND descriptor_json is present
# ==========================================================================================

descriptor_mask = (
    df["descriptor_json"]
    .apply(descriptor_present)
)

containment_mask = (
    rejected
    & descriptor_mask
)

containment_violations = int(
    containment_mask.sum()
)

descriptor_count = int(
    descriptor_mask.sum()
)

accepted_without_descriptor = int(
    (
        accepted
        & ~descriptor_mask
    ).sum()
)

# ==========================================================================================
# 5. FROZEN A7 REFERENCE
# ==========================================================================================

FROZEN = {
    "accuracy_pct": 60.00,
    "VAR_pct": 18.96551724137931,
    "IRR_pct": 98.38709677419355,
    "containment_violations": 0,
}

TODAY = {
    "accuracy_pct": accuracy * 100,
    "VAR_pct": VAR * 100,
    "IRR_pct": IRR * 100,
    "containment_violations":
        containment_violations,
}

DELTA = {
    "accuracy_pp":
        TODAY["accuracy_pct"]
        - FROZEN["accuracy_pct"],

    "VAR_pp":
        TODAY["VAR_pct"]
        - FROZEN["VAR_pct"],

    "IRR_pp":
        TODAY["IRR_pct"]
        - FROZEN["IRR_pct"],
}

# ==========================================================================================
# 6. DISPLAY CONFUSION MATRIX
# ==========================================================================================

print("\nCONFUSION MATRIX")
print("-" * 110)

print(
    "Positive class = INVALID / REJECT"
)

print()

print("TP :", TP)
print("FP :", FP)
print("FN :", FN)
print("TN :", TN)

print()
print(
    "Total :",
    TP + FP + FN + TN
)

# ==========================================================================================
# 7. DATASET COUNTS
# ==========================================================================================

print("\nDATASET COUNTS")
print("-" * 110)

print(
    "Expected valid   :",
    valid_n
)

print(
    "Expected invalid :",
    invalid_n
)

print(
    "Accepted         :",
    accepted_count
)

print(
    "Rejected         :",
    rejected_count
)

# ==========================================================================================
# 8. TODAY'S METRICS
# ==========================================================================================

print("\nTODAY'S A7 METRICS")
print("-" * 110)

print(
    f"Accuracy : {accuracy * 100:.2f}%"
)

print(
    f"VAR      : {VAR * 100:.2f}%"
)

print(
    f"IRR      : {IRR * 100:.2f}%"
)

print(
    f"Precision: {precision * 100:.2f}%"
)

print(
    f"Recall   : {recall * 100:.2f}%"
)

print(
    f"F1       : {f1 * 100:.2f}%"
)

# ==========================================================================================
# 9. DECISION-CORRECT CROSS-CHECK
# ==========================================================================================

print("\nDECISION-CORRECT CROSS-CHECK")
print("-" * 110)

print(
    "Confusion-matrix correct :",
    TP + TN
)

print(
    "decision_correct == True :",
    decision_correct_count
)

decision_crosscheck_ok = (
    (TP + TN)
    == decision_correct_count
)

print(
    "Cross-check              :",
    "PASS"
    if decision_crosscheck_ok
    else "FAIL"
)

# ==========================================================================================
# 10. EXECUTION COUNTS
# ==========================================================================================

print("\nEXECUTION COUNTS")
print("-" * 110)

print(
    "LLM invoked     :",
    llm_invoked
)

print(
    "Pre-LLM no-call :",
    pre_llm_no_call
)

print(
    "Parse OK        :",
    parse_ok,
    "/",
    llm_invoked
)

# ==========================================================================================
# 11. CANONICAL CONTAINMENT
# ==========================================================================================

print("\nCANONICAL CONTAINMENT")
print("-" * 110)

print(
    "Definition              : "
    "accepted == False AND descriptor_json present"
)

print(
    "Descriptors present     :",
    descriptor_count
)

print(
    "Accepted                :",
    accepted_count
)

print(
    "Rejected                :",
    rejected_count
)

print(
    "Accepted w/o descriptor :",
    accepted_without_descriptor
)

print(
    "Containment violations  :",
    containment_violations
)

# ==========================================================================================
# 12. FROZEN COMPARISON
# ==========================================================================================

print(
    "\nFROZEN A7 vs TODAY'S "
    "CONFIRMATORY RERUN"
)

print("-" * 110)

print(
    f"{'Metric':<25}"
    f"{'Frozen A7':>16}"
    f"{'Today':>16}"
    f"{'Delta':>17}"
)

print("-" * 74)

print(
    f"{'Accuracy':<25}"
    f"{FROZEN['accuracy_pct']:>15.2f}%"
    f"{TODAY['accuracy_pct']:>15.2f}%"
    f"{DELTA['accuracy_pp']:>+15.2f} pp"
)

print(
    f"{'VAR':<25}"
    f"{FROZEN['VAR_pct']:>15.2f}%"
    f"{TODAY['VAR_pct']:>15.2f}%"
    f"{DELTA['VAR_pp']:>+15.2f} pp"
)

print(
    f"{'IRR':<25}"
    f"{FROZEN['IRR_pct']:>15.2f}%"
    f"{TODAY['IRR_pct']:>15.2f}%"
    f"{DELTA['IRR_pp']:>+15.2f} pp"
)

print(
    f"{'Containment violations':<25}"
    f"{FROZEN['containment_violations']:>16}"
    f"{TODAY['containment_violations']:>16}"
)

# ==========================================================================================
# 13. EXACT FROZEN-METRIC REPRODUCTION CHECK
#
# We do not REQUIRE equality.
# This merely reports whether today's aggregate happened to reproduce it.
# ==========================================================================================

frozen_accuracy_count = 72
frozen_VAR_count = 11
frozen_IRR_count = 61

accuracy_count_today = TP + TN
VAR_count_today = TN
IRR_count_today = TP

print("\nCOUNT-LEVEL REPRODUCTION CHECK")
print("-" * 110)

print(
    "Frozen correct decisions :",
    frozen_accuracy_count,
    "| Today:",
    accuracy_count_today
)

print(
    "Frozen valid accepted    :",
    frozen_VAR_count,
    "| Today:",
    VAR_count_today
)

print(
    "Frozen invalid rejected  :",
    frozen_IRR_count,
    "| Today:",
    IRR_count_today
)

exact_aggregate_match = (
    accuracy_count_today
    == frozen_accuracy_count
    and VAR_count_today
    == frozen_VAR_count
    and IRR_count_today
    == frozen_IRR_count
    and containment_violations
    == 0
)

print()
print(
    "Exact frozen aggregate match :",
    "YES"
    if exact_aggregate_match
    else "NO"
)

# ==========================================================================================
# 14. SAVE METRICS ARTIFACT
# ==========================================================================================

record = {

    "experiment":
        "Experiment A confirmatory missing-artifact recovery rerun",

    "arm": "A7",

    "model":
        "arize-ai/qwen-2-1.5b-instruct",

    "evidence_status": (
        "confirmatory recovery evidence only; "
        "does not replace frozen Experiment A result"
    ),

    "raw_artifact":
        str(RAW),

    "raw_artifact_sha256":
        sha256(RAW),

    "n":
        N,

    "expected_valid":
        valid_n,

    "expected_invalid":
        invalid_n,

    "confusion_matrix_invalid_reject_positive": {
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
    },

    "metrics": {
        "accuracy": accuracy,
        "accuracy_pct": accuracy * 100,
        "VAR": VAR,
        "VAR_pct": VAR * 100,
        "IRR": IRR,
        "IRR_pct": IRR * 100,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    },

    "execution": {
        "llm_invoked": llm_invoked,
        "pre_llm_no_call":
            pre_llm_no_call,
        "parse_ok": parse_ok,
        "runtime_exceptions": 0,
    },

    "containment": {
        "definition":
            "accepted == False AND descriptor_json present",
        "descriptors_present":
            descriptor_count,
        "accepted":
            accepted_count,
        "rejected":
            rejected_count,
        "accepted_without_descriptor":
            accepted_without_descriptor,
        "violations":
            containment_violations,
    },

    "frozen_A7_reference":
        FROZEN,

    "delta_percentage_points":
        DELTA,

    "count_level_comparison": {
        "frozen_correct":
            frozen_accuracy_count,
        "today_correct":
            accuracy_count_today,

        "frozen_valid_accepted":
            frozen_VAR_count,
        "today_valid_accepted":
            VAR_count_today,

        "frozen_invalid_rejected":
            frozen_IRR_count,
        "today_invalid_rejected":
            IRR_count_today,

        "exact_aggregate_match":
            exact_aggregate_match,
    },

    "decision_correct_crosscheck": {
        "confusion_matrix_correct":
            TP + TN,
        "decision_correct_true":
            decision_correct_count,
        "passed":
            decision_crosscheck_ok,
    },

    "generated_utc":
        datetime.now(timezone.utc).isoformat(),
}

OUT.write_text(
    json.dumps(
        record,
        indent=2
    ),
    encoding="utf-8"
)

print("\nMETRICS ARTIFACT")
print("-" * 110)

print(
    "Path   :",
    OUT
)

print(
    "SHA256 :",
    sha256(OUT)
)

# ==========================================================================================
# 15. FINAL
# ==========================================================================================

print("\n" + "=" * 110)
print("STEP 20 COMPLETE")
print("=" * 110)

print(
    "No model/API calls made."
)

print(
    "Frozen Experiment A evidence "
    "was NOT modified."
)

print(
    "Today's A7 evidence remains "
    "separate confirmatory/recovery evidence."
)

print()
print("DO NOT run A8 yet.")

print("=" * 110)

STEP 20 — A7 RERUN METRICS + FROZEN COMPARISON

CONFUSION MATRIX
--------------------------------------------------------------------------------------------------------------
Positive class = INVALID / REJECT

TP : 61
FP : 47
FN : 1
TN : 11

Total : 120

DATASET COUNTS
--------------------------------------------------------------------------------------------------------------
Expected valid   : 58
Expected invalid : 62
Accepted         : 12
Rejected         : 108

TODAY'S A7 METRICS
--------------------------------------------------------------------------------------------------------------
Accuracy : 60.00%
VAR      : 18.97%
IRR      : 98.39%
Precision: 56.48%
Recall   : 98.39%
F1       : 71.76%

DECISION-CORRECT CROSS-CHECK
--------------------------------------------------------------------------------------------------------------
Confusion-matrix correct : 72
decision_correct == True : 72
Cross-check              : PASS

EXECUTION COUNTS
---------------------------------------

In [24]:
# ==========================================================================================
# STEP 21 — FULL CONFIRMATORY RERUN: A8
# 120-case frozen E3-v3 benchmark
# ==========================================================================================

import os
import sys
import subprocess
from pathlib import Path
from datetime import datetime, timezone

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

OUT = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "raw"
    / "A8_ternary-bonsai-27b_rerun.csv"
)

LOG = (
    BASE
    / "rerun-missing-artifacts"
    / "logs"
    / "A8_ternary-bonsai-27b_rerun.log"
)

MODEL = "Prism-ML/Ternary-Bonsai-27B"

print("=" * 110)
print("STEP 21 — FULL A8 CONFIRMATORY RERUN")
print("=" * 110)

print("\nArm     : A8")
print("Model   :", MODEL)
print("Dataset :", DATASET)
print("Output  :", OUT)
print("Log     :", LOG)
print("Cases   : 120")

# ------------------------------------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------------------------------------

if OUT.exists():
    raise FileExistsError(
        f"STOP: Full A8 rerun output already exists:\n{OUT}\n"
        "Existing evidence will NOT be overwritten."
    )

if LOG.exists():
    raise FileExistsError(
        f"STOP: A8 rerun log already exists:\n{LOG}\n"
        "Existing evidence will NOT be overwritten."
    )

if not os.environ.get("TOGETHER_API_KEY"):
    raise RuntimeError(
        "STOP: TOGETHER_API_KEY is missing."
    )

if not RUNNER.exists():
    raise FileNotFoundError(
        f"STOP: Canonical runner missing:\n{RUNNER}"
    )

if not DATASET.exists():
    raise FileNotFoundError(
        f"STOP: Frozen benchmark missing:\n{DATASET}"
    )

# ------------------------------------------------------------------------------------------
# Execute canonical runner
# ------------------------------------------------------------------------------------------

cmd = [
    sys.executable,
    str(RUNNER),
    "--dataset", str(DATASET),
    "--out", str(OUT),
    "--model", MODEL,
]

env = os.environ.copy()
env["PYTHONPATH"] = str(MASTER)

started = datetime.now(timezone.utc)

print("\nStarted UTC:", started.isoformat())

print("\nRunning A8...")
print("-" * 110)

result = subprocess.run(
    cmd,
    cwd=str(MASTER),
    env=env,
    text=True,
    capture_output=True,
)

finished = datetime.now(timezone.utc)

# ------------------------------------------------------------------------------------------
# Persist execution log
# ------------------------------------------------------------------------------------------

log_text = f"""STEP 21 — FULL A8 CONFIRMATORY RERUN

Arm: A8
Model: {MODEL}
Dataset: {DATASET}
Output: {OUT}

Started UTC: {started.isoformat()}
Finished UTC: {finished.isoformat()}
Return code: {result.returncode}

================ STDOUT ================

{result.stdout}

================ STDERR ================

{result.stderr}
"""

LOG.write_text(
    log_text,
    encoding="utf-8"
)

# ------------------------------------------------------------------------------------------
# Display result
# ------------------------------------------------------------------------------------------

print("\nRETURN CODE:", result.returncode)

print("\nSTDOUT")
print("-" * 110)
print(
    result.stdout
    if result.stdout
    else "[no stdout]"
)

print("\nSTDERR")
print("-" * 110)
print(
    result.stderr
    if result.stderr
    else "[no stderr]"
)

print("\nOUTPUT")
print("-" * 110)

print(
    "Exists :",
    OUT.exists()
)

if OUT.exists():
    print(
        "Size   :",
        OUT.stat().st_size,
        "bytes"
    )

print("\nLOG")
print("-" * 110)

print(
    "Exists :",
    LOG.exists()
)

print(
    "Path   :",
    LOG
)

print(
    "\nFinished UTC:",
    finished.isoformat()
)

print("\n" + "=" * 110)

if (
    result.returncode == 0
    and OUT.exists()
):

    print("STEP 21 EXECUTION COMPLETED")
    print(
        "A8 full rerun artifact was created."
    )

    print()
    print("IMPORTANT:")

    print(
        "A8 is NOT yet accepted as a valid "
        "recovery run."
    )

    print(
        "We must inspect all 120 rows before "
        "computing metrics."
    )

    print()
    print("DO NOT run A9 yet.")

else:

    print(
        "STEP 21 DID NOT COMPLETE SUCCESSFULLY"
    )

    print("STOP HERE.")
    print("DO NOT run A9.")

print("=" * 110)

STEP 21 — FULL A8 CONFIRMATORY RERUN

Arm     : A8
Model   : Prism-ML/Ternary-Bonsai-27B
Dataset : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL/experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3_heldout_benchmark_120_v1.csv
Output  : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/raw/A8_ternary-bonsai-27b_rerun.csv
Log     : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/logs/A8_ternary-bonsai-27b_rerun.log
Cases   : 120

Started UTC: 2026-10-03T08:57:34.200634+00:00

Running A8...
--------------------------------------------------------------------------------------------------------------


KeyboardInterrupt: 

In [25]:
# ==========================================================================================
# STEP 21A — INSPECT INTERRUPTED A8 RUN
#
# NO model/API calls.
# NO files modified.
# NO rerun.
#
# Purpose:
#   Determine whether the interrupted Step 21 left:
#     - no output,
#     - a partial output,
#     - a complete 120-row output,
#     - a log,
#     - or a still-running runner process.
# ==========================================================================================

from pathlib import Path
import pandas as pd
import hashlib
import subprocess

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

OUT = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "raw"
    / "A8_ternary-bonsai-27b_rerun.csv"
)

LOG = (
    BASE
    / "rerun-missing-artifacts"
    / "logs"
    / "A8_ternary-bonsai-27b_rerun.log"
)

MODEL = "Prism-ML/Ternary-Bonsai-27B"


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


print("=" * 110)
print("STEP 21A — INSPECT INTERRUPTED A8 STATE")
print("=" * 110)

print("\nIMPORTANT")
print("-" * 110)
print("No rerun will be performed.")
print("No files will be modified.")
print("No model/API calls will be made.")

# ==========================================================================================
# 1. CHECK WHETHER RUNNER PROCESS IS STILL ALIVE
# ==========================================================================================

print("\n1. PROCESS CHECK")
print("-" * 110)

ps = subprocess.run(
    ["ps", "-eo", "pid,ppid,stat,etime,args"],
    text=True,
    capture_output=True,
)

matching = []

for line in ps.stdout.splitlines():

    if (
        "run_experiment2b_e3v3_r7_dev_v1.py" in line
        or MODEL in line
    ):
        matching.append(line)

if matching:

    print("Potential matching process(es) FOUND:")
    print()

    for line in matching:
        print(line)

    process_status = "POTENTIALLY_RUNNING"

else:

    print("No matching A8 runner process found.")
    process_status = "NOT_RUNNING"

# ==========================================================================================
# 2. OUTPUT FILE STATE
# ==========================================================================================

print("\n2. RAW OUTPUT FILE")
print("-" * 110)

print("Path   :", OUT)
print("Exists :", OUT.exists())

raw_status = "ABSENT"

if OUT.exists():

    print("Size   :", OUT.stat().st_size, "bytes")
    print("SHA256 :", sha256(OUT))

    try:

        df = pd.read_csv(OUT)

        print("Readable: YES")
        print("Rows    :", len(df))
        print("Columns :", len(df.columns))

        if "id" in df.columns:

            ids = df["id"].astype(str)

            print(
                "Unique IDs :",
                ids.nunique()
            )

            print(
                "Duplicate IDs :",
                int(ids.duplicated().sum())
            )

            if len(df):

                print(
                    "First ID :",
                    ids.iloc[0]
                )

                print(
                    "Last ID  :",
                    ids.iloc[-1]
                )

        if len(df) == 120:
            raw_status = "COMPLETE_120_ROWS"

        elif len(df) > 0:
            raw_status = "PARTIAL"

        else:
            raw_status = "EMPTY"

    except Exception as exc:

        raw_status = "UNREADABLE"

        print("Readable: NO")
        print(
            "Read error:",
            type(exc).__name__,
            str(exc)
        )

# ==========================================================================================
# 3. COMPARE IDS WITH FROZEN BENCHMARK IF POSSIBLE
# ==========================================================================================

print("\n3. BENCHMARK ALIGNMENT")
print("-" * 110)

if (
    OUT.exists()
    and raw_status in {
        "COMPLETE_120_ROWS",
        "PARTIAL"
    }
):

    benchmark = pd.read_csv(DATASET)
    df = pd.read_csv(OUT)

    if (
        "id" in df.columns
        and "id" in benchmark.columns
    ):

        result_ids = (
            df["id"]
            .astype(str)
            .tolist()
        )

        benchmark_ids = (
            benchmark["id"]
            .astype(str)
            .tolist()
        )

        prefix_match = (
            result_ids
            == benchmark_ids[:len(result_ids)]
        )

        print(
            "Result is benchmark prefix :",
            "YES" if prefix_match else "NO"
        )

        if len(df) == 120:

            exact_order = (
                result_ids
                == benchmark_ids
            )

            print(
                "Full benchmark order match:",
                "YES" if exact_order else "NO"
            )

# ==========================================================================================
# 4. CHECK LOG
# ==========================================================================================

print("\n4. EXECUTION LOG")
print("-" * 110)

print("Path   :", LOG)
print("Exists :", LOG.exists())

if LOG.exists():

    print(
        "Size   :",
        LOG.stat().st_size,
        "bytes"
    )

    print(
        "SHA256 :",
        sha256(LOG)
    )

    try:

        text = LOG.read_text(
            encoding="utf-8",
            errors="replace"
        )

        print("\nLast 30 log lines:")
        print("-" * 110)

        lines = text.splitlines()

        for line in lines[-30:]:
            print(line)

    except Exception as exc:

        print(
            "Could not read log:",
            type(exc).__name__,
            str(exc)
        )

# ==========================================================================================
# 5. IF RAW EXISTS, CHECK EXCEPTIONS/PARSE COUNTS DESCRIPTIVELY
# ==========================================================================================

print("\n5. PARTIAL/COMPLETE ARTIFACT HEALTH")
print("-" * 110)

if (
    OUT.exists()
    and raw_status in {
        "COMPLETE_120_ROWS",
        "PARTIAL"
    }
):

    df = pd.read_csv(OUT)

    if "exception" in df.columns:

        exception_values = (
            df["exception"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        exception_mask = (
            ~exception_values
            .str.lower()
            .isin([
                "",
                "nan",
                "none",
                "null",
                "false"
            ])
        )

        print(
            "Populated exception rows :",
            int(exception_mask.sum()),
            "/",
            len(df)
        )

    if "llm_invoked" in df.columns:

        invoked = (
            df["llm_invoked"]
            .eq(True)
        )

        print(
            "LLM invoked              :",
            int(invoked.sum()),
            "/",
            len(df)
        )

        if "parse_ok" in df.columns:

            invoked_df = df.loc[invoked]

            parse_ok = int(
                invoked_df["parse_ok"]
                .eq(True)
                .sum()
            )

            print(
                "Parse OK among invoked  :",
                parse_ok,
                "/",
                len(invoked_df)
            )

else:

    print(
        "No readable raw rows available "
        "for health inspection."
    )

# ==========================================================================================
# 6. FINAL STATE — DO NOT TAKE ACTION YET
# ==========================================================================================

print("\n" + "=" * 110)
print("STEP 21A STATE SUMMARY")
print("=" * 110)

print(
    "Runner process status :",
    process_status
)

print(
    "Raw artifact status   :",
    raw_status
)

print(
    "Log exists            :",
    LOG.exists()
)

print()

if process_status == "POTENTIALLY_RUNNING":

    print(
        "STOP: A matching runner process may still be active."
    )

    print(
        "Do NOT rerun A8 and do NOT kill anything yet."
    )

elif raw_status == "COMPLETE_120_ROWS":

    print(
        "A complete 120-row A8 artifact exists despite "
        "the notebook KeyboardInterrupt."
    )

    print(
        "Do NOT rerun it. We must validate this existing "
        "artifact before deciding whether it is usable."
    )

elif raw_status == "PARTIAL":

    print(
        "A partial A8 artifact exists."
    )

    print(
        "Preserve it exactly as-is. Do NOT overwrite or "
        "resume it yet."
    )

elif raw_status == "ABSENT":

    print(
        "No A8 raw artifact exists."
    )

    print(
        "The interrupted attempt still needs to be documented "
        "before a fresh attempt is considered."
    )

else:

    print(
        "The A8 artifact requires further inspection."
    )

print()
print("NO model/API calls made.")
print("NO files modified.")
print("DO NOT rerun A8.")
print("DO NOT run A9.")
print("=" * 110)

STEP 21A — INSPECT INTERRUPTED A8 STATE

IMPORTANT
--------------------------------------------------------------------------------------------------------------
No rerun will be performed.
No files will be modified.
No model/API calls will be made.

1. PROCESS CHECK
--------------------------------------------------------------------------------------------------------------
No matching A8 runner process found.

2. RAW OUTPUT FILE
--------------------------------------------------------------------------------------------------------------
Path   : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/raw/A8_ternary-bonsai-27b_rerun.csv
Exists : False

3. BENCHMARK ALIGNMENT
--------------------------------------------------------------------------------------------------------------

4. EXECUTION LOG
--------------------------------------------------------------------------------------------------------------
Path   : /content/drive/MyDr

In [26]:
# ==========================================================================================
# STEP 21B — DOCUMENT INTERRUPTED A8 ATTEMPT
#
# NO model/API calls.
# NO frozen evidence modified.
# Creates ONE provenance record for the interrupted Step 21 attempt.
# ==========================================================================================

from pathlib import Path
import json
import hashlib

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

INTENDED_RAW = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "raw"
    / "A8_ternary-bonsai-27b_rerun.csv"
)

INTENDED_LOG = (
    BASE
    / "rerun-missing-artifacts"
    / "logs"
    / "A8_ternary-bonsai-27b_rerun.log"
)

OUT = (
    BASE
    / "rerun-missing-artifacts"
    / "manifests"
    / "A8_interrupted_attempt_20261003_v1.json"
)

MODEL = "Prism-ML/Ternary-Bonsai-27B"

STARTED_UTC = "2026-10-03T08:57:34.200634+00:00"


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


print("=" * 110)
print("STEP 21B — DOCUMENT INTERRUPTED A8 ATTEMPT")
print("=" * 110)

# ------------------------------------------------------------------------------------------
# Safety
# ------------------------------------------------------------------------------------------

if OUT.exists():
    raise FileExistsError(
        f"STOP: provenance record already exists:\n{OUT}\n"
        "Existing evidence will NOT be overwritten."
    )

if not RUNNER.exists():
    raise FileNotFoundError(
        f"STOP: canonical runner missing:\n{RUNNER}"
    )

if not DATASET.exists():
    raise FileNotFoundError(
        f"STOP: frozen dataset missing:\n{DATASET}"
    )

# Step 21A established these should not exist.
# Verify again immediately before recording provenance.

raw_exists = INTENDED_RAW.exists()
log_exists = INTENDED_LOG.exists()

if raw_exists or log_exists:

    raise RuntimeError(
        "STOP: filesystem state changed since Step 21A.\n"
        f"Raw exists: {raw_exists}\n"
        f"Log exists: {log_exists}\n"
        "Do not create the interruption record until this is investigated."
    )

# ------------------------------------------------------------------------------------------
# Provenance record
# ------------------------------------------------------------------------------------------

record = {

    "record_type":
        "interrupted_confirmatory_rerun_attempt",

    "experiment":
        "Experiment A missing-artifact confirmatory recovery",

    "arm":
        "A8",

    "model":
        MODEL,

    "attempt":
        1,

    "started_utc":
        STARTED_UTC,

    "termination_observation":
        "Parent Colab cell raised KeyboardInterrupt while waiting in subprocess.run().",

    "subprocess_return_code":
        None,

    "subprocess_completion_status":
        "UNKNOWN_DUE_TO_PARENT_KEYBOARD_INTERRUPT",

    "post_interrupt_inspection": {

        "runner_process_still_running":
            False,

        "raw_output_exists":
            False,

        "execution_log_exists":
            False,

        "recoverable_result_rows":
            0,
    },

    "interpretation": (
        "The interrupted attempt produced no surviving raw result "
        "artifact and no execution log. The interruption is recorded "
        "for provenance only and is not treated as an evaluable A8 run, "
        "a model failure, an API failure, or a benchmark result."
    ),

    "rerun_policy": (
        "A subsequent fresh A8 attempt may be performed as a new "
        "confirmatory recovery attempt. It must create new raw and log "
        "artifact names and must not be represented as continuation or "
        "completion of this interrupted attempt."
    ),

    "canonical_runner": {
        "path":
            str(RUNNER),

        "sha256":
            sha256(RUNNER),
    },

    "frozen_dataset": {
        "path":
            str(DATASET),

        "sha256":
            sha256(DATASET),
    },

    "intended_raw_path_attempt_1":
        str(INTENDED_RAW),

    "intended_log_path_attempt_1":
        str(INTENDED_LOG),

    "frozen_experiment_modified":
        False,

    "model_or_api_call_during_this_documentation_step":
        False,
}

OUT.write_text(
    json.dumps(
        record,
        indent=2
    ),
    encoding="utf-8"
)

print("\nPROVENANCE RECORD")
print("-" * 110)

print("Path   :", OUT)
print("SHA256 :", sha256(OUT))

print("\nRECORDED STATE")
print("-" * 110)

print("Arm                    : A8")
print("Attempt                : 1")
print("Started UTC            :", STARTED_UTC)
print("Termination            : KeyboardInterrupt in parent Colab cell")
print("Surviving raw artifact : NO")
print("Surviving log          : NO")
print("Surviving result rows  : 0")
print("Evaluable benchmark run: NO")

print("\nINTERPRETATION")
print("-" * 110)

print(
    "Attempt 1 is provenance-only. It is NOT classified as a "
    "model/API failure and contributes no A8 benchmark metrics."
)

print(
    "A future A8 execution must be recorded as a NEW attempt."
)

print("\n" + "=" * 110)
print("STEP 21B COMPLETE")
print("=" * 110)

print("No model/API calls made.")
print("No frozen Experiment-A evidence modified.")
print("No raw result reconstructed.")
print()
print("DO NOT rerun A8 yet.")
print("DO NOT run A9.")
print("=" * 110)

STEP 21B — DOCUMENT INTERRUPTED A8 ATTEMPT

PROVENANCE RECORD
--------------------------------------------------------------------------------------------------------------
Path   : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/manifests/A8_interrupted_attempt_20261003_v1.json
SHA256 : 07f5df22379747c8873a67318f34f5cc1967604c4d1862d6d2ffdb4d52028274

RECORDED STATE
--------------------------------------------------------------------------------------------------------------
Arm                    : A8
Attempt                : 1
Started UTC            : 2026-10-03T08:57:34.200634+00:00
Termination            : KeyboardInterrupt in parent Colab cell
Surviving raw artifact : NO
Surviving log          : NO
Surviving result rows  : 0
Evaluable benchmark run: NO

INTERPRETATION
--------------------------------------------------------------------------------------------------------------
Attempt 1 is provenance-only. It is NOT classified as a mod

In [27]:
# ==========================================================================================
# STEP 22 — FULL CONFIRMATORY RERUN: A8 — ATTEMPT 2
#
# Fresh execution after documented non-evaluable Attempt 1.
# Makes real model/API calls.
# ==========================================================================================

import os
import sys
import subprocess
from pathlib import Path
from datetime import datetime, timezone

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

OUT = (
    BASE
    / "rerun-missing-artifacts"
    / "results"
    / "raw"
    / "A8_ternary-bonsai-27b_rerun_attempt2.csv"
)

LOG = (
    BASE
    / "rerun-missing-artifacts"
    / "logs"
    / "A8_ternary-bonsai-27b_rerun_attempt2.log"
)

MODEL = "Prism-ML/Ternary-Bonsai-27B"

print("=" * 110)
print("STEP 22 — FULL A8 CONFIRMATORY RERUN — ATTEMPT 2")
print("=" * 110)

print("\nArm     : A8")
print("Attempt : 2")
print("Model   :", MODEL)
print("Dataset :", DATASET)
print("Output  :", OUT)
print("Log     :", LOG)
print("Cases   : 120")

# ------------------------------------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------------------------------------

if OUT.exists():
    raise FileExistsError(
        f"STOP: Attempt-2 output already exists:\n{OUT}\n"
        "Existing evidence will NOT be overwritten."
    )

if LOG.exists():
    raise FileExistsError(
        f"STOP: Attempt-2 log already exists:\n{LOG}\n"
        "Existing evidence will NOT be overwritten."
    )

if not os.environ.get("TOGETHER_API_KEY"):
    raise RuntimeError(
        "STOP: TOGETHER_API_KEY is missing."
    )

if not RUNNER.exists():
    raise FileNotFoundError(
        f"STOP: Canonical runner missing:\n{RUNNER}"
    )

if not DATASET.exists():
    raise FileNotFoundError(
        f"STOP: Frozen benchmark missing:\n{DATASET}"
    )

# ------------------------------------------------------------------------------------------
# Execute canonical runner
# ------------------------------------------------------------------------------------------

cmd = [
    sys.executable,
    str(RUNNER),
    "--dataset", str(DATASET),
    "--out", str(OUT),
    "--model", MODEL,
]

env = os.environ.copy()
env["PYTHONPATH"] = str(MASTER)

started = datetime.now(timezone.utc)

print("\nStarted UTC:", started.isoformat())
print("\nRunning A8 Attempt 2...")
print("-" * 110)

try:

    result = subprocess.run(
        cmd,
        cwd=str(MASTER),
        env=env,
        text=True,
        capture_output=True,
    )

except KeyboardInterrupt:

    print()
    print("=" * 110)
    print("STEP 22 INTERRUPTED")
    print("=" * 110)

    print(
        "The parent Colab cell received KeyboardInterrupt."
    )

    print(
        "Do NOT rerun this cell."
    )

    print(
        "Do NOT delete any output that may have been created."
    )

    print(
        "We must inspect the filesystem/process state first."
    )

    print()
    print("DO NOT run A9.")

    raise

finished = datetime.now(timezone.utc)

# ------------------------------------------------------------------------------------------
# Persist execution log only after subprocess returned normally
# ------------------------------------------------------------------------------------------

log_text = f"""STEP 22 — FULL A8 CONFIRMATORY RERUN — ATTEMPT 2

Arm: A8
Attempt: 2
Model: {MODEL}
Dataset: {DATASET}
Output: {OUT}

Started UTC: {started.isoformat()}
Finished UTC: {finished.isoformat()}
Return code: {result.returncode}

================ STDOUT ================

{result.stdout}

================ STDERR ================

{result.stderr}
"""

LOG.write_text(
    log_text,
    encoding="utf-8"
)

# ------------------------------------------------------------------------------------------
# Display execution result
# ------------------------------------------------------------------------------------------

print("\nRETURN CODE:", result.returncode)

print("\nSTDOUT")
print("-" * 110)

print(
    result.stdout
    if result.stdout
    else "[no stdout]"
)

print("\nSTDERR")
print("-" * 110)

print(
    result.stderr
    if result.stderr
    else "[no stderr]"
)

print("\nOUTPUT")
print("-" * 110)

print(
    "Exists :",
    OUT.exists()
)

if OUT.exists():

    print(
        "Size   :",
        OUT.stat().st_size,
        "bytes"
    )

print("\nLOG")
print("-" * 110)

print(
    "Exists :",
    LOG.exists()
)

print(
    "Path   :",
    LOG
)

print(
    "\nFinished UTC:",
    finished.isoformat()
)

# ------------------------------------------------------------------------------------------
# Final execution-level assessment
# ------------------------------------------------------------------------------------------

print("\n" + "=" * 110)

if (
    result.returncode == 0
    and OUT.exists()
):

    print(
        "STEP 22 EXECUTION COMPLETED"
    )

    print(
        "A8 Attempt-2 raw artifact was created."
    )

    print()
    print("IMPORTANT:")

    print(
        "This is only an execution-level success."
    )

    print(
        "A8 Attempt 2 is NOT yet accepted as "
        "a technically valid recovery run."
    )

    print(
        "We must inspect all 120 rows, exceptions, "
        "ID alignment, parsing, and containment."
    )

    print()
    print("DO NOT run A9 yet.")

else:

    print(
        "STEP 22 DID NOT COMPLETE SUCCESSFULLY"
    )

    print(
        "STOP HERE."
    )

    print(
        "Do NOT rerun automatically."
    )

    print(
        "DO NOT run A9."
    )

print("=" * 110)

STEP 22 — FULL A8 CONFIRMATORY RERUN — ATTEMPT 2

Arm     : A8
Attempt : 2
Model   : Prism-ML/Ternary-Bonsai-27B
Dataset : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL/experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3_heldout_benchmark_120_v1.csv
Output  : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/raw/A8_ternary-bonsai-27b_rerun_attempt2.csv
Log     : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/logs/A8_ternary-bonsai-27b_rerun_attempt2.log
Cases   : 120

Started UTC: 2026-10-03T09:06:12.451633+00:00

Running A8 Attempt 2...
--------------------------------------------------------------------------------------------------------------

STEP 22 INTERRUPTED
The parent Colab cell received KeyboardInterrupt.
Do NOT rerun this cell.
Do NOT delete any output that may have been created.
We must inspect the filesystem/proc

KeyboardInterrupt: 

In [28]:
# ==========================================================================================
# STEP 22A — SAVE CURRENT RECOVERY CHECKPOINT
#
# Purpose:
#   Preserve an inventory of ALL evidence currently present under
#   rerun-missing-artifacts so that work can resume safely after
#   a Colab disconnect/runtime reset.
#
# NO model/API calls.
# NO existing evidence modified.
# NO frozen Experiment-A evidence modified.
# ==========================================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

WORKSPACE = BASE / "rerun-missing-artifacts"

CHECKPOINT_DIR = WORKSPACE / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_JSON = (
    CHECKPOINT_DIR
    / "recovery_checkpoint_after_A7_before_A8_completion_v1.json"
)

CHECKPOINT_TXT = (
    CHECKPOINT_DIR
    / "recovery_checkpoint_after_A7_before_A8_completion_v1.txt"
)

print("=" * 110)
print("STEP 22A — SAVE CURRENT RECOVERY CHECKPOINT")
print("=" * 110)

# ------------------------------------------------------------------------------------------
# Safety — never overwrite an existing checkpoint
# ------------------------------------------------------------------------------------------

if CHECKPOINT_JSON.exists():
    raise FileExistsError(
        f"STOP: checkpoint already exists:\n{CHECKPOINT_JSON}"
    )

if CHECKPOINT_TXT.exists():
    raise FileExistsError(
        f"STOP: checkpoint already exists:\n{CHECKPOINT_TXT}"
    )

if not WORKSPACE.exists():
    raise FileNotFoundError(
        f"STOP: recovery workspace missing:\n{WORKSPACE}"
    )


# ------------------------------------------------------------------------------------------
# SHA-256 helper
# ------------------------------------------------------------------------------------------

def sha256(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


# ------------------------------------------------------------------------------------------
# Inventory every existing file
#
# Exclude the checkpoint files currently being created.
# ------------------------------------------------------------------------------------------

inventory = []

for path in sorted(WORKSPACE.rglob("*")):

    if not path.is_file():
        continue

    if path in {
        CHECKPOINT_JSON,
        CHECKPOINT_TXT
    }:
        continue

    relative = path.relative_to(WORKSPACE)

    inventory.append({
        "relative_path": str(relative),
        "absolute_path": str(path),
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
        "modified_utc": datetime.fromtimestamp(
            path.stat().st_mtime,
            tz=timezone.utc
        ).isoformat(),
    })


# ------------------------------------------------------------------------------------------
# Check important known artifacts explicitly
# ------------------------------------------------------------------------------------------

KNOWN = {

    "A2_raw":
        WORKSPACE
        / "results/raw/A2_llama-3.3-70b_rerun.csv",

    "A2_step15_metrics_preserved":
        WORKSPACE
        / "results/metrics/A2_llama-3.3-70b_rerun_metrics_v1.json",

    "A2_canonical_containment":
        WORKSPACE
        / "results/metrics/A2_canonical_containment_recheck_v1.json",

    "A7_raw":
        WORKSPACE
        / "results/raw/A7_qwen-2-1.5b_rerun.csv",

    "A7_metrics":
        WORKSPACE
        / "results/metrics/A7_qwen-2-1.5b_rerun_metrics_v1.json",

    "A8_attempt1_provenance":
        WORKSPACE
        / "manifests/A8_interrupted_attempt_20261003_v1.json",

    "A8_attempt2_raw":
        WORKSPACE
        / "results/raw/A8_ternary-bonsai-27b_rerun_attempt2.csv",

    "A8_attempt2_log":
        WORKSPACE
        / "logs/A8_ternary-bonsai-27b_rerun_attempt2.log",
}


known_state = {}

for name, path in KNOWN.items():

    if path.exists():

        known_state[name] = {
            "exists": True,
            "path": str(path),
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        }

    else:

        known_state[name] = {
            "exists": False,
            "path": str(path),
        }


# ------------------------------------------------------------------------------------------
# Record established experimental state
# ------------------------------------------------------------------------------------------

checkpoint = {

    "checkpoint_type":
        "confirmatory_recovery_checkpoint",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "workspace":
        str(WORKSPACE),

    "purpose": (
        "Persistent checkpoint for resuming Experiment-A "
        "missing-artifact confirmatory reruns after a Colab "
        "disconnect or runtime reset."
    ),

    "evidence_boundary": (
        "All reruns in this workspace are confirmatory/recovery "
        "evidence only and do not replace frozen Experiment-A results."
    ),

    "completed_arms": {

        "A2": {
            "status":
                "COMPLETE_AND_VALIDATED",

            "today_accuracy_pct":
                82.50,

            "today_VAR_pct":
                82.76,

            "today_IRR_pct":
                82.26,

            "canonical_containment_violations":
                0,

            "frozen_accuracy_pct":
                83.33,

            "frozen_VAR_pct":
                82.76,

            "frozen_IRR_pct":
                83.87,

            "frozen_containment_violations":
                0,
        },

        "A7": {
            "status":
                "COMPLETE_AND_VALIDATED",

            "today_accuracy_pct":
                60.00,

            "today_VAR_pct":
                18.97,

            "today_IRR_pct":
                98.39,

            "canonical_containment_violations":
                0,

            "frozen_accuracy_pct":
                60.00,

            "frozen_VAR_pct":
                18.97,

            "frozen_IRR_pct":
                98.39,

            "frozen_containment_violations":
                0,

            "aggregate_reproduction":
                "EXACT",
        },
    },

    "A8": {

        "attempt_1": {
            "status":
                "NON_EVALUABLE_INTERRUPTED",

            "raw_rows_surviving":
                0,

            "raw_artifact":
                False,

            "execution_log":
                False,

            "classification":
                "provenance only; not model/API failure",
        },

        "attempt_2": {
            "status":
                "IN_PROGRESS_OR_NOT_YET_VALIDATED",

            "instruction_after_resume": (
                "Inspect filesystem and process state before "
                "rerunning or accepting A8 Attempt 2."
            ),
        },
    },

    "A9": {
        "status":
            "NOT_STARTED_FULL_RERUN",
    },

    "canonical_containment_definition": (
        "accepted == False AND descriptor_json is present"
    ),

    "known_artifacts":
        known_state,

    "workspace_file_count_before_checkpoint":
        len(inventory),

    "workspace_inventory":
        inventory,

    "resume_rule": (
        "After runtime reset, remount Drive and inspect this "
        "checkpoint plus current A8 Attempt-2 filesystem state. "
        "Do not repeat A2 or A7."
    ),
}


# ------------------------------------------------------------------------------------------
# Save JSON checkpoint
# ------------------------------------------------------------------------------------------

CHECKPOINT_JSON.write_text(
    json.dumps(
        checkpoint,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------------------------------------
# Save human-readable checkpoint
# ------------------------------------------------------------------------------------------

lines = []

lines.append(
    "EXPERIMENT-A MISSING-ARTIFACT RECOVERY CHECKPOINT"
)

lines.append("=" * 90)

lines.append(
    f"Created UTC: {checkpoint['created_utc']}"
)

lines.append("")

lines.append("COMPLETED")
lines.append("-" * 90)

lines.append(
    "A2 : COMPLETE + VALIDATED"
)

lines.append(
    "     Today: Accuracy 82.50%, VAR 82.76%, "
    "IRR 82.26%, containment 0"
)

lines.append(
    "     Frozen: Accuracy 83.33%, VAR 82.76%, "
    "IRR 83.87%, containment 0"
)

lines.append("")

lines.append(
    "A7 : COMPLETE + VALIDATED"
)

lines.append(
    "     Today: Accuracy 60.00%, VAR 18.97%, "
    "IRR 98.39%, containment 0"
)

lines.append(
    "     Frozen: Accuracy 60.00%, VAR 18.97%, "
    "IRR 98.39%, containment 0"
)

lines.append(
    "     Aggregate reproduction: EXACT"
)

lines.append("")

lines.append("A8")
lines.append("-" * 90)

lines.append(
    "Attempt 1: NON-EVALUABLE / INTERRUPTED"
)

lines.append(
    "No raw artifact, no log, zero surviving rows."
)

lines.append(
    "Attempt 2: IN PROGRESS OR NOT YET VALIDATED."
)

lines.append(
    "After reconnect: inspect A8 Attempt-2 state BEFORE rerunning."
)

lines.append("")

lines.append("A9")
lines.append("-" * 90)

lines.append(
    "Full confirmatory rerun NOT STARTED."
)

lines.append("")

lines.append("CANONICAL CONTAINMENT")
lines.append("-" * 90)

lines.append(
    "Violation = accepted == False AND descriptor_json is present"
)

lines.append("")

lines.append("RESUME INSTRUCTION")
lines.append("-" * 90)

lines.append(
    "1. Mount Google Drive."
)

lines.append(
    "2. Open rerun-missing-artifacts/checkpoints."
)

lines.append(
    "3. Verify this checkpoint."
)

lines.append(
    "4. Inspect A8 Attempt-2 raw/log/process state."
)

lines.append(
    "5. DO NOT repeat A2 or A7."
)

lines.append(
    "6. Continue A8, then A9."
)

lines.append("")

lines.append("ARTIFACT INVENTORY")
lines.append("-" * 90)

for item in inventory:

    lines.append(
        f"{item['relative_path']} | "
        f"{item['size_bytes']} bytes | "
        f"SHA256={item['sha256']}"
    )

CHECKPOINT_TXT.write_text(
    "\n".join(lines),
    encoding="utf-8"
)


# ------------------------------------------------------------------------------------------
# Final hashes
# ------------------------------------------------------------------------------------------

print("\nCHECKPOINT CREATED")
print("-" * 110)

print(
    "JSON :",
    CHECKPOINT_JSON
)

print(
    "SHA  :",
    sha256(CHECKPOINT_JSON)
)

print()

print(
    "TXT  :",
    CHECKPOINT_TXT
)

print(
    "SHA  :",
    sha256(CHECKPOINT_TXT)
)

print("\nWORKSPACE INVENTORY")
print("-" * 110)

print(
    "Files protected by inventory:",
    len(inventory)
)

print()

for key, value in known_state.items():

    print(
        f"{key:<30}:",
        "PRESENT"
        if value["exists"]
        else "ABSENT"
    )


print("\n" + "=" * 110)
print("CHECKPOINT STATUS: SAVED TO GOOGLE DRIVE")
print("=" * 110)

print()
print("SAFE RESUME POINT:")
print("A2 = COMPLETE")
print("A7 = COMPLETE")
print("A8 Attempt 1 = DOCUMENTED")
print("A8 Attempt 2 = INSPECT AFTER RESUME")
print("A9 = NOT STARTED")

print()
print(
    "If Colab disconnects, DO NOT rerun A2 or A7."
)

print(
    "After reconnecting, inspect A8 Attempt 2 first."
)

print()
print("No model/API calls made.")
print("No existing evidence modified.")
print("No frozen Experiment-A evidence modified.")

print("=" * 110)

STEP 22A — SAVE CURRENT RECOVERY CHECKPOINT

CHECKPOINT CREATED
--------------------------------------------------------------------------------------------------------------
JSON : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/checkpoints/recovery_checkpoint_after_A7_before_A8_completion_v1.json
SHA  : 280d1228520d0a79d6d3cd2d3b03a40e8297adcaf55a43bb7cec1873d35af7c5

TXT  : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/checkpoints/recovery_checkpoint_after_A7_before_A8_completion_v1.txt
SHA  : d6e73dcf63fbcb3253a6ddba8406132cf04e320c80b36afa81f55cc4e234d7f1

WORKSPACE INVENTORY
--------------------------------------------------------------------------------------------------------------
Files protected by inventory: 13

A2_raw                        : PRESENT
A2_step15_metrics_preserved   : PRESENT
A2_canonical_containment      : PRESENT
A7_raw                        : PRESENT
A7_metrics     

In [29]:
# ==========================================================================================
# STEP 22B — INSPECT INTERRUPTED A8 ATTEMPT 2
#
# NO model/API calls.
# NO files modified.
# DO NOT rerun A8.
# ==========================================================================================

from pathlib import Path
import pandas as pd
import hashlib
import subprocess

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

OUT = (
    BASE
    / "rerun-missing-artifacts"
    / "results/raw/"
    "A8_ternary-bonsai-27b_rerun_attempt2.csv"
)

LOG = (
    BASE
    / "rerun-missing-artifacts"
    / "logs/"
    "A8_ternary-bonsai-27b_rerun_attempt2.log"
)

MODEL = "Prism-ML/Ternary-Bonsai-27B"


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)
    return h.hexdigest()


print("=" * 110)
print("STEP 22B — INSPECT INTERRUPTED A8 ATTEMPT 2")
print("=" * 110)

# ==========================================================================================
# 1. PROCESS CHECK
# ==========================================================================================

print("\n1. PROCESS CHECK")
print("-" * 110)

ps = subprocess.run(
    ["ps", "-eo", "pid,ppid,stat,etime,args"],
    text=True,
    capture_output=True,
)

matching = []

for line in ps.stdout.splitlines():

    if (
        "run_experiment2b_e3v3_r7_dev_v1.py" in line
        or MODEL in line
    ):
        matching.append(line)

if matching:

    process_status = "POTENTIALLY_RUNNING"

    print("Potential matching process(es) FOUND:")
    print()

    for line in matching:
        print(line)

else:

    process_status = "NOT_RUNNING"

    print("No matching A8 runner process found.")


# ==========================================================================================
# 2. RAW OUTPUT
# ==========================================================================================

print("\n2. ATTEMPT-2 RAW OUTPUT")
print("-" * 110)

print("Path   :", OUT)
print("Exists :", OUT.exists())

raw_status = "ABSENT"
row_count = 0

if OUT.exists():

    print(
        "Size   :",
        OUT.stat().st_size,
        "bytes"
    )

    print(
        "SHA256 :",
        sha256(OUT)
    )

    try:

        df = pd.read_csv(OUT)

        row_count = len(df)

        print("Readable: YES")
        print("Rows    :", row_count)
        print("Columns :", len(df.columns))

        if "id" in df.columns and row_count:

            ids = df["id"].astype(str)

            print(
                "Unique IDs    :",
                ids.nunique()
            )

            print(
                "Duplicate IDs :",
                int(ids.duplicated().sum())
            )

            print(
                "First ID      :",
                ids.iloc[0]
            )

            print(
                "Last ID       :",
                ids.iloc[-1]
            )

        if row_count == 120:
            raw_status = "COMPLETE_120_ROWS"

        elif row_count > 0:
            raw_status = "PARTIAL"

        else:
            raw_status = "EMPTY"

    except Exception as exc:

        raw_status = "UNREADABLE"

        print("Readable: NO")
        print(
            "Error   :",
            type(exc).__name__,
            str(exc)
        )


# ==========================================================================================
# 3. BENCHMARK PREFIX / ORDER
# ==========================================================================================

print("\n3. BENCHMARK ALIGNMENT")
print("-" * 110)

if (
    OUT.exists()
    and raw_status in {
        "PARTIAL",
        "COMPLETE_120_ROWS"
    }
):

    df = pd.read_csv(OUT)
    benchmark = pd.read_csv(DATASET)

    if (
        "id" in df.columns
        and "id" in benchmark.columns
    ):

        result_ids = (
            df["id"]
            .astype(str)
            .tolist()
        )

        benchmark_ids = (
            benchmark["id"]
            .astype(str)
            .tolist()
        )

        prefix_match = (
            result_ids
            == benchmark_ids[:len(result_ids)]
        )

        print(
            "Benchmark prefix match :",
            "YES" if prefix_match else "NO"
        )

        if len(df) == 120:

            full_match = (
                result_ids == benchmark_ids
            )

            print(
                "Full order match      :",
                "YES" if full_match else "NO"
            )

else:

    print(
        "No readable result rows available "
        "for alignment check."
    )


# ==========================================================================================
# 4. LOG CHECK
# ==========================================================================================

print("\n4. ATTEMPT-2 LOG")
print("-" * 110)

print("Path   :", LOG)
print("Exists :", LOG.exists())

if LOG.exists():

    print(
        "Size   :",
        LOG.stat().st_size,
        "bytes"
    )

    print(
        "SHA256 :",
        sha256(LOG)
    )

    text = LOG.read_text(
        encoding="utf-8",
        errors="replace"
    )

    print("\nLast 30 lines:")
    print("-" * 110)

    for line in text.splitlines()[-30:]:
        print(line)


# ==========================================================================================
# 5. HEALTH CHECK IF ROWS SURVIVED
# ==========================================================================================

print("\n5. SURVIVING ROW HEALTH")
print("-" * 110)

if raw_status in {
    "PARTIAL",
    "COMPLETE_120_ROWS"
}:

    df = pd.read_csv(OUT)

    if "exception" in df.columns:

        exc = (
            df["exception"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        exc_mask = ~exc.str.lower().isin(
            [
                "",
                "nan",
                "none",
                "null",
                "false"
            ]
        )

        print(
            "Populated exceptions :",
            int(exc_mask.sum()),
            "/",
            len(df)
        )

    if "llm_invoked" in df.columns:

        invoked = (
            df["llm_invoked"].eq(True)
        )

        print(
            "LLM invoked          :",
            int(invoked.sum()),
            "/",
            len(df)
        )

        if "parse_ok" in df.columns:

            invoked_df = df.loc[invoked]

            parse_success = int(
                invoked_df["parse_ok"]
                .eq(True)
                .sum()
            )

            print(
                "Parse OK             :",
                parse_success,
                "/",
                len(invoked_df)
            )

else:

    print(
        "No surviving result rows."
    )


# ==========================================================================================
# 6. FINAL ASSESSMENT
# ==========================================================================================

print("\n" + "=" * 110)
print("STEP 22B — ATTEMPT-2 STATE")
print("=" * 110)

print(
    "Runner process :",
    process_status
)

print(
    "Raw status     :",
    raw_status
)

print(
    "Rows surviving :",
    row_count
)

print(
    "Log exists     :",
    LOG.exists()
)

print()

if process_status == "POTENTIALLY_RUNNING":

    print(
        "STOP: An A8 runner may still be active."
    )

    print(
        "Do not start another A8 attempt."
    )

elif raw_status == "COMPLETE_120_ROWS":

    print(
        "IMPORTANT: Attempt 2 produced a complete "
        "120-row artifact."
    )

    print(
        "Do NOT rerun A8. We will validate this "
        "existing artifact next."
    )

elif raw_status == "PARTIAL":

    print(
        "Attempt 2 left a partial artifact."
    )

    print(
        "Do NOT delete or overwrite it."
    )

    print(
        "We will preserve/hash it before deciding "
        "how to proceed."
    )

elif raw_status == "ABSENT":

    print(
        "Attempt 2 produced no surviving raw artifact."
    )

    print(
        "We will document Attempt 2 and change the "
        "execution strategy before any Attempt 3."
    )

else:

    print(
        "Attempt 2 requires further investigation."
    )

print()
print("NO model/API calls made.")
print("NO files modified.")
print("DO NOT run A8 again yet.")
print("DO NOT run A9.")
print("=" * 110)

STEP 22B — INSPECT INTERRUPTED A8 ATTEMPT 2

1. PROCESS CHECK
--------------------------------------------------------------------------------------------------------------
No matching A8 runner process found.

2. ATTEMPT-2 RAW OUTPUT
--------------------------------------------------------------------------------------------------------------
Path   : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/raw/A8_ternary-bonsai-27b_rerun_attempt2.csv
Exists : False

3. BENCHMARK ALIGNMENT
--------------------------------------------------------------------------------------------------------------
No readable result rows available for alignment check.

4. ATTEMPT-2 LOG
--------------------------------------------------------------------------------------------------------------
Path   : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/logs/A8_ternary-bonsai-27b_rerun_attempt2.log
Exists : False

5

In [30]:
# ==========================================================================================
# STEP 22C — DOCUMENT INTERRUPTED A8 ATTEMPT 2
#
# NO model/API calls.
# NO existing evidence modified.
# Creates one provenance record only.
# ==========================================================================================

from pathlib import Path
import json
import hashlib

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

INTENDED_RAW = (
    BASE
    / "rerun-missing-artifacts/results/raw/"
    "A8_ternary-bonsai-27b_rerun_attempt2.csv"
)

INTENDED_LOG = (
    BASE
    / "rerun-missing-artifacts/logs/"
    "A8_ternary-bonsai-27b_rerun_attempt2.log"
)

OUT = (
    BASE
    / "rerun-missing-artifacts/manifests/"
    "A8_interrupted_attempt2_20261003_v1.json"
)

MODEL = "Prism-ML/Ternary-Bonsai-27B"

STARTED_UTC = "2026-10-03T09:06:12.451633+00:00"


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


print("=" * 110)
print("STEP 22C — DOCUMENT INTERRUPTED A8 ATTEMPT 2")
print("=" * 110)

# ------------------------------------------------------------------------------------------
# Safety
# ------------------------------------------------------------------------------------------

if OUT.exists():
    raise FileExistsError(
        f"STOP: provenance record already exists:\n{OUT}"
    )

if not RUNNER.exists():
    raise FileNotFoundError(
        f"STOP: canonical runner missing:\n{RUNNER}"
    )

if not DATASET.exists():
    raise FileNotFoundError(
        f"STOP: frozen dataset missing:\n{DATASET}"
    )

# Step 22B established both should be absent.
# Verify again before recording.

if INTENDED_RAW.exists() or INTENDED_LOG.exists():

    raise RuntimeError(
        "STOP: filesystem state changed since Step 22B.\n"
        f"Raw exists: {INTENDED_RAW.exists()}\n"
        f"Log exists: {INTENDED_LOG.exists()}"
    )


# ------------------------------------------------------------------------------------------
# Record
# ------------------------------------------------------------------------------------------

record = {

    "record_type":
        "interrupted_confirmatory_rerun_attempt",

    "experiment":
        "Experiment A missing-artifact confirmatory recovery",

    "arm":
        "A8",

    "model":
        MODEL,

    "attempt":
        2,

    "started_utc":
        STARTED_UTC,

    "termination_observation":
        "Parent Colab cell raised KeyboardInterrupt while waiting in subprocess.run().",

    "subprocess_return_code":
        None,

    "subprocess_completion_status":
        "UNKNOWN_DUE_TO_PARENT_KEYBOARD_INTERRUPT",

    "post_interrupt_inspection": {

        "runner_process_still_running":
            False,

        "raw_output_exists":
            False,

        "execution_log_exists":
            False,

        "recoverable_result_rows":
            0,
    },

    "interpretation": (
        "Attempt 2 produced no surviving raw result artifact "
        "and no execution log. It is therefore non-evaluable. "
        "The interruption is recorded for provenance and is not "
        "classified as a model failure, API failure, or benchmark result."
    ),

    "next_execution_policy": (
        "If A8 is attempted again, use a new Attempt-3 artifact "
        "name and change execution from captured subprocess output "
        "to live-streamed output so case-level progress is visible."
    ),

    "canonical_runner": {
        "path": str(RUNNER),
        "sha256": sha256(RUNNER),
    },

    "frozen_dataset": {
        "path": str(DATASET),
        "sha256": sha256(DATASET),
    },

    "intended_raw_path_attempt_2":
        str(INTENDED_RAW),

    "intended_log_path_attempt_2":
        str(INTENDED_LOG),

    "frozen_experiment_modified":
        False,

    "model_or_api_call_during_documentation":
        False,
}


OUT.write_text(
    json.dumps(
        record,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------------------------------------
# Result
# ------------------------------------------------------------------------------------------

print("\nPROVENANCE RECORD")
print("-" * 110)

print("Path   :", OUT)
print("SHA256 :", sha256(OUT))

print("\nRECORDED STATE")
print("-" * 110)

print("Arm                    : A8")
print("Attempt                : 2")
print("Started UTC            :", STARTED_UTC)
print("Termination            : KeyboardInterrupt in parent Colab cell")
print("Surviving raw artifact : NO")
print("Surviving log          : NO")
print("Surviving result rows  : 0")
print("Evaluable benchmark run: NO")

print("\nA8 ATTEMPT HISTORY")
print("-" * 110)

print("Attempt 1 : INTERRUPTED / NON-EVALUABLE / 0 surviving rows")
print("Attempt 2 : INTERRUPTED / NON-EVALUABLE / 0 surviving rows")

print("\nNEXT EXECUTION POLICY")
print("-" * 110)

print(
    "Attempt 3, if performed, must use NEW artifact names."
)

print(
    "Attempt 3 will use LIVE subprocess output rather than "
    "capture_output=True."
)

print("\n" + "=" * 110)
print("STEP 22C COMPLETE")
print("=" * 110)

print("No model/API calls made.")
print("No frozen Experiment-A evidence modified.")
print("No raw result reconstructed.")
print()
print("DO NOT run A8 yet.")
print("DO NOT run A9.")
print("=" * 110)

STEP 22C — DOCUMENT INTERRUPTED A8 ATTEMPT 2

PROVENANCE RECORD
--------------------------------------------------------------------------------------------------------------
Path   : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/manifests/A8_interrupted_attempt2_20261003_v1.json
SHA256 : bfc5f09702dcf619230a86cb61a2bc4f22e0aff91df62f18d2e65a01161bd33c

RECORDED STATE
--------------------------------------------------------------------------------------------------------------
Arm                    : A8
Attempt                : 2
Started UTC            : 2026-10-03T09:06:12.451633+00:00
Termination            : KeyboardInterrupt in parent Colab cell
Surviving raw artifact : NO
Surviving log          : NO
Surviving result rows  : 0
Evaluable benchmark run: NO

A8 ATTEMPT HISTORY
--------------------------------------------------------------------------------------------------------------
Attempt 1 : INTERRUPTED / NON-EVALUABLE / 0 survivin

In [1]:
# ==========================================================================================
# STEP 23 — FULL A8 CONFIRMATORY RERUN — ATTEMPT 3
# LIVE-STREAMING EXECUTION
#
# Improvements over Attempts 1/2:
#   1. Progress appears LIVE in Colab.
#   2. stdout/stderr are written LIVE to a persistent Drive log.
#   3. New Attempt-3 artifact names.
#   4. No automatic retry.
#   5. Existing evidence is never overwritten.
# ==========================================================================================

import os
import sys
import subprocess
from pathlib import Path
from datetime import datetime, timezone

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

OUT = (
    BASE
    / "rerun-missing-artifacts/results/raw/"
    "A8_ternary-bonsai-27b_rerun_attempt3.csv"
)

LOG = (
    BASE
    / "rerun-missing-artifacts/logs/"
    "A8_ternary-bonsai-27b_rerun_attempt3_live.log"
)

MODEL = "Prism-ML/Ternary-Bonsai-27B"

print("=" * 110)
print("STEP 23 — FULL A8 CONFIRMATORY RERUN — ATTEMPT 3 — LIVE")
print("=" * 110)

print("\nArm     : A8")
print("Attempt : 3")
print("Model   :", MODEL)
print("Dataset :", DATASET)
print("Output  :", OUT)
print("Log     :", LOG)
print("Cases   : 120")

# ==========================================================================================
# 1. SAFETY CHECKS
# ==========================================================================================

if OUT.exists():
    raise FileExistsError(
        f"STOP: Attempt-3 raw artifact already exists:\n{OUT}"
    )

if LOG.exists():
    raise FileExistsError(
        f"STOP: Attempt-3 live log already exists:\n{LOG}"
    )

if not os.environ.get("TOGETHER_API_KEY"):
    raise RuntimeError(
        "STOP: TOGETHER_API_KEY is missing."
    )

if not RUNNER.exists():
    raise FileNotFoundError(
        f"STOP: canonical runner missing:\n{RUNNER}"
    )

if not DATASET.exists():
    raise FileNotFoundError(
        f"STOP: frozen dataset missing:\n{DATASET}"
    )

# ==========================================================================================
# 2. COMMAND
# ==========================================================================================

cmd = [
    sys.executable,
    "-u",                    # unbuffered Python output
    str(RUNNER),
    "--dataset", str(DATASET),
    "--out", str(OUT),
    "--model", MODEL,
]

env = os.environ.copy()
env["PYTHONPATH"] = str(MASTER)

# Force unbuffered output from child Python process.
env["PYTHONUNBUFFERED"] = "1"

started = datetime.now(timezone.utc)

print("\nStarted UTC:", started.isoformat())

print("\nLIVE PROGRESS")
print("-" * 110)

# ==========================================================================================
# 3. WRITE LOG HEADER BEFORE MODEL EXECUTION
#
# Unlike Attempts 1/2, this log exists from the beginning.
# ==========================================================================================

with open(
    LOG,
    "x",
    encoding="utf-8",
    buffering=1
) as log:

    log.write(
        "STEP 23 — FULL A8 CONFIRMATORY RERUN — ATTEMPT 3 — LIVE\n"
    )

    log.write(f"Arm: A8\n")
    log.write(f"Attempt: 3\n")
    log.write(f"Model: {MODEL}\n")
    log.write(f"Dataset: {DATASET}\n")
    log.write(f"Output: {OUT}\n")
    log.write(f"Started UTC: {started.isoformat()}\n")
    log.write("=" * 110 + "\n")
    log.flush()

    process = None

    try:

        # Merge stderr into stdout so everything is streamed
        # in chronological order.

        process = subprocess.Popen(
            cmd,
            cwd=str(MASTER),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        # ----------------------------------------------------------
        # LIVE STREAM
        # ----------------------------------------------------------

        for line in iter(
            process.stdout.readline,
            ""
        ):

            # Show immediately in Colab
            print(
                line,
                end="",
                flush=True
            )

            # Persist immediately to Google Drive
            log.write(line)
            log.flush()

        process.stdout.close()

        return_code = process.wait()

        finished = datetime.now(timezone.utc)

        log.write("\n" + "=" * 110 + "\n")
        log.write(
            f"Finished UTC: {finished.isoformat()}\n"
        )
        log.write(
            f"Return code: {return_code}\n"
        )
        log.flush()

    except KeyboardInterrupt:

        interrupted = datetime.now(timezone.utc)

        print()
        print("=" * 110)
        print("STEP 23 — KEYBOARD INTERRUPT RECEIVED")
        print("=" * 110)

        print(
            "Do NOT rerun A8 automatically."
        )

        print(
            "The live log has already been preserved "
            "to Google Drive."
        )

        log.write("\n" + "=" * 110 + "\n")
        log.write(
            "PARENT COLAB KEYBOARD INTERRUPT RECEIVED\n"
        )
        log.write(
            f"Interrupted UTC: {interrupted.isoformat()}\n"
        )
        log.flush()

        # We deliberately do NOT delete any raw artifact.
        # We also do not classify the run here.

        raise

# ==========================================================================================
# 4. NORMAL-COMPLETION REPORT
# ==========================================================================================

print("\n" + "=" * 110)
print("STEP 23 — PROCESS RETURNED")
print("=" * 110)

print(
    "Return code :",
    return_code
)

print(
    "Raw exists  :",
    OUT.exists()
)

if OUT.exists():

    print(
        "Raw size    :",
        OUT.stat().st_size,
        "bytes"
    )

print(
    "Log exists  :",
    LOG.exists()
)

print(
    "Log size    :",
    LOG.stat().st_size,
    "bytes"
)

print(
    "Finished UTC:",
    finished.isoformat()
)

print()

if (
    return_code == 0
    and OUT.exists()
):

    print(
        "STEP 23 EXECUTION COMPLETED."
    )

    print(
        "A8 Attempt 3 produced a raw artifact."
    )

    print()
    print(
        "IMPORTANT: Do NOT compute metrics yet."
    )

    print(
        "We must validate all 120 rows first."
    )

else:

    print(
        "STEP 23 DID NOT PRODUCE A NORMAL "
        "SUCCESSFUL EXECUTION."
    )

    print(
        "STOP. Do not retry automatically."
    )

print()
print("DO NOT run A9 yet.")
print("=" * 110)

STEP 23 — FULL A8 CONFIRMATORY RERUN — ATTEMPT 3 — LIVE

Arm     : A8
Attempt : 3
Model   : Prism-ML/Ternary-Bonsai-27B
Dataset : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL/experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3_heldout_benchmark_120_v1.csv
Output  : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/raw/A8_ternary-bonsai-27b_rerun_attempt3.csv
Log     : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/logs/A8_ternary-bonsai-27b_rerun_attempt3_live.log
Cases   : 120


RuntimeError: STOP: TOGETHER_API_KEY is missing.

In [6]:
# ==========================================================================================
# STEP 23A — RESTORE TOGETHER API KEY AFTER COLAB RUNTIME RESET
#
# NO model/API calls.
# API key is NOT written to Google Drive.
# ==========================================================================================

import os
from getpass import getpass

print("=" * 90)
print("STEP 23A — RESTORE TOGETHER API KEY")
print("=" * 90)

# Remove any empty/stale value
os.environ.pop("TOGETHER_API_KEY", None)

key = getpass("Enter TOGETHER_API_KEY: ").strip()

if not key:
    raise RuntimeError("STOP: No API key entered.")

os.environ["TOGETHER_API_KEY"] = key

# Remove local variable after placing it in environment
del key

print()
print("TOGETHER_API_KEY present :",
      bool(os.environ.get("TOGETHER_API_KEY")))

print("Key length              :",
      len(os.environ["TOGETHER_API_KEY"]))

print("Key value               : HIDDEN")

print()
print("=" * 90)
print("API KEY RESTORED")
print("=" * 90)
print("No API/model call made.")
print("API key was not written to Drive.")

STEP 23A — RESTORE TOGETHER API KEY
Enter TOGETHER_API_KEY: ··········

TOGETHER_API_KEY present : True
Key length              : 2096
Key value               : HIDDEN

API KEY RESTORED
No API/model call made.
API key was not written to Drive.


In [4]:
import os
import sys
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

OUT = BASE / (
    "rerun-missing-artifacts/results/smoke/"
    "A8_direct_diagnostic_1case.csv"
)

MODEL = "Prism-ML/Ternary-Bonsai-27B"

print("=" * 90)
print("A8 DIRECT ONE-CASE DIAGNOSTIC")
print("=" * 90)

print("Python :", sys.executable)
print("Runner :", RUNNER)
print("Dataset:", DATASET)
print("Model  :", MODEL)
print("Output :", OUT)

assert RUNNER.exists(), "Runner missing"
assert DATASET.exists(), "Dataset missing"
assert os.environ.get("TOGETHER_API_KEY"), "TOGETHER_API_KEY missing"

if OUT.exists():
    raise RuntimeError(
        f"STOP: diagnostic output already exists:\n{OUT}"
    )

os.environ["PYTHONPATH"] = str(MASTER)

print("\nStarting ONE case only...\n")

%run "$RUNNER" \
    --dataset "$DATASET" \
    --out "$OUT" \
    --model "$MODEL" \
    --limit 1

A8 DIRECT ONE-CASE DIAGNOSTIC
Python : /usr/bin/python3
Runner : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL/experiment-2b/implementation/e3_v3/run_experiment2b_e3v3_r7_dev_v1.py
Dataset: /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL/experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3_heldout_benchmark_120_v1.csv
Model  : Prism-ML/Ternary-Bonsai-27B
Output : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/smoke/A8_direct_diagnostic_1case.csv


AssertionError: Runner missing

In [5]:
# ==========================================================================================
# STEP 23B — RESTORE / VERIFY GOOGLE DRIVE AFTER RUNTIME RESET
#
# NO model/API calls.
# NO experiment files modified.
# ==========================================================================================

from pathlib import Path
from google.colab import drive

print("=" * 100)
print("STEP 23B — RESTORE / VERIFY GOOGLE DRIVE")
print("=" * 100)

MOUNT = Path("/content/drive")
MYDRIVE = Path("/content/drive/MyDrive")

if not MYDRIVE.exists():
    print("\nGoogle Drive is not currently mounted.")
    print("Mounting now...\n")

    drive.mount("/content/drive")

else:
    print("\nGoogle Drive already appears mounted.")

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

CHECKPOINT = BASE / (
    "rerun-missing-artifacts/checkpoints/"
    "recovery_checkpoint_after_A7_before_A8_completion_v1.json"
)

print("\nPATH VERIFICATION")
print("-" * 100)

print("MyDrive    :", MYDRIVE.exists())
print("Base       :", BASE.exists())
print("Master     :", MASTER.exists())
print("Runner     :", RUNNER.exists())
print("Dataset    :", DATASET.exists())
print("Checkpoint :", CHECKPOINT.exists())

print("\nAPI ENVIRONMENT")
print("-" * 100)

import os

print(
    "TOGETHER_API_KEY present :",
    bool(os.environ.get("TOGETHER_API_KEY"))
)

print("\n" + "=" * 100)

if (
    MYDRIVE.exists()
    and BASE.exists()
    and MASTER.exists()
    and RUNNER.exists()
    and DATASET.exists()
    and CHECKPOINT.exists()
):

    print("DRIVE/EXPERIMENT STATE: RESTORED")

else:

    print("DRIVE/EXPERIMENT STATE: NOT FULLY RESTORED")
    print("STOP — do not run A8.")

print("=" * 100)
print("No model/API calls made.")
print("No experiment evidence modified.")

STEP 23B — RESTORE / VERIFY GOOGLE DRIVE

Google Drive is not currently mounted.
Mounting now...

Mounted at /content/drive

PATH VERIFICATION
----------------------------------------------------------------------------------------------------
MyDrive    : True
Base       : True
Master     : True
Runner     : True
Dataset    : True
Checkpoint : True

API ENVIRONMENT
----------------------------------------------------------------------------------------------------
TOGETHER_API_KEY present : True

DRIVE/EXPERIMENT STATE: RESTORED
No model/API calls made.
No experiment evidence modified.


In [7]:
# ==========================================================================================
# STEP R1 — RESTORE EXPERIMENT STATE FROM SAVED CHECKPOINT
#
# NO model/API calls.
# NO experiment files modified.
# Reads and verifies the saved checkpoint only.
# ==========================================================================================

from google.colab import drive
from pathlib import Path
import json
import hashlib
import os

print("=" * 110)
print("STEP R1 — RESTORE EXPERIMENT STATE FROM SAVED CHECKPOINT")
print("=" * 110)

# ------------------------------------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------------------------------------

MYDRIVE = Path("/content/drive/MyDrive")

if not MYDRIVE.exists():

    print("\nGoogle Drive is not mounted.")
    print("Mounting Google Drive...\n")

    drive.mount("/content/drive")

else:

    print("\nGoogle Drive already mounted.")


# ------------------------------------------------------------------------------------------
# 2. Define persistent paths
# ------------------------------------------------------------------------------------------

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

WORKSPACE = BASE / "rerun-missing-artifacts"

CHECKPOINT = WORKSPACE / (
    "checkpoints/"
    "recovery_checkpoint_after_A7_before_A8_completion_v1.json"
)

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)


# ------------------------------------------------------------------------------------------
# 3. SHA helper
# ------------------------------------------------------------------------------------------

def sha256(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


# ------------------------------------------------------------------------------------------
# 4. Verify checkpoint itself
# ------------------------------------------------------------------------------------------

EXPECTED_CHECKPOINT_SHA = (
    "280d1228520d0a79d6d3cd2d3b03a40e8297adcaf55a43bb7cec1873d35af7c5"
)

print("\nCHECKPOINT VERIFICATION")
print("-" * 110)

print("Path   :", CHECKPOINT)
print("Exists :", CHECKPOINT.exists())

if not CHECKPOINT.exists():

    raise FileNotFoundError(
        f"STOP: saved checkpoint not found:\n{CHECKPOINT}"
    )

actual_checkpoint_sha = sha256(CHECKPOINT)

print("SHA256 :", actual_checkpoint_sha)

if actual_checkpoint_sha != EXPECTED_CHECKPOINT_SHA:

    raise RuntimeError(
        "STOP: checkpoint SHA-256 does not match "
        "the checkpoint we previously created."
    )

print("SHA    : MATCH")


# ------------------------------------------------------------------------------------------
# 5. Load checkpoint
# ------------------------------------------------------------------------------------------

with open(
    CHECKPOINT,
    "r",
    encoding="utf-8"
) as f:

    checkpoint = json.load(f)

print("\nCheckpoint loaded successfully.")


# ------------------------------------------------------------------------------------------
# 6. Verify canonical runner and dataset
# ------------------------------------------------------------------------------------------

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)

EXPECTED_DATASET_SHA = (
    "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de"
)

print("\nCANONICAL INPUT VERIFICATION")
print("-" * 110)

print("Runner exists  :", RUNNER.exists())
print("Dataset exists :", DATASET.exists())

if not RUNNER.exists():

    raise FileNotFoundError(
        f"STOP: canonical runner missing:\n{RUNNER}"
    )

if not DATASET.exists():

    raise FileNotFoundError(
        f"STOP: canonical dataset missing:\n{DATASET}"
    )

runner_sha = sha256(RUNNER)
dataset_sha = sha256(DATASET)

print("Runner SHA     :", runner_sha)
print("Dataset SHA    :", dataset_sha)

if runner_sha != EXPECTED_RUNNER_SHA:

    raise RuntimeError(
        "STOP: canonical runner SHA mismatch."
    )

if dataset_sha != EXPECTED_DATASET_SHA:

    raise RuntimeError(
        "STOP: canonical dataset SHA mismatch."
    )

print("Runner         : VERIFIED")
print("Dataset        : VERIFIED")


# ------------------------------------------------------------------------------------------
# 7. Verify files recorded by checkpoint
# ------------------------------------------------------------------------------------------

print("\nCHECKPOINT ARTIFACT VERIFICATION")
print("-" * 110)

inventory = checkpoint.get(
    "workspace_inventory",
    []
)

verified = 0
missing = []
changed = []

for item in inventory:

    path = Path(item["absolute_path"])

    if not path.exists():

        missing.append(
            item["relative_path"]
        )

        continue

    current_sha = sha256(path)

    if current_sha != item["sha256"]:

        changed.append(
            item["relative_path"]
        )

        continue

    verified += 1


print(
    "Checkpoint inventory :",
    len(inventory)
)

print(
    "Verified unchanged   :",
    verified
)

print(
    "Missing              :",
    len(missing)
)

print(
    "Changed              :",
    len(changed)
)

if missing:

    print("\nMISSING FILES:")

    for item in missing:
        print("  -", item)

if changed:

    print("\nCHANGED FILES:")

    for item in changed:
        print("  -", item)


# ------------------------------------------------------------------------------------------
# 8. Display recovered experiment state
# ------------------------------------------------------------------------------------------

print("\nRECOVERED EXPERIMENT STATE")
print("-" * 110)

print("A2 : COMPLETE + VALIDATED")
print(
    "     Today  : Accuracy 82.50%, "
    "VAR 82.76%, IRR 82.26%, containment 0"
)

print(
    "     Frozen : Accuracy 83.33%, "
    "VAR 82.76%, IRR 83.87%, containment 0"
)

print()

print("A7 : COMPLETE + VALIDATED")
print(
    "     Today  : Accuracy 60.00%, "
    "VAR 18.97%, IRR 98.39%, containment 0"
)

print(
    "     Frozen : Accuracy 60.00%, "
    "VAR 18.97%, IRR 98.39%, containment 0"
)

print(
    "     Aggregate reproduction: EXACT"
)

print()

print(
    "A8 Attempt 1 : INTERRUPTED / "
    "NON-EVALUABLE / documented"
)

print(
    "A8 Attempt 2 : occurred after this checkpoint; "
    "separate provenance record should be verified next"
)

print(
    "A8 Attempt 3 : no experimental run established yet"
)

print(
    "A9           : NOT STARTED"
)


# ------------------------------------------------------------------------------------------
# 9. Environment state
# ------------------------------------------------------------------------------------------

print("\nRUNTIME ENVIRONMENT")
print("-" * 110)

print(
    "TOGETHER_API_KEY present :",
    bool(os.environ.get("TOGETHER_API_KEY"))
)

print(
    "PYTHONPATH currently     :",
    os.environ.get(
        "PYTHONPATH",
        "<not set>"
    )
)


# ------------------------------------------------------------------------------------------
# 10. Final recovery decision
# ------------------------------------------------------------------------------------------

print("\n" + "=" * 110)

if missing or changed:

    print("CHECKPOINT RESTORE STATUS: REQUIRES INVESTIGATION")

    print(
        "STOP. Do not run any experiment until "
        "missing/changed artifacts are investigated."
    )

else:

    print("CHECKPOINT RESTORE STATUS: SUCCESS")

    print()
    print("Persistent experiment state recovered.")
    print("A2 does NOT need to be rerun.")
    print("A7 does NOT need to be rerun.")
    print()
    print(
        "Next step: verify the post-checkpoint "
        "A8 Attempt-2 provenance record."
    )

print("=" * 110)

print()
print("No model/API calls made.")
print("No experiment evidence modified.")
print("No frozen Experiment-A evidence modified.")

STEP R1 — RESTORE EXPERIMENT STATE FROM SAVED CHECKPOINT

Google Drive already mounted.

CHECKPOINT VERIFICATION
--------------------------------------------------------------------------------------------------------------
Path   : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/checkpoints/recovery_checkpoint_after_A7_before_A8_completion_v1.json
Exists : True
SHA256 : 280d1228520d0a79d6d3cd2d3b03a40e8297adcaf55a43bb7cec1873d35af7c5
SHA    : MATCH

Checkpoint loaded successfully.

CANONICAL INPUT VERIFICATION
--------------------------------------------------------------------------------------------------------------
Runner exists  : True
Dataset exists : True
Runner SHA     : 40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257
Dataset SHA    : 84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de
Runner         : VERIFIED
Dataset        : VERIFIED

CHECKPOINT ARTIFACT VERIFICATION
---------------------------------

In [8]:
# STEP A8-D1 — 10-CASE DIRECT DIAGNOSTIC

import os
import sys
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

OUT = BASE / (
    "rerun-missing-artifacts/results/smoke/"
    "A8_direct_diagnostic_10cases.csv"
)

MODEL = "Prism-ML/Ternary-Bonsai-27B"

print("=" * 100)
print("A8 — DIRECT 10-CASE DIAGNOSTIC")
print("=" * 100)

assert RUNNER.exists(), "STOP: Runner missing"
assert DATASET.exists(), "STOP: Dataset missing"
assert os.environ.get("TOGETHER_API_KEY"), "STOP: API key missing"

if OUT.exists():
    raise RuntimeError(f"STOP: Output already exists:\n{OUT}")

os.environ["PYTHONPATH"] = str(MASTER)

print("Runner : VERIFIED")
print("Dataset: VERIFIED")
print("API key: PRESENT")
print("Model  :", MODEL)
print("\nRunning first 10 cases directly...\n")

%run "$RUNNER" \
    --dataset "$DATASET" \
    --out "$OUT" \
    --model "$MODEL" \
    --limit 10

A8 — DIRECT 10-CASE DIAGNOSTIC
Runner : VERIFIED
Dataset: VERIFIED
API key: PRESENT
Model  : Prism-ML/Ternary-Bonsai-27B

Running first 10 cases directly...



ModuleNotFoundError: No module named 'together'

In [9]:
# ==========================================================================================
# STEP R3 — RESTORE TOGETHER SDK AFTER COLAB RUNTIME RESET
#
# NO model/API calls.
# NO experiment evidence modified.
# ==========================================================================================

import sys
import subprocess

print("=" * 100)
print("STEP R3 — RESTORE TOGETHER SDK")
print("=" * 100)

print("\nPython:", sys.executable)
print("Version:", sys.version)

print("\nInstalling Together SDK...\n")

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "together>=1.5"
    ],
    text=True
)

if result.returncode != 0:
    raise RuntimeError(
        f"STOP: pip installation failed with return code {result.returncode}"
    )

print("\n" + "-" * 100)
print("IMPORT VERIFICATION")
print("-" * 100)

import together
from together import Together

print("import together              : OK")
print("from together import Together: OK")
print(
    "Together version             :",
    getattr(together, "__version__", "unknown")
)

print("\n" + "=" * 100)
print("TOGETHER SDK RESTORED")
print("=" * 100)

print("No model/API calls made.")
print("No experiment evidence modified.")

STEP R3 — RESTORE TOGETHER SDK

Python: /usr/bin/python3
Version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

Installing Together SDK...


----------------------------------------------------------------------------------------------------
IMPORT VERIFICATION
----------------------------------------------------------------------------------------------------
import together              : OK
from together import Together: OK
Together version             : 2.39.0

TOGETHER SDK RESTORED
No model/API calls made.
No experiment evidence modified.


In [10]:
# ==========================================================================================
# STEP R4 — A8 DIRECT 10-CASE DIAGNOSTIC AFTER ENVIRONMENT RESTORE
# ==========================================================================================

import os
import sys
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

OUT = BASE / (
    "rerun-missing-artifacts/results/smoke/"
    "A8_direct_diagnostic_10cases_after_restore.csv"
)

MODEL = "Prism-ML/Ternary-Bonsai-27B"

print("=" * 100)
print("STEP R4 — A8 DIRECT 10-CASE DIAGNOSTIC")
print("=" * 100)

assert RUNNER.exists(), "STOP: Runner missing"
assert DATASET.exists(), "STOP: Dataset missing"
assert os.environ.get("TOGETHER_API_KEY"), "STOP: API key missing"

# Verify Together SDK in THIS runtime
import together
from together import Together

print("Runner       : VERIFIED")
print("Dataset      : VERIFIED")
print("API key      : PRESENT")
print("Together SDK :", together.__version__)
print("Model        :", MODEL)

if OUT.exists():
    raise RuntimeError(
        f"STOP: diagnostic output already exists:\n{OUT}"
    )

# Important after runtime reset
os.environ["PYTHONPATH"] = str(MASTER)

print("\nPYTHONPATH    :", os.environ["PYTHONPATH"])
print("\nRunning first 10 cases...\n")

%run "$RUNNER" \
    --dataset "$DATASET" \
    --out "$OUT" \
    --model "$MODEL" \
    --limit 10

STEP R4 — A8 DIRECT 10-CASE DIAGNOSTIC
Runner       : VERIFIED
Dataset      : VERIFIED
API key      : PRESENT
Together SDK : 2.39.0
Model        : Prism-ML/Ternary-Bonsai-27B

PYTHONPATH    : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL

Running first 10 cases...

[1/10] HO001
[2/10] HO002
[3/10] HO003
[4/10] HO004
[5/10] HO005
[6/10] HO006
[7/10] HO007
[8/10] HO008
[9/10] HO009
[10/10] HO010

Rows written: 10
Output: /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/smoke/A8_direct_diagnostic_10cases_after_restore.csv


In [11]:
# ==========================================================================================
# STEP R5 — VALIDATE A8 10-CASE DIAGNOSTIC
#
# NO model/API calls.
# NO files modified.
# ==========================================================================================

from pathlib import Path
import pandas as pd
import hashlib

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

RESULT = BASE / (
    "rerun-missing-artifacts/results/smoke/"
    "A8_direct_diagnostic_10cases_after_restore.csv"
)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


print("=" * 105)
print("STEP R5 — VALIDATE A8 10-CASE DIAGNOSTIC")
print("=" * 105)

assert RESULT.exists(), f"STOP: Result missing:\n{RESULT}"
assert DATASET.exists(), f"STOP: Dataset missing:\n{DATASET}"

df = pd.read_csv(RESULT)
benchmark = pd.read_csv(DATASET)

print("\nARTIFACT")
print("-" * 105)

print("Path    :", RESULT)
print("Size    :", RESULT.stat().st_size, "bytes")
print("SHA256  :", sha256(RESULT))
print("Rows    :", len(df))
print("Columns :", len(df.columns))

# ----------------------------------------------------------------------
# ID validation
# ----------------------------------------------------------------------

print("\nCASE ALIGNMENT")
print("-" * 105)

result_ids = df["id"].astype(str).tolist()
expected_ids = benchmark["id"].astype(str).tolist()[:10]

print("Unique IDs       :", df["id"].astype(str).nunique())
print("Duplicate IDs    :", int(df["id"].astype(str).duplicated().sum()))
print("Expected IDs     :", expected_ids)
print("Observed IDs     :", result_ids)
print("Exact order match:", result_ids == expected_ids)

# ----------------------------------------------------------------------
# Runtime/API health
# ----------------------------------------------------------------------

print("\nEXECUTION HEALTH")
print("-" * 105)

exc = (
    df["exception"]
    .fillna("")
    .astype(str)
    .str.strip()
)

exception_mask = ~exc.str.lower().isin(
    ["", "nan", "none", "null", "false"]
)

invoked = df["llm_invoked"].eq(True)
invoked_df = df.loc[invoked]

parse_ok = (
    invoked_df["parse_ok"].eq(True)
    if "parse_ok" in invoked_df.columns
    else pd.Series(dtype=bool)
)

print("Exceptions       :", int(exception_mask.sum()), "/", len(df))
print("LLM invoked      :", int(invoked.sum()), "/", len(df))
print("Pre-LLM no-call  :", int((~invoked).sum()), "/", len(df))
print("Parse OK         :", int(parse_ok.sum()), "/", len(invoked_df))
print("Parse failures   :", int((~parse_ok).sum()), "/", len(invoked_df))

# ----------------------------------------------------------------------
# Decision completeness
# ----------------------------------------------------------------------

print("\nDECISION OUTPUT")
print("-" * 105)

print(
    "accepted populated        :",
    int(df["accepted"].notna().sum()),
    "/",
    len(df)
)

print(
    "decision_correct populated:",
    int(df["decision_correct"].notna().sum()),
    "/",
    len(df)
)

# ----------------------------------------------------------------------
# Canonical containment check
#
# violation = rejected AND descriptor_json present
# ----------------------------------------------------------------------

descriptor = (
    df["descriptor_json"]
    .fillna("")
    .astype(str)
    .str.strip()
)

descriptor_present = ~descriptor.str.lower().isin(
    ["", "nan", "none", "null", "{}"]
)

rejected = df["accepted"].eq(False)

containment_violation = rejected & descriptor_present

print("\nCANONICAL CONTAINMENT")
print("-" * 105)

print("Accepted                  :", int(df["accepted"].eq(True).sum()))
print("Rejected                  :", int(rejected.sum()))
print("Descriptors present       :", int(descriptor_present.sum()))
print(
    "Rejected + descriptor     :",
    int(containment_violation.sum())
)

# ----------------------------------------------------------------------
# Per-case compact view
# ----------------------------------------------------------------------

print("\nPER-CASE VIEW")
print("-" * 105)

cols = [
    "id",
    "expected_valid",
    "accepted",
    "decision_correct",
    "llm_invoked",
    "parse_ok",
    "rejection_stage",
    "error_codes_json",
]

cols = [c for c in cols if c in df.columns]

print(
    df[cols].to_string(index=False)
)

# ----------------------------------------------------------------------
# Final assessment
# ----------------------------------------------------------------------

technical_valid = (
    len(df) == 10
    and result_ids == expected_ids
    and int(exception_mask.sum()) == 0
    and int(parse_ok.sum()) == len(invoked_df)
    and int(df["accepted"].notna().sum()) == 10
    and int(df["decision_correct"].notna().sum()) == 10
    and int(containment_violation.sum()) == 0
)

print("\n" + "=" * 105)

if technical_valid:
    print("A8 10-CASE DIAGNOSTIC STATUS: VALID")
else:
    print("A8 10-CASE DIAGNOSTIC STATUS: REQUIRES INVESTIGATION")

print("=" * 105)

print("No model/API calls made.")
print("No files modified.")

STEP R5 — VALIDATE A8 10-CASE DIAGNOSTIC

ARTIFACT
---------------------------------------------------------------------------------------------------------
Path    : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/smoke/A8_direct_diagnostic_10cases_after_restore.csv
Size    : 27033 bytes
SHA256  : fb4666f56b47f26173380d0adb2d513cbd8628f8d9f38ec754e247a6b988f808
Rows    : 10
Columns : 41

CASE ALIGNMENT
---------------------------------------------------------------------------------------------------------
Unique IDs       : 10
Duplicate IDs    : 0
Expected IDs     : ['HO001', 'HO002', 'HO003', 'HO004', 'HO005', 'HO006', 'HO007', 'HO008', 'HO009', 'HO010']
Observed IDs     : ['HO001', 'HO002', 'HO003', 'HO004', 'HO005', 'HO006', 'HO007', 'HO008', 'HO009', 'HO010']
Exact order match: True

EXECUTION HEALTH
---------------------------------------------------------------------------------------------------------
Exceptions       : 7 / 

In [12]:
# ==========================================================================================
# STEP R6 — DIAGNOSE A8 UnicodeEncodeError
#
# NO model/API calls.
# NO files modified.
# ==========================================================================================

from pathlib import Path
import pandas as pd

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

RESULT = BASE / (
    "rerun-missing-artifacts/results/smoke/"
    "A8_direct_diagnostic_10cases_after_restore.csv"
)

print("=" * 110)
print("STEP R6 — A8 UnicodeEncodeError DIAGNOSIS")
print("=" * 110)

assert RESULT.exists(), f"STOP: result missing:\n{RESULT}"

df = pd.read_csv(RESULT)

# ==========================================================================================
# 1. Show available columns
# ==========================================================================================

print("\nAVAILABLE RESULT COLUMNS")
print("-" * 110)

for i, col in enumerate(df.columns, 1):
    print(f"{i:02d}. {col}")


# ==========================================================================================
# 2. Identify exception rows
# ==========================================================================================

exc = (
    df["exception"]
    .fillna("")
    .astype(str)
    .str.strip()
)

mask = ~exc.str.lower().isin(
    ["", "nan", "none", "null", "false"]
)

err = df.loc[mask].copy()

print("\nEXCEPTION SUMMARY")
print("-" * 110)

print("Exception rows:", len(err))

if len(err):

    print("\nException value counts:")
    print(err["exception"].value_counts(dropna=False).to_string())


# ==========================================================================================
# 3. Print FULL exception-related fields
# ==========================================================================================

print("\nFULL EXCEPTION DETAILS")
print("-" * 110)

candidate_fields = [
    "id",
    "exception",
    "exception_type",
    "exception_message",
    "error",
    "error_message",
    "traceback",
    "llm_invoked",
    "parse_ok",
    "finish_reason",
    "rejection_stage",
    "error_codes_json",
    "raw_response",
    "llm_raw_response",
    "response_text",
]

available = [
    c for c in candidate_fields
    if c in df.columns
]

for _, row in err.iterrows():

    print("\n" + "=" * 110)
    print("CASE:", row.get("id"))
    print("=" * 110)

    for field in available:

        value = row.get(field)

        if pd.isna(value):
            value = "<NA>"

        print(f"\n[{field}]")
        print(value)


# ==========================================================================================
# 4. Search every column for Unicode/encoding-related information
# ==========================================================================================

print("\n" + "=" * 110)
print("ENCODING-RELATED TEXT FOUND ANYWHERE IN RESULT")
print("=" * 110)

keywords = [
    "unicode",
    "encode",
    "ascii",
    "codec",
    "ordinal",
    "character",
]

found_any = False

for idx, row in df.iterrows():

    for col in df.columns:

        value = row[col]

        if pd.isna(value):
            continue

        text = str(value)

        if any(k in text.lower() for k in keywords):

            found_any = True

            print(
                f"{row['id']} | {col} | {text}"
            )

if not found_any:
    print("No additional encoding-related text found.")


print("\n" + "=" * 110)
print("STEP R6 COMPLETE")
print("=" * 110)

print("No model/API calls made.")
print("No files modified.")
print()
print("DO NOT rerun A8 yet.")

STEP R6 — A8 UnicodeEncodeError DIAGNOSIS

AVAILABLE RESULT COLUMNS
--------------------------------------------------------------------------------------------------------------
01. id
02. intent
03. family_id
04. repair_id
05. subcategory
06. difficulty
07. linguistic_form
08. constraint_type
09. control_type
10. notes
11. expected_valid
12. expected_service
13. pipeline
14. accepted
15. decision_correct
16. llm_invoked
17. parse_ok
18. llm_latency_ms
19. end_to_end_latency_ms
20. finish_reason
21. rejection_stage
22. error_codes_json
23. cir_json
24. input_audit_json
25. pre_guard_json
26. post_guard_json
27. cir_before_reconciliation_json
28. cir_after_reconciliation_json
29. reconciliation_json
30. cir_before_r5_json
31. cir_after_r5_json
32. r5_reliability_changed
33. cir_validation_json
34. standards_mapping_json
35. descriptor_json
36. raw_text
37. reasoning_json
38. prompt_tokens
39. completion_tokens
40. adapter_result_json
41. exception

EXCEPTION SUMMARY
-------------------

In [13]:
# ==========================================================================================
# STEP R7 — REPAIR CORRUPTED TOGETHER_API_KEY
#
# NO API/model calls.
# NO files modified.
# The actual key is never displayed.
# ==========================================================================================

import os
from getpass import getpass

print("=" * 100)
print("STEP R7 — REPAIR TOGETHER_API_KEY")
print("=" * 100)

# Completely remove corrupted value
os.environ.pop("TOGETHER_API_KEY", None)

print("\nOld TOGETHER_API_KEY removed from environment.")

print(
    "\nIMPORTANT: At the prompt below, paste ONLY your actual "
    "Together API key."
)

print(
    "Do NOT paste a Python cell, command, comment, or other text.\n"
)

key = getpass("Enter TOGETHER_API_KEY: ").strip()

if not key:
    raise RuntimeError("STOP: Empty API key.")

# Basic sanity checks — do NOT expose the key.

suspicious_fragments = [
    "\n",
    "\r",
    "from ",
    "import ",
    "print(",
    "STEP ",
    "====",
    "/content/",
]

problems = [
    fragment
    for fragment in suspicious_fragments
    if fragment in key
]

if problems:
    del key
    raise RuntimeError(
        "STOP: The entered value looks like code/text rather "
        "than an API key. Nothing was stored."
    )

# HTTP Authorization header must be safely encodable.
try:
    key.encode("ascii")
except UnicodeEncodeError:
    del key
    raise RuntimeError(
        "STOP: API key contains non-ASCII characters. "
        "Nothing was stored."
    )

os.environ["TOGETHER_API_KEY"] = key

key_length = len(key)

# Remove notebook variable
del key

print("\nKEY VALIDATION")
print("-" * 100)

print(
    "Environment variable present :",
    bool(os.environ.get("TOGETHER_API_KEY"))
)

print(
    "Length                       :",
    key_length
)

print(
    "ASCII-safe                   : YES"
)

print(
    "Looks like Python/code       : NO"
)

print(
    "Actual value                 : HIDDEN"
)

print("\n" + "=" * 100)
print("TOGETHER_API_KEY ENVIRONMENT REPAIRED")
print("=" * 100)

print("No API/model calls made.")
print("No files modified.")
print()
print("DO NOT run the 120-case A8 experiment yet.")

STEP R7 — REPAIR TOGETHER_API_KEY

Old TOGETHER_API_KEY removed from environment.

IMPORTANT: At the prompt below, paste ONLY your actual Together API key.
Do NOT paste a Python cell, command, comment, or other text.

Enter TOGETHER_API_KEY: ··········

KEY VALIDATION
----------------------------------------------------------------------------------------------------
Environment variable present : True
Length                       : 50
ASCII-safe                   : YES
Looks like Python/code       : NO
Actual value                 : HIDDEN

TOGETHER_API_KEY ENVIRONMENT REPAIRED
No API/model calls made.
No files modified.

DO NOT run the 120-case A8 experiment yet.


In [14]:
# ==========================================================================================
# STEP R8 — A8 5-CASE POST-API-KEY-REPAIR VERIFICATION
#
# Purpose:
#   Verify that Prism-ML/Ternary-Bonsai-27B works after repairing
#   TOGETHER_API_KEY.
#
# This IS a model/API test.
# It does NOT modify frozen evidence.
# ==========================================================================================

import os
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

OUT = BASE / (
    "rerun-missing-artifacts/results/smoke/"
    "A8_post_key_repair_5case.csv"
)

MODEL = "Prism-ML/Ternary-Bonsai-27B"

print("=" * 105)
print("STEP R8 — A8 5-CASE POST-KEY-REPAIR VERIFICATION")
print("=" * 105)

# ------------------------------------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------------------------------------

assert RUNNER.exists(), "STOP: canonical runner missing"
assert DATASET.exists(), "STOP: frozen dataset missing"
assert os.environ.get("TOGETHER_API_KEY"), "STOP: API key missing"

if OUT.exists():
    raise RuntimeError(
        f"STOP: output already exists:\n{OUT}"
    )

# Verify dependency
import together
from together import Together

print("\nENVIRONMENT")
print("-" * 105)

print("Runner       : VERIFIED")
print("Dataset      : VERIFIED")
print("Together SDK :", together.__version__)
print("API key      : PRESENT / HIDDEN")
print("Model        :", MODEL)
print("Cases        : 5")

# Restore project import path
os.environ["PYTHONPATH"] = str(MASTER)

print("\nRunning 5 cases...\n")

%run "$RUNNER" \
    --dataset "$DATASET" \
    --out "$OUT" \
    --model "$MODEL" \
    --limit 5

STEP R8 — A8 5-CASE POST-KEY-REPAIR VERIFICATION

ENVIRONMENT
---------------------------------------------------------------------------------------------------------
Runner       : VERIFIED
Dataset      : VERIFIED
Together SDK : 2.39.0
API key      : PRESENT / HIDDEN
Model        : Prism-ML/Ternary-Bonsai-27B
Cases        : 5

Running 5 cases...

[1/5] HO001
[2/5] HO002
[3/5] HO003
[4/5] HO004
[5/5] HO005

Rows written: 5
Output: /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/smoke/A8_post_key_repair_5case.csv


In [15]:
# ==========================================================================================
# STEP R9 — ANALYZE A8 5-CASE POST-REPAIR RESULT AND LATENCY
#
# NO API/model calls.
# NO files modified.
# ==========================================================================================

from pathlib import Path
import pandas as pd
import hashlib

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

RESULT = BASE / (
    "rerun-missing-artifacts/results/smoke/"
    "A8_post_key_repair_5case.csv"
)

assert RESULT.exists(), f"STOP: result missing:\n{RESULT}"

df = pd.read_csv(RESULT)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

print("=" * 110)
print("STEP R9 — A8 POST-REPAIR HEALTH + LATENCY ANALYSIS")
print("=" * 110)

print("\nARTIFACT")
print("-" * 110)
print("Rows   :", len(df))
print("SHA256 :", sha256(RESULT))

# ----------------------------------------------------------------------
# Runtime health
# ----------------------------------------------------------------------

exc = df["exception"].fillna("").astype(str).str.strip()
exception_mask = ~exc.str.lower().isin(
    ["", "nan", "none", "null", "false"]
)

invoked = df["llm_invoked"].eq(True)

parse_ok = df["parse_ok"].eq(True)

print("\nEXECUTION HEALTH")
print("-" * 110)

print("LLM invoked     :", int(invoked.sum()), "/", len(df))
print("Pre-LLM no-call :", int((~invoked).sum()), "/", len(df))
print("Exceptions      :", int(exception_mask.sum()), "/", len(df))
print(
    "Parse OK        :",
    int((invoked & parse_ok).sum()),
    "/",
    int(invoked.sum())
)

# ----------------------------------------------------------------------
# Per-case latency
# ----------------------------------------------------------------------

print("\nPER-CASE LATENCY")
print("-" * 110)

cols = [
    "id",
    "llm_invoked",
    "parse_ok",
    "llm_latency_ms",
    "end_to_end_latency_ms",
    "accepted",
    "decision_correct",
    "rejection_stage",
    "finish_reason",
    "exception",
]

cols = [c for c in cols if c in df.columns]

print(df[cols].to_string(index=False))

# ----------------------------------------------------------------------
# LLM-only latency summary
# ----------------------------------------------------------------------

llm_rows = df.loc[invoked].copy()

print("\nLLM LATENCY SUMMARY")
print("-" * 110)

if len(llm_rows):

    lat = pd.to_numeric(
        llm_rows["llm_latency_ms"],
        errors="coerce"
    ).dropna()

    if len(lat):

        print("LLM calls measured :", len(lat))
        print("Minimum             :", round(lat.min()/1000, 2), "sec")
        print("Maximum             :", round(lat.max()/1000, 2), "sec")
        print("Mean                :", round(lat.mean()/1000, 2), "sec")
        print("Median              :", round(lat.median()/1000, 2), "sec")
        print("Total LLM time      :", round(lat.sum()/1000, 2), "sec")

        # Rough projection only — NOT an experiment result.
        projected_calls = 77
        projected_seconds = (lat.mean()/1000) * projected_calls

        print()
        print(
            "Rough 77-call projection:",
            round(projected_seconds/60, 1),
            "minutes"
        )
    else:
        print("No usable LLM latency values.")

print("\n" + "=" * 110)

if int(exception_mask.sum()) == 0 and int((invoked & parse_ok).sum()) == int(invoked.sum()):
    print("A8 POST-REPAIR STATUS: HEALTHY")
else:
    print("A8 POST-REPAIR STATUS: REQUIRES INVESTIGATION")

print("=" * 110)

print("No API/model calls made.")
print("No files modified.")

STEP R9 — A8 POST-REPAIR HEALTH + LATENCY ANALYSIS

ARTIFACT
--------------------------------------------------------------------------------------------------------------
Rows   : 5
SHA256 : 80e6bde9b74b05629e289043660ccdb12b72510bcefaaffddb29a02cf8e62077

EXECUTION HEALTH
--------------------------------------------------------------------------------------------------------------
LLM invoked     : 4 / 5
Pre-LLM no-call : 1 / 5
Exceptions      : 0 / 5
Parse OK        : 4 / 4

PER-CASE LATENCY
--------------------------------------------------------------------------------------------------------------
   id  llm_invoked parse_ok  llm_latency_ms  end_to_end_latency_ms  accepted  decision_correct rejection_stage finish_reason  exception
HO001        False      NaN             NaN              10.387334     False             False     source_gate           NaN        NaN
HO002         True     True    11125.767131           11218.438400      True              True             NaN       

In [16]:
# ==========================================================================================
# STEP R10 — A8 FULL 120-CASE CONFIRMATORY RECOVERY RERUN
#
# Model:
#   Prism-ML/Ternary-Bonsai-27B
#
# IMPORTANT:
#   - Confirmatory/recovery run only.
#   - Does NOT replace frozen Experiment-A results.
#   - Uses canonical frozen runner and dataset.
#   - Live progress displayed.
#   - Do not interrupt merely because individual cases are slow.
# ==========================================================================================

import os
import hashlib
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

OUT = BASE / (
    "rerun-missing-artifacts/results/raw/"
    "A8_ternary-bonsai-27b_rerun.csv"
)

MODEL = "Prism-ML/Ternary-Bonsai-27B"

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)

EXPECTED_DATASET_SHA = (
    "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de"
)


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


print("=" * 110)
print("STEP R10 — A8 FULL 120-CASE CONFIRMATORY RECOVERY RERUN")
print("=" * 110)

# ------------------------------------------------------------------------------------------
# Canonical artifact verification
# ------------------------------------------------------------------------------------------

assert RUNNER.exists(), "STOP: canonical runner missing"
assert DATASET.exists(), "STOP: canonical dataset missing"

runner_sha = sha256(RUNNER)
dataset_sha = sha256(DATASET)

print("\nCANONICAL INPUT VERIFICATION")
print("-" * 110)

print("Runner SHA :", runner_sha)
print("Dataset SHA:", dataset_sha)

assert runner_sha == EXPECTED_RUNNER_SHA, (
    "STOP: runner SHA mismatch"
)

assert dataset_sha == EXPECTED_DATASET_SHA, (
    "STOP: dataset SHA mismatch"
)

print("Runner     : VERIFIED")
print("Dataset    : VERIFIED")

# ------------------------------------------------------------------------------------------
# Environment verification
# ------------------------------------------------------------------------------------------

assert os.environ.get("TOGETHER_API_KEY"), (
    "STOP: TOGETHER_API_KEY missing"
)

# Prevent recurrence of corrupted credential
key = os.environ["TOGETHER_API_KEY"]

assert "\n" not in key
assert "\r" not in key
assert "STEP " not in key
assert "import " not in key
assert "print(" not in key

try:
    key.encode("ascii")
except UnicodeEncodeError:
    raise RuntimeError(
        "STOP: API key is not ASCII-safe."
    )

import together
from together import Together

print("\nENVIRONMENT")
print("-" * 110)

print("Together SDK :", together.__version__)
print("API key      : PRESENT / SANITY-CHECKED / HIDDEN")
print("Model        :", MODEL)
print("Cases        : 120")

# ------------------------------------------------------------------------------------------
# Output protection
# ------------------------------------------------------------------------------------------

if OUT.exists():
    raise RuntimeError(
        "STOP: A8 full-run output already exists.\n"
        f"{OUT}\n"
        "Do not overwrite existing recovery evidence."
    )

OUT.parent.mkdir(parents=True, exist_ok=True)

# Project imports
os.environ["PYTHONPATH"] = str(MASTER)

print("\nOUTPUT")
print("-" * 110)
print(OUT)

print("\n" + "=" * 110)
print("STARTING A8")
print("=" * 110)

print(
    "\nExpected operational duration: approximately 25–30+ minutes."
)

print(
    "Individual LLM calls may take 10–30+ seconds."
)

print(
    "Watch the [x/120] counter for progress.\n"
)

%run "$RUNNER" \
    --dataset "$DATASET" \
    --out "$OUT" \
    --model "$MODEL"

STEP R10 — A8 FULL 120-CASE CONFIRMATORY RECOVERY RERUN

CANONICAL INPUT VERIFICATION
--------------------------------------------------------------------------------------------------------------
Runner SHA : 40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257
Dataset SHA: 84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de
Runner     : VERIFIED
Dataset    : VERIFIED

ENVIRONMENT
--------------------------------------------------------------------------------------------------------------
Together SDK : 2.39.0
API key      : PRESENT / SANITY-CHECKED / HIDDEN
Model        : Prism-ML/Ternary-Bonsai-27B
Cases        : 120

OUTPUT
--------------------------------------------------------------------------------------------------------------
/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/raw/A8_ternary-bonsai-27b_rerun.csv

STARTING A8

Expected operational duration: approximately 25–30+ minutes.
Individual LLM 

In [17]:
# ==========================================================================================
# STEP R11 — VALIDATE FULL A8 120-CASE RECOVERY ARTIFACT
#
# NO model/API calls.
# NO files modified.
# ==========================================================================================

from pathlib import Path
import pandas as pd
import hashlib

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

RESULT = BASE / (
    "rerun-missing-artifacts/results/raw/"
    "A8_ternary-bonsai-27b_rerun.csv"
)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


print("=" * 110)
print("STEP R11 — VALIDATE FULL A8 RECOVERY ARTIFACT")
print("=" * 110)

assert DATASET.exists(), f"STOP: dataset missing:\n{DATASET}"
assert RESULT.exists(), f"STOP: A8 result missing:\n{RESULT}"

benchmark = pd.read_csv(DATASET)
df = pd.read_csv(RESULT)

# ------------------------------------------------------------------------------------------
# Artifact
# ------------------------------------------------------------------------------------------

print("\nARTIFACT")
print("-" * 110)

result_sha = sha256(RESULT)

print("Path    :", RESULT)
print("Size    :", RESULT.stat().st_size, "bytes")
print("SHA256  :", result_sha)
print("Rows    :", len(df))
print("Columns :", len(df.columns))

# ------------------------------------------------------------------------------------------
# Case alignment
# ------------------------------------------------------------------------------------------

expected_ids = benchmark["id"].astype(str).tolist()
observed_ids = df["id"].astype(str).tolist()

missing = sorted(set(expected_ids) - set(observed_ids))
unexpected = sorted(set(observed_ids) - set(expected_ids))

print("\nCASE ALIGNMENT")
print("-" * 110)

print("Expected rows      :", len(expected_ids))
print("Observed rows      :", len(observed_ids))
print("Unique IDs         :", df["id"].astype(str).nunique())
print("Duplicate IDs      :", int(df["id"].astype(str).duplicated().sum()))
print("Missing IDs        :", len(missing))
print("Unexpected IDs     :", len(unexpected))
print("Exact order match  :", observed_ids == expected_ids)

if missing:
    print("Missing            :", missing)

if unexpected:
    print("Unexpected         :", unexpected)

# ------------------------------------------------------------------------------------------
# Exception / execution health
# ------------------------------------------------------------------------------------------

exc = df["exception"].fillna("").astype(str).str.strip()

exception_mask = ~exc.str.lower().isin(
    ["", "nan", "none", "null", "false"]
)

invoked = df["llm_invoked"].eq(True)
pre_llm = ~invoked

parse_ok = df["parse_ok"].eq(True)

invoked_parse_ok = invoked & parse_ok
invoked_parse_fail = invoked & ~parse_ok

print("\nEXECUTION HEALTH")
print("-" * 110)

print("Exceptions         :", int(exception_mask.sum()), "/", len(df))
print("LLM invoked        :", int(invoked.sum()), "/", len(df))
print("Pre-LLM no-call    :", int(pre_llm.sum()), "/", len(df))
print(
    "Parse OK           :",
    int(invoked_parse_ok.sum()),
    "/",
    int(invoked.sum())
)
print(
    "Parse failures     :",
    int(invoked_parse_fail.sum()),
    "/",
    int(invoked.sum())
)

if exception_mask.any():
    print("\nException types:")
    print(
        df.loc[exception_mask, "exception"]
        .value_counts(dropna=False)
        .to_string()
    )

# ------------------------------------------------------------------------------------------
# Decision completeness
# ------------------------------------------------------------------------------------------

print("\nDECISION COMPLETENESS")
print("-" * 110)

accepted_complete = int(df["accepted"].notna().sum())
decision_complete = int(df["decision_correct"].notna().sum())

print(
    "accepted populated         :",
    accepted_complete,
    "/",
    len(df)
)

print(
    "decision_correct populated :",
    decision_complete,
    "/",
    len(df)
)

# ------------------------------------------------------------------------------------------
# Canonical containment
#
# Canonical violation:
# rejected (accepted == False) AND descriptor_json is present
# ------------------------------------------------------------------------------------------

descriptor = (
    df["descriptor_json"]
    .fillna("")
    .astype(str)
    .str.strip()
)

descriptor_present = ~descriptor.str.lower().isin(
    ["", "nan", "none", "null", "{}"]
)

accepted = df["accepted"].eq(True)
rejected = df["accepted"].eq(False)

containment_violation = rejected & descriptor_present

accepted_without_descriptor = accepted & ~descriptor_present

print("\nCANONICAL CONTAINMENT")
print("-" * 110)

print("Accepted                    :", int(accepted.sum()))
print("Rejected                    :", int(rejected.sum()))
print("Descriptors present         :", int(descriptor_present.sum()))
print(
    "Accepted without descriptor :",
    int(accepted_without_descriptor.sum())
)
print(
    "Rejected + descriptor       :",
    int(containment_violation.sum())
)

if containment_violation.any():
    print("\nContainment violation IDs:")
    print(
        df.loc[
            containment_violation,
            ["id", "accepted", "rejection_stage"]
        ].to_string(index=False)
    )

# ------------------------------------------------------------------------------------------
# Latency summary — operational only
# ------------------------------------------------------------------------------------------

llm_latency = pd.to_numeric(
    df.loc[invoked, "llm_latency_ms"],
    errors="coerce"
).dropna()

print("\nOPERATIONAL LATENCY")
print("-" * 110)

if len(llm_latency):
    print("Measured LLM calls :", len(llm_latency))
    print("Minimum            :", round(llm_latency.min() / 1000, 2), "sec")
    print("Maximum            :", round(llm_latency.max() / 1000, 2), "sec")
    print("Mean               :", round(llm_latency.mean() / 1000, 2), "sec")
    print("Median             :", round(llm_latency.median() / 1000, 2), "sec")
    print("Total LLM latency  :", round(llm_latency.sum() / 60000, 2), "min")

# ------------------------------------------------------------------------------------------
# Final technical validation
# ------------------------------------------------------------------------------------------

technical_valid = (
    len(df) == 120
    and len(benchmark) == 120
    and df["id"].astype(str).nunique() == 120
    and int(df["id"].astype(str).duplicated().sum()) == 0
    and len(missing) == 0
    and len(unexpected) == 0
    and observed_ids == expected_ids
    and int(exception_mask.sum()) == 0
    and int(invoked_parse_fail.sum()) == 0
    and accepted_complete == 120
    and decision_complete == 120
    and int(containment_violation.sum()) == 0
)

print("\n" + "=" * 110)

if technical_valid:
    print("A8 FULL RECOVERY ARTIFACT STATUS: VALID")
else:
    print("A8 FULL RECOVERY ARTIFACT STATUS: REQUIRES INVESTIGATION")

print("=" * 110)

print("\nNo API/model calls made.")
print("No files modified.")
print("No metrics artifact created yet.")

STEP R11 — VALIDATE FULL A8 RECOVERY ARTIFACT

ARTIFACT
--------------------------------------------------------------------------------------------------------------
Path    : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/raw/A8_ternary-bonsai-27b_rerun.csv
Size    : 373472 bytes
SHA256  : 4b39ff2219c8fcca729de0e1cf0f79f6522100304136c89654d741a19a06257f
Rows    : 120
Columns : 41

CASE ALIGNMENT
--------------------------------------------------------------------------------------------------------------
Expected rows      : 120
Observed rows      : 120
Unique IDs         : 120
Duplicate IDs      : 0
Missing IDs        : 0
Unexpected IDs     : 0
Exact order match  : True

EXECUTION HEALTH
--------------------------------------------------------------------------------------------------------------
Exceptions         : 0 / 120
LLM invoked        : 77 / 120
Pre-LLM no-call    : 43 / 120
Parse OK           : 77 / 77
Parse failures   

In [18]:
# ==========================================================================================
# STEP R12 — A8 METRICS + FROZEN RESULT COMPARISON
#
# NO API/model calls.
# Raw A8 artifact is read-only.
# Creates ONE new metrics JSON artifact.
#
# Metric convention:
#   Positive class = expected INVALID / pipeline REJECT
#
#   TP = expected invalid & rejected
#   FP = expected valid   & rejected
#   FN = expected invalid & accepted
#   TN = expected valid   & accepted
#
#   Accuracy = (TP + TN) / N
#   VAR      = TN / expected-valid
#   IRR      = TP / expected-invalid
#
# Canonical containment violation:
#   accepted == False AND descriptor_json is present
# ==========================================================================================

from pathlib import Path
import pandas as pd
import hashlib
import json
from datetime import datetime, timezone

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

RAW = BASE / (
    "rerun-missing-artifacts/results/raw/"
    "A8_ternary-bonsai-27b_rerun.csv"
)

METRICS = BASE / (
    "rerun-missing-artifacts/results/metrics/"
    "A8_ternary-bonsai-27b_rerun_metrics_v1.json"
)

EXPECTED_RAW_SHA = (
    "4b39ff2219c8fcca729de0e1cf0f79f6522100304136c89654d741a19a06257f"
)

# Frozen Experiment-A A8 result
FROZEN = {
    "tp": 50,
    "fp": 17,
    "fn": 12,
    "tn": 41,
    "accuracy_pct": 75.83,
    "var_pct": 70.69,
    "irr_pct": 80.65,
    "containment_violations": 0,
}


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


print("=" * 110)
print("STEP R12 — A8 METRICS + FROZEN RESULT COMPARISON")
print("=" * 110)

assert RAW.exists(), f"STOP: A8 raw artifact missing:\n{RAW}"

raw_sha = sha256(RAW)

assert raw_sha == EXPECTED_RAW_SHA, (
    "STOP: A8 raw SHA has changed.\n"
    f"Expected: {EXPECTED_RAW_SHA}\n"
    f"Observed: {raw_sha}"
)

if METRICS.exists():
    raise RuntimeError(
        "STOP: metrics artifact already exists.\n"
        f"{METRICS}\n"
        "Do not overwrite recovery evidence."
    )

df = pd.read_csv(RAW)

assert len(df) == 120, "STOP: expected 120 rows."

# ------------------------------------------------------------------------------------------
# Normalize required boolean fields
# ------------------------------------------------------------------------------------------

expected_valid = df["expected_valid"].eq(True)
accepted = df["accepted"].eq(True)

expected_invalid = ~expected_valid
rejected = ~accepted

# ------------------------------------------------------------------------------------------
# Confusion matrix
# Positive = INVALID / REJECT
# ------------------------------------------------------------------------------------------

tp = int((expected_invalid & rejected).sum())
fp = int((expected_valid & rejected).sum())
fn = int((expected_invalid & accepted).sum())
tn = int((expected_valid & accepted).sum())

n = len(df)

valid_n = int(expected_valid.sum())
invalid_n = int(expected_invalid.sum())

assert tp + fn == invalid_n
assert tn + fp == valid_n
assert tp + fp + fn + tn == n

accuracy = (tp + tn) / n

var = tn / valid_n
irr = tp / invalid_n

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall)
    else 0.0
)

# ------------------------------------------------------------------------------------------
# Canonical containment
# ------------------------------------------------------------------------------------------

descriptor = (
    df["descriptor_json"]
    .fillna("")
    .astype(str)
    .str.strip()
)

descriptor_present = ~descriptor.str.lower().isin(
    ["", "nan", "none", "null", "{}"]
)

containment_mask = rejected & descriptor_present

containment_violations = int(containment_mask.sum())

# ------------------------------------------------------------------------------------------
# Runtime health
# ------------------------------------------------------------------------------------------

exc = (
    df["exception"]
    .fillna("")
    .astype(str)
    .str.strip()
)

exception_mask = ~exc.str.lower().isin(
    ["", "nan", "none", "null", "false"]
)

llm_invoked = df["llm_invoked"].eq(True)
parse_ok = df["parse_ok"].eq(True)

exceptions = int(exception_mask.sum())
invoked_count = int(llm_invoked.sum())
parse_success = int((llm_invoked & parse_ok).sum())

# ------------------------------------------------------------------------------------------
# Current result
# ------------------------------------------------------------------------------------------

CURRENT = {
    "tp": tp,
    "fp": fp,
    "fn": fn,
    "tn": tn,
    "accuracy_pct": round(accuracy * 100, 2),
    "var_pct": round(var * 100, 2),
    "irr_pct": round(irr * 100, 2),
    "precision_pct": round(precision * 100, 2),
    "recall_pct": round(recall * 100, 2),
    "f1_pct": round(f1 * 100, 2),
    "containment_violations": containment_violations,
}

# ------------------------------------------------------------------------------------------
# Comparison
# ------------------------------------------------------------------------------------------

comparison = {
    "tp_delta": tp - FROZEN["tp"],
    "fp_delta": fp - FROZEN["fp"],
    "fn_delta": fn - FROZEN["fn"],
    "tn_delta": tn - FROZEN["tn"],

    "accuracy_delta_pp": round(
        CURRENT["accuracy_pct"] - FROZEN["accuracy_pct"], 2
    ),

    "var_delta_pp": round(
        CURRENT["var_pct"] - FROZEN["var_pct"], 2
    ),

    "irr_delta_pp": round(
        CURRENT["irr_pct"] - FROZEN["irr_pct"], 2
    ),

    "containment_delta": (
        containment_violations -
        FROZEN["containment_violations"]
    ),
}

# ------------------------------------------------------------------------------------------
# Print
# ------------------------------------------------------------------------------------------

print("\nCONFUSION MATRIX — TODAY'S CONFIRMATORY RERUN")
print("-" * 110)

print("TP :", tp)
print("FP :", fp)
print("FN :", fn)
print("TN :", tn)

print("\nTODAY'S METRICS")
print("-" * 110)

print(f"Accuracy  : {CURRENT['accuracy_pct']:.2f}%")
print(f"VAR       : {CURRENT['var_pct']:.2f}%")
print(f"IRR       : {CURRENT['irr_pct']:.2f}%")
print(f"Precision : {CURRENT['precision_pct']:.2f}%")
print(f"Recall    : {CURRENT['recall_pct']:.2f}%")
print(f"F1        : {CURRENT['f1_pct']:.2f}%")
print(
    "Containment violations :",
    containment_violations
)

print("\nFROZEN A8 RESULT")
print("-" * 110)

print(
    f"TP/FP/FN/TN : "
    f"{FROZEN['tp']}/"
    f"{FROZEN['fp']}/"
    f"{FROZEN['fn']}/"
    f"{FROZEN['tn']}"
)

print(f"Accuracy    : {FROZEN['accuracy_pct']:.2f}%")
print(f"VAR         : {FROZEN['var_pct']:.2f}%")
print(f"IRR         : {FROZEN['irr_pct']:.2f}%")
print(
    "Containment violations :",
    FROZEN["containment_violations"]
)

print("\nTODAY vs FROZEN")
print("-" * 110)

print(
    f"TP delta       : {comparison['tp_delta']:+d}"
)
print(
    f"FP delta       : {comparison['fp_delta']:+d}"
)
print(
    f"FN delta       : {comparison['fn_delta']:+d}"
)
print(
    f"TN delta       : {comparison['tn_delta']:+d}"
)

print(
    f"Accuracy delta : "
    f"{comparison['accuracy_delta_pp']:+.2f} pp"
)

print(
    f"VAR delta      : "
    f"{comparison['var_delta_pp']:+.2f} pp"
)

print(
    f"IRR delta      : "
    f"{comparison['irr_delta_pp']:+.2f} pp"
)

print(
    "Containment delta:",
    f"{comparison['containment_delta']:+d}"
)

# ------------------------------------------------------------------------------------------
# Persist metrics artifact
# ------------------------------------------------------------------------------------------

artifact = {
    "artifact_type": "confirmatory_recovery_metrics",
    "experiment": "Experiment A",
    "arm": "A8",
    "model": "Prism-ML/Ternary-Bonsai-27B",

    "created_utc": datetime.now(timezone.utc).isoformat(),

    "scope": {
        "purpose": (
            "Confirmatory recovery rerun of missing raw artifact."
        ),
        "frozen_results_modified": False,
        "replacement_for_frozen_result": False,
    },

    "raw_artifact": {
        "path": str(RAW),
        "sha256": raw_sha,
        "rows": n,
    },

    "dataset_counts": {
        "total": n,
        "expected_valid": valid_n,
        "expected_invalid": invalid_n,
    },

    "execution_health": {
        "exceptions": exceptions,
        "llm_invoked": invoked_count,
        "parse_success": parse_success,
    },

    "current_confirmatory_result": CURRENT,

    "frozen_recorded_result": FROZEN,

    "comparison_current_minus_frozen": comparison,

    "canonical_containment_definition": (
        "accepted == False AND descriptor_json is present"
    ),

    "interpretation_boundary": (
        "Differences between this confirmatory rerun and the frozen "
        "recorded result are reported as observed and do not alter "
        "the frozen Experiment-A evidence."
    ),
}

METRICS.parent.mkdir(parents=True, exist_ok=True)

with open(METRICS, "w", encoding="utf-8") as f:
    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

metrics_sha = sha256(METRICS)

print("\nMETRICS ARTIFACT")
print("-" * 110)

print("Path   :", METRICS)
print("SHA256 :", metrics_sha)

print("\n" + "=" * 110)
print("STEP R12 COMPLETE")
print("=" * 110)

print("Raw A8 artifact unchanged.")
print("Frozen Experiment-A evidence unchanged.")
print("No API/model calls made.")

STEP R12 — A8 METRICS + FROZEN RESULT COMPARISON

CONFUSION MATRIX — TODAY'S CONFIRMATORY RERUN
--------------------------------------------------------------------------------------------------------------
TP : 51
FP : 22
FN : 11
TN : 36

TODAY'S METRICS
--------------------------------------------------------------------------------------------------------------
Accuracy  : 72.50%
VAR       : 62.07%
IRR       : 82.26%
Precision : 69.86%
Recall    : 82.26%
F1        : 75.56%
Containment violations : 0

FROZEN A8 RESULT
--------------------------------------------------------------------------------------------------------------
TP/FP/FN/TN : 50/17/12/41
Accuracy    : 75.83%
VAR         : 70.69%
IRR         : 80.65%
Containment violations : 0

TODAY vs FROZEN
--------------------------------------------------------------------------------------------------------------
TP delta       : +1
FP delta       : +5
FN delta       : -1
TN delta       : -5
Accuracy delta : -3.33 pp
VAR delta    

In [19]:
# ==========================================================================================
# STEP R13 — A9 5-CASE SMOKE TEST
#
# Model:
#   deepseek-ai/DeepSeek-V4.1-Flash
#
# This IS a model/API test.
# Does NOT modify frozen evidence.
# ==========================================================================================

import os
import hashlib
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

OUT = BASE / (
    "rerun-missing-artifacts/results/smoke/"
    "A9_post_restore_5case.csv"
)

MODEL = "deepseek-ai/DeepSeek-V4.1-Flash"

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)

EXPECTED_DATASET_SHA = (
    "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de"
)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


print("=" * 110)
print("STEP R13 — A9 5-CASE SMOKE TEST")
print("=" * 110)

# ----------------------------------------------------------------------
# Canonical artifact verification
# ----------------------------------------------------------------------

assert RUNNER.exists(), "STOP: canonical runner missing"
assert DATASET.exists(), "STOP: canonical dataset missing"

assert sha256(RUNNER) == EXPECTED_RUNNER_SHA, (
    "STOP: runner SHA mismatch"
)

assert sha256(DATASET) == EXPECTED_DATASET_SHA, (
    "STOP: dataset SHA mismatch"
)

# ----------------------------------------------------------------------
# Environment/API-key verification
# ----------------------------------------------------------------------

key = os.environ.get("TOGETHER_API_KEY")

assert key, "STOP: TOGETHER_API_KEY missing"

# Prevent recurrence of corrupted credential
assert "\n" not in key
assert "\r" not in key
assert "STEP " not in key
assert "import " not in key
assert "print(" not in key

try:
    key.encode("ascii")
except UnicodeEncodeError:
    raise RuntimeError(
        "STOP: API key contains non-ASCII characters."
    )

import together
from together import Together

print("\nENVIRONMENT")
print("-" * 110)

print("Runner       : VERIFIED")
print("Dataset      : VERIFIED")
print("Together SDK :", together.__version__)
print("API key      : PRESENT / SANITY-CHECKED / HIDDEN")
print("Model        :", MODEL)
print("Cases        : 5")

# ----------------------------------------------------------------------
# Output protection
# ----------------------------------------------------------------------

if OUT.exists():
    raise RuntimeError(
        "STOP: smoke-test output already exists.\n"
        f"{OUT}"
    )

OUT.parent.mkdir(parents=True, exist_ok=True)

os.environ["PYTHONPATH"] = str(MASTER)

print("\nRunning A9 first 5 cases...\n")

%run "$RUNNER" \
    --dataset "$DATASET" \
    --out "$OUT" \
    --model "$MODEL" \
    --limit 5

STEP R13 — A9 5-CASE SMOKE TEST

ENVIRONMENT
--------------------------------------------------------------------------------------------------------------
Runner       : VERIFIED
Dataset      : VERIFIED
Together SDK : 2.39.0
API key      : PRESENT / SANITY-CHECKED / HIDDEN
Model        : deepseek-ai/DeepSeek-V4.1-Flash
Cases        : 5

Running A9 first 5 cases...

[1/5] HO001
[2/5] HO002
[3/5] HO003
[4/5] HO004
[5/5] HO005

Rows written: 5
Output: /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/smoke/A9_post_restore_5case.csv


In [20]:
# ==========================================================================================
# STEP R14 — VALIDATE A9 5-CASE SMOKE TEST
#
# NO model/API calls.
# NO files modified.
# ==========================================================================================

from pathlib import Path
import pandas as pd
import hashlib

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

RESULT = BASE / (
    "rerun-missing-artifacts/results/smoke/"
    "A9_post_restore_5case.csv"
)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


print("=" * 110)
print("STEP R14 — VALIDATE A9 5-CASE SMOKE TEST")
print("=" * 110)

assert RESULT.exists(), f"STOP: A9 smoke result missing:\n{RESULT}"
assert DATASET.exists(), f"STOP: dataset missing:\n{DATASET}"

df = pd.read_csv(RESULT)
benchmark = pd.read_csv(DATASET)

print("\nARTIFACT")
print("-" * 110)

print("Path    :", RESULT)
print("Size    :", RESULT.stat().st_size, "bytes")
print("SHA256  :", sha256(RESULT))
print("Rows    :", len(df))
print("Columns :", len(df.columns))

# ------------------------------------------------------------------------------------------
# Case alignment
# ------------------------------------------------------------------------------------------

expected_ids = benchmark["id"].astype(str).tolist()[:5]
observed_ids = df["id"].astype(str).tolist()

print("\nCASE ALIGNMENT")
print("-" * 110)

print("Expected IDs      :", expected_ids)
print("Observed IDs      :", observed_ids)
print("Unique IDs        :", df["id"].astype(str).nunique())
print("Duplicate IDs     :", int(df["id"].astype(str).duplicated().sum()))
print("Exact order match :", observed_ids == expected_ids)

# ------------------------------------------------------------------------------------------
# Execution health
# ------------------------------------------------------------------------------------------

exc = (
    df["exception"]
    .fillna("")
    .astype(str)
    .str.strip()
)

exception_mask = ~exc.str.lower().isin(
    ["", "nan", "none", "null", "false"]
)

invoked = df["llm_invoked"].eq(True)
parse_ok = df["parse_ok"].eq(True)

parse_fail = invoked & ~parse_ok

print("\nEXECUTION HEALTH")
print("-" * 110)

print("Exceptions      :", int(exception_mask.sum()), "/", len(df))
print("LLM invoked     :", int(invoked.sum()), "/", len(df))
print("Pre-LLM no-call :", int((~invoked).sum()), "/", len(df))
print(
    "Parse OK        :",
    int((invoked & parse_ok).sum()),
    "/",
    int(invoked.sum())
)
print(
    "Parse failures  :",
    int(parse_fail.sum()),
    "/",
    int(invoked.sum())
)

if exception_mask.any():
    print("\nEXCEPTIONS:")
    print(
        df.loc[
            exception_mask,
            ["id", "exception"]
        ].to_string(index=False)
    )

# ------------------------------------------------------------------------------------------
# Canonical containment
# ------------------------------------------------------------------------------------------

descriptor = (
    df["descriptor_json"]
    .fillna("")
    .astype(str)
    .str.strip()
)

descriptor_present = ~descriptor.str.lower().isin(
    ["", "nan", "none", "null", "{}"]
)

accepted = df["accepted"].eq(True)
rejected = df["accepted"].eq(False)

containment_violation = rejected & descriptor_present

print("\nCANONICAL CONTAINMENT")
print("-" * 110)

print("Accepted              :", int(accepted.sum()))
print("Rejected              :", int(rejected.sum()))
print("Descriptors present   :", int(descriptor_present.sum()))
print(
    "Rejected + descriptor :",
    int(containment_violation.sum())
)

# ------------------------------------------------------------------------------------------
# Per-case view
# ------------------------------------------------------------------------------------------

print("\nPER-CASE VIEW")
print("-" * 110)

cols = [
    "id",
    "expected_valid",
    "accepted",
    "decision_correct",
    "llm_invoked",
    "parse_ok",
    "llm_latency_ms",
    "rejection_stage",
    "finish_reason",
    "error_codes_json",
]

cols = [c for c in cols if c in df.columns]

print(df[cols].to_string(index=False))

# ------------------------------------------------------------------------------------------
# Final status
# ------------------------------------------------------------------------------------------

valid = (
    len(df) == 5
    and observed_ids == expected_ids
    and df["id"].astype(str).nunique() == 5
    and int(exception_mask.sum()) == 0
    and int(parse_fail.sum()) == 0
    and int(df["accepted"].notna().sum()) == 5
    and int(df["decision_correct"].notna().sum()) == 5
    and int(containment_violation.sum()) == 0
)

print("\n" + "=" * 110)

if valid:
    print("A9 5-CASE SMOKE STATUS: HEALTHY")
else:
    print("A9 5-CASE SMOKE STATUS: REQUIRES INVESTIGATION")

print("=" * 110)

print("No API/model calls made.")
print("No files modified.")

STEP R14 — VALIDATE A9 5-CASE SMOKE TEST

ARTIFACT
--------------------------------------------------------------------------------------------------------------
Path    : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/smoke/A9_post_restore_5case.csv
Size    : 18660 bytes
SHA256  : 12685498d3a3d5324ff8484e85e230e6de684661d1991a91c17fc73908e99f0b
Rows    : 5
Columns : 41

CASE ALIGNMENT
--------------------------------------------------------------------------------------------------------------
Expected IDs      : ['HO001', 'HO002', 'HO003', 'HO004', 'HO005']
Observed IDs      : ['HO001', 'HO002', 'HO003', 'HO004', 'HO005']
Unique IDs        : 5
Duplicate IDs     : 0
Exact order match : True

EXECUTION HEALTH
--------------------------------------------------------------------------------------------------------------
Exceptions      : 0 / 5
LLM invoked     : 4 / 5
Pre-LLM no-call : 1 / 5
Parse OK        : 4 / 4
Parse failures  : 0 

In [21]:
# ==========================================================================================
# STEP R15 — A9 FULL 120-CASE CONFIRMATORY RECOVERY RERUN
#
# Model:
#   deepseek-ai/DeepSeek-V4.1-Flash
#
# IMPORTANT:
#   - Confirmatory/recovery run only.
#   - Does NOT replace frozen Experiment-A results.
#   - Uses canonical frozen runner and dataset.
#   - Live progress displayed.
# ==========================================================================================

import os
import hashlib
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

RUNNER = MASTER / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

OUT = BASE / (
    "rerun-missing-artifacts/results/raw/"
    "A9_deepseek-v4.1-flash_rerun.csv"
)

MODEL = "deepseek-ai/DeepSeek-V4.1-Flash"

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)

EXPECTED_DATASET_SHA = (
    "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de"
)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


print("=" * 110)
print("STEP R15 — A9 FULL 120-CASE CONFIRMATORY RECOVERY RERUN")
print("=" * 110)

# ------------------------------------------------------------------------------------------
# Verify canonical artifacts
# ------------------------------------------------------------------------------------------

assert RUNNER.exists(), "STOP: canonical runner missing"
assert DATASET.exists(), "STOP: canonical dataset missing"

runner_sha = sha256(RUNNER)
dataset_sha = sha256(DATASET)

print("\nCANONICAL INPUT VERIFICATION")
print("-" * 110)

print("Runner SHA :", runner_sha)
print("Dataset SHA:", dataset_sha)

assert runner_sha == EXPECTED_RUNNER_SHA, (
    "STOP: runner SHA mismatch"
)

assert dataset_sha == EXPECTED_DATASET_SHA, (
    "STOP: dataset SHA mismatch"
)

print("Runner     : VERIFIED")
print("Dataset    : VERIFIED")

# ------------------------------------------------------------------------------------------
# Verify runtime/API environment
# ------------------------------------------------------------------------------------------

key = os.environ.get("TOGETHER_API_KEY")

assert key, "STOP: TOGETHER_API_KEY missing"

# Protect against recurrence of corrupted credential
assert "\n" not in key
assert "\r" not in key
assert "STEP " not in key
assert "import " not in key
assert "print(" not in key

try:
    key.encode("ascii")
except UnicodeEncodeError:
    raise RuntimeError(
        "STOP: API key contains non-ASCII characters."
    )

import together
from together import Together

print("\nENVIRONMENT")
print("-" * 110)

print("Together SDK :", together.__version__)
print("API key      : PRESENT / SANITY-CHECKED / HIDDEN")
print("Model        :", MODEL)
print("Cases        : 120")

# ------------------------------------------------------------------------------------------
# Protect existing evidence
# ------------------------------------------------------------------------------------------

if OUT.exists():
    raise RuntimeError(
        "STOP: A9 full-run output already exists.\n"
        f"{OUT}\n"
        "Do not overwrite existing recovery evidence."
    )

OUT.parent.mkdir(parents=True, exist_ok=True)

os.environ["PYTHONPATH"] = str(MASTER)

print("\nOUTPUT")
print("-" * 110)
print(OUT)

print("\n" + "=" * 110)
print("STARTING A9")
print("=" * 110)
print()

# ------------------------------------------------------------------------------------------
# Execute canonical runner
# ------------------------------------------------------------------------------------------

%run "$RUNNER" \
    --dataset "$DATASET" \
    --out "$OUT" \
    --model "$MODEL"

STEP R15 — A9 FULL 120-CASE CONFIRMATORY RECOVERY RERUN

CANONICAL INPUT VERIFICATION
--------------------------------------------------------------------------------------------------------------
Runner SHA : 40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257
Dataset SHA: 84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de
Runner     : VERIFIED
Dataset    : VERIFIED

ENVIRONMENT
--------------------------------------------------------------------------------------------------------------
Together SDK : 2.39.0
API key      : PRESENT / SANITY-CHECKED / HIDDEN
Model        : deepseek-ai/DeepSeek-V4.1-Flash
Cases        : 120

OUTPUT
--------------------------------------------------------------------------------------------------------------
/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/raw/A9_deepseek-v4.1-flash_rerun.csv

STARTING A9

[1/120] HO001
[2/120] HO002
[3/120] HO003
[4/120] HO004
[5/120] HO005
[

In [22]:
# ==========================================================================================
# STEP R16 — VALIDATE FULL A9 120-CASE RECOVERY ARTIFACT
#
# NO model/API calls.
# NO files modified.
# ==========================================================================================

from pathlib import Path
import pandas as pd
import hashlib

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

MASTER = BASE / "experiment-1-v2-together_FINAL"

DATASET = MASTER / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

RESULT = BASE / (
    "rerun-missing-artifacts/results/raw/"
    "A9_deepseek-v4.1-flash_rerun.csv"
)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


print("=" * 110)
print("STEP R16 — VALIDATE FULL A9 RECOVERY ARTIFACT")
print("=" * 110)

assert DATASET.exists(), f"STOP: dataset missing:\n{DATASET}"
assert RESULT.exists(), f"STOP: A9 result missing:\n{RESULT}"

benchmark = pd.read_csv(DATASET)
df = pd.read_csv(RESULT)

# ------------------------------------------------------------------------------------------
# Artifact
# ------------------------------------------------------------------------------------------

print("\nARTIFACT")
print("-" * 110)

result_sha = sha256(RESULT)

print("Path    :", RESULT)
print("Size    :", RESULT.stat().st_size, "bytes")
print("SHA256  :", result_sha)
print("Rows    :", len(df))
print("Columns :", len(df.columns))

# ------------------------------------------------------------------------------------------
# Case alignment
# ------------------------------------------------------------------------------------------

expected_ids = benchmark["id"].astype(str).tolist()
observed_ids = df["id"].astype(str).tolist()

missing = sorted(set(expected_ids) - set(observed_ids))
unexpected = sorted(set(observed_ids) - set(expected_ids))

print("\nCASE ALIGNMENT")
print("-" * 110)

print("Expected rows      :", len(expected_ids))
print("Observed rows      :", len(observed_ids))
print("Unique IDs         :", df["id"].astype(str).nunique())
print("Duplicate IDs      :", int(df["id"].astype(str).duplicated().sum()))
print("Missing IDs        :", len(missing))
print("Unexpected IDs     :", len(unexpected))
print("Exact order match  :", observed_ids == expected_ids)

# ------------------------------------------------------------------------------------------
# Execution health
# ------------------------------------------------------------------------------------------

exc = (
    df["exception"]
    .fillna("")
    .astype(str)
    .str.strip()
)

exception_mask = ~exc.str.lower().isin(
    ["", "nan", "none", "null", "false"]
)

invoked = df["llm_invoked"].eq(True)
parse_ok = df["parse_ok"].eq(True)

parse_fail = invoked & ~parse_ok

print("\nEXECUTION HEALTH")
print("-" * 110)

print("Exceptions         :", int(exception_mask.sum()), "/", len(df))
print("LLM invoked        :", int(invoked.sum()), "/", len(df))
print("Pre-LLM no-call    :", int((~invoked).sum()), "/", len(df))
print(
    "Parse OK           :",
    int((invoked & parse_ok).sum()),
    "/",
    int(invoked.sum())
)
print(
    "Parse failures     :",
    int(parse_fail.sum()),
    "/",
    int(invoked.sum())
)

if exception_mask.any():
    print("\nEXCEPTIONS")
    print("-" * 110)

    print(
        df.loc[
            exception_mask,
            ["id", "exception"]
        ].to_string(index=False)
    )

# ------------------------------------------------------------------------------------------
# Decision completeness
# ------------------------------------------------------------------------------------------

accepted_complete = int(df["accepted"].notna().sum())
decision_complete = int(df["decision_correct"].notna().sum())

print("\nDECISION COMPLETENESS")
print("-" * 110)

print(
    "accepted populated         :",
    accepted_complete,
    "/",
    len(df)
)

print(
    "decision_correct populated :",
    decision_complete,
    "/",
    len(df)
)

# ------------------------------------------------------------------------------------------
# Canonical containment
#
# Violation:
#   accepted == False AND descriptor_json is present
# ------------------------------------------------------------------------------------------

descriptor = (
    df["descriptor_json"]
    .fillna("")
    .astype(str)
    .str.strip()
)

descriptor_present = ~descriptor.str.lower().isin(
    ["", "nan", "none", "null", "{}"]
)

accepted = df["accepted"].eq(True)
rejected = df["accepted"].eq(False)

containment_violation = rejected & descriptor_present
accepted_without_descriptor = accepted & ~descriptor_present

print("\nCANONICAL CONTAINMENT")
print("-" * 110)

print("Accepted                    :", int(accepted.sum()))
print("Rejected                    :", int(rejected.sum()))
print("Descriptors present         :", int(descriptor_present.sum()))
print(
    "Accepted without descriptor :",
    int(accepted_without_descriptor.sum())
)
print(
    "Rejected + descriptor       :",
    int(containment_violation.sum())
)

if containment_violation.any():
    print("\nCONTAINMENT VIOLATION IDs")
    print("-" * 110)

    print(
        df.loc[
            containment_violation,
            ["id", "accepted", "rejection_stage"]
        ].to_string(index=False)
    )

# ------------------------------------------------------------------------------------------
# Operational latency
# ------------------------------------------------------------------------------------------

llm_latency = pd.to_numeric(
    df.loc[invoked, "llm_latency_ms"],
    errors="coerce"
).dropna()

print("\nOPERATIONAL LATENCY")
print("-" * 110)

if len(llm_latency):
    print("Measured LLM calls :", len(llm_latency))
    print("Minimum            :", round(llm_latency.min() / 1000, 3), "sec")
    print("Maximum            :", round(llm_latency.max() / 1000, 3), "sec")
    print("Mean               :", round(llm_latency.mean() / 1000, 3), "sec")
    print("Median             :", round(llm_latency.median() / 1000, 3), "sec")
    print("Total LLM latency  :", round(llm_latency.sum() / 60000, 2), "min")

# ------------------------------------------------------------------------------------------
# Technical validation
# ------------------------------------------------------------------------------------------

technical_valid = (
    len(df) == 120
    and len(benchmark) == 120
    and df["id"].astype(str).nunique() == 120
    and int(df["id"].astype(str).duplicated().sum()) == 0
    and len(missing) == 0
    and len(unexpected) == 0
    and observed_ids == expected_ids
    and int(exception_mask.sum()) == 0
    and int(parse_fail.sum()) == 0
    and accepted_complete == 120
    and decision_complete == 120
    and int(containment_violation.sum()) == 0
)

print("\n" + "=" * 110)

if technical_valid:
    print("A9 FULL RECOVERY ARTIFACT STATUS: VALID")
else:
    print("A9 FULL RECOVERY ARTIFACT STATUS: REQUIRES INVESTIGATION")

print("=" * 110)

print("\nNo API/model calls made.")
print("No files modified.")
print("No metrics artifact created yet.")


STEP R16 — VALIDATE FULL A9 RECOVERY ARTIFACT

ARTIFACT
--------------------------------------------------------------------------------------------------------------
Path    : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts/results/raw/A9_deepseek-v4.1-flash_rerun.csv
Size    : 369086 bytes
SHA256  : 16eb0b8c9d6a75d84bcb1b29758f87d0263ddf9fa51262b029b1c9865fb7d6a3
Rows    : 120
Columns : 41

CASE ALIGNMENT
--------------------------------------------------------------------------------------------------------------
Expected rows      : 120
Observed rows      : 120
Unique IDs         : 120
Duplicate IDs      : 0
Missing IDs        : 0
Unexpected IDs     : 0
Exact order match  : True

EXECUTION HEALTH
--------------------------------------------------------------------------------------------------------------
Exceptions         : 0 / 120
LLM invoked        : 77 / 120
Pre-LLM no-call    : 43 / 120
Parse OK           : 77 / 77
Parse failures  

In [23]:
# ==========================================================================================
# STEP R17 — A9 METRICS + FROZEN RESULT COMPARISON
#
# NO API/model calls.
# Raw A9 artifact is read-only.
# Creates ONE new metrics JSON artifact.
#
# Positive class = expected INVALID / pipeline REJECT
#
# TP = expected invalid & rejected
# FP = expected valid   & rejected
# FN = expected invalid & accepted
# TN = expected valid   & accepted
#
# Accuracy = (TP + TN) / N
# VAR      = TN / expected-valid
# IRR      = TP / expected-invalid
#
# Canonical containment violation:
# rejected (accepted == False) AND descriptor_json is present
# ==========================================================================================

from pathlib import Path
import pandas as pd
import hashlib
import json
from datetime import datetime, timezone

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

RAW = BASE / (
    "rerun-missing-artifacts/results/raw/"
    "A9_deepseek-v4.1-flash_rerun.csv"
)

METRICS = BASE / (
    "rerun-missing-artifacts/results/metrics/"
    "A9_deepseek-v4.1-flash_rerun_metrics_v1.json"
)

EXPECTED_RAW_SHA = (
    "16eb0b8c9d6a75d84bcb1b29758f87d0263ddf9fa51262b029b1c9865fb7d6a3"
)

# Frozen Experiment-A A9 result
FROZEN = {
    "tp": 50,
    "fp": 7,
    "fn": 12,
    "tn": 51,
    "accuracy_pct": 84.17,
    "var_pct": 87.93,
    "irr_pct": 80.65,
    "containment_violations": 0,
}


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


print("=" * 110)
print("STEP R17 — A9 METRICS + FROZEN RESULT COMPARISON")
print("=" * 110)

assert RAW.exists(), f"STOP: A9 raw artifact missing:\n{RAW}"

raw_sha = sha256(RAW)

assert raw_sha == EXPECTED_RAW_SHA, (
    "STOP: A9 raw SHA has changed.\n"
    f"Expected: {EXPECTED_RAW_SHA}\n"
    f"Observed: {raw_sha}"
)

if METRICS.exists():
    raise RuntimeError(
        "STOP: A9 metrics artifact already exists.\n"
        f"{METRICS}\n"
        "Do not overwrite recovery evidence."
    )

df = pd.read_csv(RAW)

assert len(df) == 120, "STOP: expected exactly 120 rows."

# ------------------------------------------------------------------------------------------
# Boolean fields
# ------------------------------------------------------------------------------------------

expected_valid = df["expected_valid"].eq(True)
expected_invalid = ~expected_valid

accepted = df["accepted"].eq(True)
rejected = df["accepted"].eq(False)

# ------------------------------------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------------------------------------

tp = int((expected_invalid & rejected).sum())
fp = int((expected_valid & rejected).sum())
fn = int((expected_invalid & accepted).sum())
tn = int((expected_valid & accepted).sum())

n = len(df)

valid_n = int(expected_valid.sum())
invalid_n = int(expected_invalid.sum())

assert valid_n == 58
assert invalid_n == 62

assert tp + fn == invalid_n
assert tn + fp == valid_n
assert tp + fp + fn + tn == n

# ------------------------------------------------------------------------------------------
# Metrics
# ------------------------------------------------------------------------------------------

accuracy = (tp + tn) / n

var = tn / valid_n
irr = tp / invalid_n

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall)
    else 0.0
)

# ------------------------------------------------------------------------------------------
# Canonical containment
# ------------------------------------------------------------------------------------------

descriptor = (
    df["descriptor_json"]
    .fillna("")
    .astype(str)
    .str.strip()
)

descriptor_present = ~descriptor.str.lower().isin(
    ["", "nan", "none", "null", "{}"]
)

containment_mask = rejected & descriptor_present

containment_violations = int(containment_mask.sum())

# ------------------------------------------------------------------------------------------
# Execution health
# ------------------------------------------------------------------------------------------

exc = (
    df["exception"]
    .fillna("")
    .astype(str)
    .str.strip()
)

exception_mask = ~exc.str.lower().isin(
    ["", "nan", "none", "null", "false"]
)

llm_invoked = df["llm_invoked"].eq(True)
parse_ok = df["parse_ok"].eq(True)

exceptions = int(exception_mask.sum())
invoked_count = int(llm_invoked.sum())
parse_success = int((llm_invoked & parse_ok).sum())

# ------------------------------------------------------------------------------------------
# Current confirmatory result
# ------------------------------------------------------------------------------------------

CURRENT = {
    "tp": tp,
    "fp": fp,
    "fn": fn,
    "tn": tn,

    "accuracy_pct": round(accuracy * 100, 2),
    "var_pct": round(var * 100, 2),
    "irr_pct": round(irr * 100, 2),

    "precision_pct": round(precision * 100, 2),
    "recall_pct": round(recall * 100, 2),
    "f1_pct": round(f1 * 100, 2),

    "containment_violations": containment_violations,
}

# ------------------------------------------------------------------------------------------
# Frozen comparison
# ------------------------------------------------------------------------------------------

comparison = {
    "tp_delta": tp - FROZEN["tp"],
    "fp_delta": fp - FROZEN["fp"],
    "fn_delta": fn - FROZEN["fn"],
    "tn_delta": tn - FROZEN["tn"],

    "accuracy_delta_pp": round(
        CURRENT["accuracy_pct"] - FROZEN["accuracy_pct"], 2
    ),

    "var_delta_pp": round(
        CURRENT["var_pct"] - FROZEN["var_pct"], 2
    ),

    "irr_delta_pp": round(
        CURRENT["irr_pct"] - FROZEN["irr_pct"], 2
    ),

    "containment_delta": (
        containment_violations -
        FROZEN["containment_violations"]
    ),
}

# ------------------------------------------------------------------------------------------
# Display results
# ------------------------------------------------------------------------------------------

print("\nCONFUSION MATRIX — TODAY'S CONFIRMATORY RERUN")
print("-" * 110)

print("TP :", tp)
print("FP :", fp)
print("FN :", fn)
print("TN :", tn)

print("\nTODAY'S METRICS")
print("-" * 110)

print(f"Accuracy  : {CURRENT['accuracy_pct']:.2f}%")
print(f"VAR       : {CURRENT['var_pct']:.2f}%")
print(f"IRR       : {CURRENT['irr_pct']:.2f}%")
print(f"Precision : {CURRENT['precision_pct']:.2f}%")
print(f"Recall    : {CURRENT['recall_pct']:.2f}%")
print(f"F1        : {CURRENT['f1_pct']:.2f}%")

print(
    "Containment violations :",
    containment_violations
)

print("\nFROZEN A9 RESULT")
print("-" * 110)

print(
    f"TP/FP/FN/TN : "
    f"{FROZEN['tp']}/"
    f"{FROZEN['fp']}/"
    f"{FROZEN['fn']}/"
    f"{FROZEN['tn']}"
)

print(f"Accuracy    : {FROZEN['accuracy_pct']:.2f}%")
print(f"VAR         : {FROZEN['var_pct']:.2f}%")
print(f"IRR         : {FROZEN['irr_pct']:.2f}%")

print(
    "Containment violations :",
    FROZEN["containment_violations"]
)

print("\nTODAY vs FROZEN")
print("-" * 110)

print(f"TP delta       : {comparison['tp_delta']:+d}")
print(f"FP delta       : {comparison['fp_delta']:+d}")
print(f"FN delta       : {comparison['fn_delta']:+d}")
print(f"TN delta       : {comparison['tn_delta']:+d}")

print(
    f"Accuracy delta : "
    f"{comparison['accuracy_delta_pp']:+.2f} pp"
)

print(
    f"VAR delta      : "
    f"{comparison['var_delta_pp']:+.2f} pp"
)

print(
    f"IRR delta      : "
    f"{comparison['irr_delta_pp']:+.2f} pp"
)

print(
    "Containment delta:",
    f"{comparison['containment_delta']:+d}"
)

# ------------------------------------------------------------------------------------------
# Persist metrics artifact
# ------------------------------------------------------------------------------------------

artifact = {
    "artifact_type": "confirmatory_recovery_metrics",

    "experiment": "Experiment A",
    "arm": "A9",
    "model": "deepseek-ai/DeepSeek-V4.1-Flash",

    "created_utc": datetime.now(timezone.utc).isoformat(),

    "scope": {
        "purpose": (
            "Confirmatory recovery rerun of missing raw artifact."
        ),
        "frozen_results_modified": False,
        "replacement_for_frozen_result": False,
    },

    "raw_artifact": {
        "path": str(RAW),
        "sha256": raw_sha,
        "rows": n,
    },

    "dataset_counts": {
        "total": n,
        "expected_valid": valid_n,
        "expected_invalid": invalid_n,
    },

    "execution_health": {
        "exceptions": exceptions,
        "llm_invoked": invoked_count,
        "parse_success": parse_success,
    },

    "current_confirmatory_result": CURRENT,

    "frozen_recorded_result": FROZEN,

    "comparison_current_minus_frozen": comparison,

    "canonical_containment_definition": (
        "accepted == False AND descriptor_json is present"
    ),

    "interpretation_boundary": (
        "Differences between this confirmatory rerun and the frozen "
        "recorded result are reported as observed and do not alter "
        "the frozen Experiment-A evidence."
    ),
}

METRICS.parent.mkdir(parents=True, exist_ok=True)

with open(METRICS, "w", encoding="utf-8") as f:
    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

metrics_sha = sha256(METRICS)

print("\nMETRICS ARTIFACT")
print("-" * 110)

print("Path   :", METRICS)
print("SHA256 :", metrics_sha)

print("\n" + "=" * 110)
print("STEP R17 COMPLETE")
print("=" * 110)

print("Raw A9 artifact unchanged.")
print("Frozen Experiment-A evidence unchanged.")
print("No API/model calls made.")

STEP R17 — A9 METRICS + FROZEN RESULT COMPARISON

CONFUSION MATRIX — TODAY'S CONFIRMATORY RERUN
--------------------------------------------------------------------------------------------------------------
TP : 50
FP : 7
FN : 12
TN : 51

TODAY'S METRICS
--------------------------------------------------------------------------------------------------------------
Accuracy  : 84.17%
VAR       : 87.93%
IRR       : 80.65%
Precision : 87.72%
Recall    : 80.65%
F1        : 84.03%
Containment violations : 0

FROZEN A9 RESULT
--------------------------------------------------------------------------------------------------------------
TP/FP/FN/TN : 50/7/12/51
Accuracy    : 84.17%
VAR         : 87.93%
IRR         : 80.65%
Containment violations : 0

TODAY vs FROZEN
--------------------------------------------------------------------------------------------------------------
TP delta       : +0
FP delta       : +0
FN delta       : +0
TN delta       : +0
Accuracy delta : +0.00 pp
VAR delta      

In [24]:
# ==========================================================================================
# STEP R18 — CONSOLIDATE CONFIRMATORY RECOVERY EVIDENCE
#
# NO API/model calls.
# NO experiment reruns.
# NO existing artifacts modified.
#
# Creates:
#   rerun-missing-artifacts/manifests/
#   experiment_A_recovery_consolidation_v1.json
# ==========================================================================================

from pathlib import Path
import hashlib
import json
from datetime import datetime, timezone

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

ROOT = BASE / "rerun-missing-artifacts"

RAW_DIR = ROOT / "results/raw"
METRICS_DIR = ROOT / "results/metrics"
MANIFEST_DIR = ROOT / "manifests"

OUT = MANIFEST_DIR / "experiment_A_recovery_consolidation_v1.json"


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


print("=" * 110)
print("STEP R18 — CONSOLIDATE CONFIRMATORY RECOVERY EVIDENCE")
print("=" * 110)

# ------------------------------------------------------------------------------------------
# Known recovery artifacts
# ------------------------------------------------------------------------------------------

artifacts = {
    "A2": {
        "raw": RAW_DIR / "A2_llama-3.3-70b_rerun.csv",
        "metrics": METRICS_DIR / "A2_canonical_containment_recheck_v1.json",

        "frozen": {
            "tp": 52,
            "fp": 10,
            "fn": 10,
            "tn": 48,
            "accuracy_pct": 83.33,
            "var_pct": 82.76,
            "irr_pct": 83.87,
            "containment_violations": 0,
        },

        "recovery": {
            "tp": 51,
            "fp": 10,
            "fn": 11,
            "tn": 48,
            "accuracy_pct": 82.50,
            "var_pct": 82.76,
            "irr_pct": 82.26,
            "containment_violations": 0,
        },
    },

    "A7": {
        "raw": RAW_DIR / "A7_qwen-2-1.5b_rerun.csv",
        "metrics": METRICS_DIR / "A7_qwen-2-1.5b_rerun_metrics_v1.json",

        "frozen": {
            "tp": 61,
            "fp": 47,
            "fn": 1,
            "tn": 11,
            "accuracy_pct": 60.00,
            "var_pct": 18.97,
            "irr_pct": 98.39,
            "containment_violations": 0,
        },

        "recovery": {
            "tp": 61,
            "fp": 47,
            "fn": 1,
            "tn": 11,
            "accuracy_pct": 60.00,
            "var_pct": 18.97,
            "irr_pct": 98.39,
            "containment_violations": 0,
        },
    },

    "A8": {
        "raw": RAW_DIR / "A8_ternary-bonsai-27b_rerun.csv",
        "metrics": METRICS_DIR / "A8_ternary-bonsai-27b_rerun_metrics_v1.json",

        "frozen": {
            "tp": 50,
            "fp": 17,
            "fn": 12,
            "tn": 41,
            "accuracy_pct": 75.83,
            "var_pct": 70.69,
            "irr_pct": 80.65,
            "containment_violations": 0,
        },

        "recovery": {
            "tp": 51,
            "fp": 22,
            "fn": 11,
            "tn": 36,
            "accuracy_pct": 72.50,
            "var_pct": 62.07,
            "irr_pct": 82.26,
            "containment_violations": 0,
        },
    },

    "A9": {
        "raw": RAW_DIR / "A9_deepseek-v4.1-flash_rerun.csv",
        "metrics": METRICS_DIR / "A9_deepseek-v4.1-flash_rerun_metrics_v1.json",

        "frozen": {
            "tp": 50,
            "fp": 7,
            "fn": 12,
            "tn": 51,
            "accuracy_pct": 84.17,
            "var_pct": 87.93,
            "irr_pct": 80.65,
            "containment_violations": 0,
        },

        "recovery": {
            "tp": 50,
            "fp": 7,
            "fn": 12,
            "tn": 51,
            "accuracy_pct": 84.17,
            "var_pct": 87.93,
            "irr_pct": 80.65,
            "containment_violations": 0,
        },
    },
}

# ------------------------------------------------------------------------------------------
# Protect existing consolidation evidence
# ------------------------------------------------------------------------------------------

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

if OUT.exists():
    raise RuntimeError(
        "STOP: consolidation manifest already exists.\n"
        f"{OUT}\n"
        "Do not overwrite it."
    )

# ------------------------------------------------------------------------------------------
# Verify artifacts and build manifest
# ------------------------------------------------------------------------------------------

consolidated = {}

print("\nARTIFACT VERIFICATION")
print("-" * 110)

for arm, info in artifacts.items():

    raw = info["raw"]
    metrics = info["metrics"]

    assert raw.exists(), (
        f"STOP: {arm} raw artifact missing:\n{raw}"
    )

    assert metrics.exists(), (
        f"STOP: {arm} metrics artifact missing:\n{metrics}"
    )

    raw_sha = sha256(raw)
    metrics_sha = sha256(metrics)

    frozen = info["frozen"]
    recovery = info["recovery"]

    deltas = {
        "tp": recovery["tp"] - frozen["tp"],
        "fp": recovery["fp"] - frozen["fp"],
        "fn": recovery["fn"] - frozen["fn"],
        "tn": recovery["tn"] - frozen["tn"],

        "accuracy_pp": round(
            recovery["accuracy_pct"] -
            frozen["accuracy_pct"],
            2
        ),

        "var_pp": round(
            recovery["var_pct"] -
            frozen["var_pct"],
            2
        ),

        "irr_pp": round(
            recovery["irr_pct"] -
            frozen["irr_pct"],
            2
        ),

        "containment": (
            recovery["containment_violations"] -
            frozen["containment_violations"]
        ),
    }

    consolidated[arm] = {
        "raw_artifact": {
            "path": str(raw),
            "sha256": raw_sha,
            "size_bytes": raw.stat().st_size,
        },

        "metrics_artifact": {
            "path": str(metrics),
            "sha256": metrics_sha,
            "size_bytes": metrics.stat().st_size,
        },

        "frozen_recorded_result": frozen,
        "confirmatory_recovery_result": recovery,
        "recovery_minus_frozen": deltas,
    }

    print(
        f"{arm}: "
        f"raw={raw_sha[:12]}...  "
        f"metrics={metrics_sha[:12]}..."
    )

# ------------------------------------------------------------------------------------------
# Important A2 provenance note
# ------------------------------------------------------------------------------------------

a2_provisional = (
    METRICS_DIR /
    "A2_llama-3.3-70b_rerun_metrics_v1.json"
)

a2_provisional_info = None

if a2_provisional.exists():
    a2_provisional_info = {
        "path": str(a2_provisional),
        "sha256": sha256(a2_provisional),
        "status": (
            "Preserved historical recovery artifact with a provisional/"
            "incorrect containment proxy. Do not use its containment field. "
            "Use A2_canonical_containment_recheck_v1.json instead."
        ),
    }

# ------------------------------------------------------------------------------------------
# Interrupted A8 attempt provenance
# ------------------------------------------------------------------------------------------

attempt1 = (
    MANIFEST_DIR /
    "A8_interrupted_attempt_20261003_v1.json"
)

attempt2 = (
    MANIFEST_DIR /
    "A8_interrupted_attempt2_20261003_v1.json"
)

a8_attempts = []

for p in [attempt1, attempt2]:
    if p.exists():
        a8_attempts.append({
            "path": str(p),
            "sha256": sha256(p),
            "classification": (
                "non-evaluable interrupted attempt; "
                "not classified as model/API failure"
            ),
        })

# ------------------------------------------------------------------------------------------
# Consolidation manifest
# ------------------------------------------------------------------------------------------

manifest = {
    "artifact_type": (
        "experiment_A_confirmatory_recovery_consolidation"
    ),

    "version": "v1",

    "created_utc": datetime.now(timezone.utc).isoformat(),

    "scope": {
        "purpose": (
            "Consolidate separately executed confirmatory/recovery "
            "reruns performed to recover missing raw artifacts."
        ),

        "arms": ["A2", "A7", "A8", "A9"],

        "frozen_results_modified": False,

        "replacement_for_frozen_experiment_A": False,

        "paper_reporting_rule": (
            "Original frozen Experiment-A results remain the primary "
            "reported results. Recovery reruns are retained separately "
            "as confirmatory/reproducibility evidence."
        ),
    },

    "metric_definition": {
        "positive_class": "expected invalid / pipeline reject",

        "tp": "expected invalid and rejected",
        "fp": "expected valid and rejected",
        "fn": "expected invalid and accepted",
        "tn": "expected valid and accepted",

        "VAR": "TN / expected-valid",
        "IRR": "TP / expected-invalid",

        "canonical_containment_violation": (
            "accepted == False AND descriptor_json is present"
        ),
    },

    "arms": consolidated,

    "a2_provisional_artifact": a2_provisional_info,

    "a8_interrupted_attempts": a8_attempts,

    "observations": {
        "A2": (
            "Recovery decision metrics differed slightly from frozen "
            "metrics; canonical containment remained zero."
        ),

        "A7": (
            "Recovery aggregate decision metrics matched frozen "
            "aggregate metrics; canonical containment remained zero."
        ),

        "A8": (
            "Recovery decision metrics differed from frozen metrics; "
            "canonical containment remained zero."
        ),

        "A9": (
            "Recovery aggregate decision metrics matched frozen "
            "aggregate metrics; canonical containment remained zero."
        ),

        "overall": (
            "Semantic decision performance showed rerun variability "
            "for some arms, while zero canonical containment violations "
            "were observed in all four confirmatory recovery runs."
        ),
    },

    "claim_boundary": (
        "These recovery observations do not establish universal model "
        "independence, hallucination freedom, zero risk, formal standards "
        "conformance, or deterministic semantic-model outputs."
    ),
}

with open(OUT, "w", encoding="utf-8") as f:
    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

manifest_sha = sha256(OUT)

# ------------------------------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------------------------------

print("\nRECOVERY SUMMARY")
print("-" * 110)

for arm in ["A2", "A7", "A8", "A9"]:

    x = consolidated[arm]

    f = x["frozen_recorded_result"]
    r = x["confirmatory_recovery_result"]
    d = x["recovery_minus_frozen"]

    print(
        f"{arm}: "
        f"ACC {f['accuracy_pct']:.2f}% -> {r['accuracy_pct']:.2f}% "
        f"({d['accuracy_pp']:+.2f} pp), "
        f"VAR {f['var_pct']:.2f}% -> {r['var_pct']:.2f}% "
        f"({d['var_pp']:+.2f} pp), "
        f"IRR {f['irr_pct']:.2f}% -> {r['irr_pct']:.2f}% "
        f"({d['irr_pp']:+.2f} pp), "
        f"CV {r['containment_violations']}"
    )

print("\nCONSOLIDATION MANIFEST")
print("-" * 110)

print("Path   :", OUT)
print("SHA256 :", manifest_sha)

print("\n" + "=" * 110)
print("STEP R18 COMPLETE")
print("=" * 110)

print("No API/model calls made.")
print("No experiment reruns performed.")
print("No existing artifacts modified.")
print("Frozen Experiment-A evidence unchanged.")

STEP R18 — CONSOLIDATE CONFIRMATORY RECOVERY EVIDENCE

ARTIFACT VERIFICATION
--------------------------------------------------------------------------------------------------------------
A2: raw=5856caf2437d...  metrics=1fbfeb43af54...
A7: raw=78b2acecc9a8...  metrics=d3668e958570...
A8: raw=4b39ff2219c8...  metrics=f9142da410f6...
A9: raw=16eb0b8c9d6a...  metrics=333c8fd77d05...

RECOVERY SUMMARY
--------------------------------------------------------------------------------------------------------------
A2: ACC 83.33% -> 82.50% (-0.83 pp), VAR 82.76% -> 82.76% (+0.00 pp), IRR 83.87% -> 82.26% (-1.61 pp), CV 0
A7: ACC 60.00% -> 60.00% (+0.00 pp), VAR 18.97% -> 18.97% (+0.00 pp), IRR 98.39% -> 98.39% (+0.00 pp), CV 0
A8: ACC 75.83% -> 72.50% (-3.33 pp), VAR 70.69% -> 62.07% (-8.62 pp), IRR 80.65% -> 82.26% (+1.61 pp), CV 0
A9: ACC 84.17% -> 84.17% (+0.00 pp), VAR 87.93% -> 87.93% (+0.00 pp), IRR 80.65% -> 80.65% (+0.00 pp), CV 0

CONSOLIDATION MANIFEST
-------------------------------

In [25]:
# ==========================================================================================
# STEP Z1 — CREATE ZIP OF RERUN-MISSING-ARTIFACTS
#
# READ-ONLY with respect to existing rerun results.
# Does NOT modify/delete/rename any existing artifact.
# Does NOT make API/model calls.
#
# Creates:
#   /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/
#   rerun-missing-artifacts.zip
# ==========================================================================================

from pathlib import Path
import shutil
import hashlib

BASE = Path(
    "/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation"
)

SOURCE = BASE / "rerun-missing-artifacts"
ZIP_BASE = BASE / "rerun-missing-artifacts"
ZIP_FILE = BASE / "rerun-missing-artifacts.zip"


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


print("=" * 100)
print("STEP Z1 — CREATE RERUN-MISSING-ARTIFACTS ZIP")
print("=" * 100)

assert SOURCE.exists(), f"STOP: source directory missing:\n{SOURCE}"
assert SOURCE.is_dir(), f"STOP: source is not a directory:\n{SOURCE}"

# Protect an existing ZIP from accidental overwrite.
if ZIP_FILE.exists():
    raise RuntimeError(
        "STOP: ZIP already exists. It has NOT been overwritten.\n"
        f"{ZIP_FILE}"
    )

# Count source files before creating ZIP.
source_files = [
    p for p in SOURCE.rglob("*")
    if p.is_file()
]

print("\nSOURCE")
print("-" * 100)
print("Directory :", SOURCE)
print("Files     :", len(source_files))

# Create ZIP.
created = shutil.make_archive(
    base_name=str(ZIP_BASE),
    format="zip",
    root_dir=str(BASE),
    base_dir="rerun-missing-artifacts",
)

created = Path(created)

assert created.exists(), "STOP: ZIP creation failed."

print("\nZIP CREATED")
print("-" * 100)
print("Path      :", created)
print("Size      :", created.stat().st_size, "bytes")
print("SHA256    :", sha256(created))

print("\n" + "=" * 100)
print("STEP Z1 COMPLETE")
print("=" * 100)

print("Existing rerun artifacts were NOT modified.")
print("No API/model calls made.")

STEP Z1 — CREATE RERUN-MISSING-ARTIFACTS ZIP

SOURCE
----------------------------------------------------------------------------------------------------
Directory : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts
Files     : 26

ZIP CREATED
----------------------------------------------------------------------------------------------------
Path      : /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/rerun-missing-artifacts.zip
Size      : 127535 bytes
SHA256    : 2a4d9caac4b495f990c72e64f424abcc2f3a317cf193f549c77be183885503af

STEP Z1 COMPLETE
Existing rerun artifacts were NOT modified.
No API/model calls made.


In [26]:
# ==========================================================================================
# STEP Z2 — DOWNLOAD ZIP TO YOUR COMPUTER
# ==========================================================================================

from google.colab import files

ZIP_FILE = (
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "rerun-missing-artifacts.zip"
)

files.download(ZIP_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>